In [7]:
# ============================================================
# MetaDiv_KRONA — USER PARAMETERS
# Change only this cell
# ============================================================

# None = process every CSV inside input_KRONA/
# Or write one filename, e.g. "subset_fungi.csv"
INPUT_FILE = None

# Deepest TAXONOMIC level shown in Krona.
# Options:
# None, "domain", "kingdom", "phylum", "class",
# "order", "family", "genus", "species"
#
# None = use the deepest taxonomic rank available in the CSV.
TAXONOMIC_CUTOFF = "genus"

# Visualization mode
#
# "global" = DEFAULT.
#            Sum ALL detected sites/samples into one Krona chart.
#            Hierarchy: taxonomy -> optional lifestyle
#
# "samples" = Keep every site/sample separate inside ONE Krona HTML.
#             "sample" becomes the FIRST hierarchy level:
#             sample -> taxonomy -> optional lifestyle
#
# Examples:
# MODE = "global"
# MODE = "samples"
MODE = "global"

# Homogenize conflicting taxonomic lineages before building Krona.
# Recommended = True.
#
# Example:
# if Russula appears under more than one family/order because different
# source annotations used different or incomplete lineages, MetaDiv_KRONA
# chooses one canonical lineage and uses it consistently.
HOMOGENIZE_TAXONOMY = True

# Add ecological annotations AFTER the selected taxonomic cutoff
# when these columns are available.
INCLUDE_PRIMARY_LIFESTYLE = False
INCLUDE_SECONDARY_LIFESTYLE = False

# Add SPPN as the deepest level
INCLUDE_SPPN = False

# Optional maximum number of aggregated paths
# None = keep all paths
MAX_KRONA_PATHS = None


In [8]:
# ============================================================
# MetaDiv_KRONA — PROGRAM
# Normally you do not need to edit this cell
# ============================================================

from pathlib import Path

# Internal fixed defaults
INPUT_DIR = Path("input_KRONA")
OUTPUT_DIR = Path("output_KRONA")
EXPORT_TOTAL_ABUNDANCE = True
DROP_ZERO_ABUNDANCE = True
FILL_UNCLASSIFIED = True
CREATE_HTML = True

# Internal mode normalization
_MODE = str(MODE).strip().lower()

if _MODE not in {"global", "samples"}:
    raise ValueError(
        "Invalid MODE. Use either:\n"
        '  MODE = "global"\n'
        '  MODE = "samples"'
    )

# Keep the existing internal control flow, but user chooses MODE.
EXPORT_PER_SAMPLE = (_MODE == "samples")


import pandas as pd
import numpy as np
import re
import html as html_lib
from pathlib import Path
from datetime import datetime
from IPython.display import display

# -----------------------------------------------------------------------------
# Embedded Krona resources
# -----------------------------------------------------------------------------
# These resources are embedded directly in the output HTML. They come from the
# same reverse-engineered Krona renderer already validated in MetaDiv Builder.
# No external KronaTools/Excel/network dependency is required at runtime.

_KRONA_LEGACY_JS = '{//-----------------------------------------------------------------------------\n// \n// PURPOSE\n// \n// Krona is a flexible tool for exploring the relative proportions of\n// hierarchical data, such as metagenomic classifications, using a\n// radial, space-filling display. It is implemented using HTML5 and\n// JavaScript, allowing charts to be explored locally or served over the\n// Internet, requiring only a current version of any major web\n// browser. Krona charts can be created using an Excel template or from\n// common bioinformatic formats using the provided conversion scripts.\n// \n// \n// COPYRIGHT LICENSE\n// \n// Copyright (c) 2011, Battelle National Biodefense Institute (BNBI);\n// all rights reserved. Authored by: Brian Ondov, Nicholas Bergman, and\n// Adam Phillippy\n// \n// This Software was prepared for the Department of Homeland Security\n// (DHS) by the Battelle National Biodefense Institute, LLC (BNBI) as\n// part of contract HSHQDC-07-C-00020 to manage and operate the National\n// Biodefense Analysis and Countermeasures Center (NBACC), a Federally\n// Funded Research and Development Center.\n// \n// Redistribution and use in source and binary forms, with or without\n// modification, are permitted provided that the following conditions are\n// met:\n// \n// * Redistributions of source code must retain the above copyright\n//   notice, this list of conditions and the following disclaimer.\n// \n// * Redistributions in binary form must reproduce the above copyright\n//   notice, this list of conditions and the following disclaimer in the\n//   documentation and/or other materials provided with the distribution.\n// \n// * Neither the name of the Battelle National Biodefense Institute nor\n//   the names of its contributors may be used to endorse or promote\n//   products derived from this software without specific prior written\n//   permission.\n// \n// THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS\n// "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT\n// LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR\n// A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT\n// HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL,\n// SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT\n// LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE,\n// DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY\n// THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT\n// (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\n// OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n// \n// \n// TRADEMARK LICENSE\n// \n// KRONA(TM) is a trademark of the Department of Homeland Security, and use\n// of the trademark is subject to the following conditions:\n// \n// * Distribution of the unchanged, official code/software using the\n//   KRONA(TM) mark is hereby permitted by the Department of Homeland\n//   Security, provided that the software is distributed without charge\n//   and modification.\n// \n// * Distribution of altered source code/software using the KRONA(TM) mark\n//   is not permitted unless written permission has been granted by the\n//   Department of Homeland Security.\n// \n// \n// FOR MORE INFORMATION VISIT\n// \n// https://github.com/marbl/Krona/wiki/\n// \n//-----------------------------------------------------------------------------\n}\nvar canvas;\nvar context;\nvar svg; // for snapshot mode\nvar collapse = true;\nvar collapseCheckBox;\nvar collapseLast;\nvar compress;\nvar compressCheckBox;\nvar maxAbsoluteDepthText;\nvar maxAbsoluteDepthButtonDecrease;\nvar maxAbsoluteDepthButtonIncrease;\nvar fontSize = 11;\nvar fontSizeText;\nvar fontSizeButtonDecrease;\nvar fontSizeButtonIncrease;\nvar fontSizeLast;\nvar radiusButtonDecrease;\nvar radiusButtonIncrease;\nvar shorten;\nvar shortenCheckBox;\nvar maxAbsoluteDepth;\nvar backButton;\nvar upButton;\nvar forwardButton;\nvar snapshotButton;\nvar snapshotMode = false;\nvar details;\nvar detailsName;\nvar search;\nvar searchResults;\nvar nSearchResults;\nvar useHueCheckBox;\nvar useHueDiv;\nvar datasetDropDown;\nvar datasetButtonLast;\nvar datasetButtonPrev;\nvar datasetButtonNext;\nvar keyControl;\nvar showKeys = true;\nvar linkButton;\nvar linkText;\nvar frame;\n// Node references. Note that the meanings of \'selected\' and \'focused\' are\n// swapped in the docs.\n//\nvar head; // the root of the entire tree\nvar selectedNode = 0; // the root of the current view\nvar focusNode = 0; // a node chosen for more info (single-click)\nvar highlightedNode = 0; // mouse hover node\nvar highlightingHidden = false;\nvar nodes = new Array();\nvar currentNodeID = 0; // to iterate while loading\nvar nodeHistory = new Array();\nvar nodeHistoryPosition = 0;\nvar dataEnabled = false; // true when supplemental files are present\n// store non-Krona GET variables so they can be passed on to links\n//\nvar getVariables = new Array();\n// selectedNodeLast is separate from the history, since we need to check\n// properties of the last node viewed when browsing through the history\n//\nvar selectedNodeLast = 0;\nvar zoomOut = false;\n// temporary zoom-in while holding the mouse button on a wedge\n//\nvar quickLook = false; // true when in quick look state\nvar mouseDown = false;\nvar mouseDownTime; // to detect mouse button hold\nvar quickLookHoldLength = 200;\nvar imageWidth;\nvar imageHeight;\nvar centerX;\nvar centerY;\nvar gRadius;\nvar updateViewNeeded = false;\n// Determines the angle that the pie chart starts at.  90 degrees makes the\n// center label consistent with the children.\n//\nvar rotationOffset = Math.PI / 2;\nvar buffer;\nvar bufferFactor = .1;\n// The maps are the small pie charts showing the current slice being viewed.\n//\nvar mapBuffer = 10;\nvar mapRadius = 0;\nvar maxMapRadius = 25;\nvar mapWidth = 150;\nvar maxLabelOverhang = Math.PI * 4.18;\n// Keys are the labeled boxes for slices in the highest level that are too thin\n// to label.\n//\nvar maxKeySizeFactor = 2; // will be multiplied by font size\nvar keySize;\nvar keys;\nvar keyBuffer = 10;\nvar currentKey;\nvar keyMinTextLeft;\nvar keyMinAngle;\nvar minRingWidthFactor = 5; // will be multiplied by font size\nvar maxPossibleDepth; // the theoretical max that can be displayed\nvar maxDisplayDepth; // the actual depth that will be displayed\nvar headerHeight = 0;//document.getElementById(\'options\').clientHeight;\nvar historySpacingFactor = 1.6; // will be multiplied by font size\nvar historyAlphaDelta = .25;\n// appearance\n//\nvar lineOpacity = 0.3;\nvar saturation = 0.5;\nvar lightnessBase = 0.6;\nvar lightnessMax = .8;\nvar thinLineWidth = .3;\nvar highlightLineWidth = 1.5;\nvar labelBoxBuffer = 6;\nvar labelBoxRounding = 15;\nvar labelWidthFudge = 1.05; // The width of unshortened labels are set slightly\n\t\t\t\t\t\t\t// longer than the name width so the animation\n\t\t\t\t\t\t\t// finishes faster.\nvar fontNormal;\nvar fontBold;\nvar fontFamily = \'sans-serif\';\n//var fontFaceBold = \'bold Arial\';\nvar nodeRadius;\nvar angleFactor;\nvar tickLength;\nvar compressedRadii;\n// colors\n//\nvar highlightFill = \'rgba(255, 255, 255, .3)\';\nvar colorUnclassified = \'rgb(220,220,220)\';\n// label staggering\n//\nvar labelOffsets; // will store the current offset at each depth\n//\n// This will store pointers to the last node that had a label in each offset (or "track") of a\n// each depth.  These will be used to shorten neighboring labels that would overlap.\n// The [nLabelNodes] index will store the last node with a radial label.\n// labelFirstNodes is the same, but to check for going all the way around and\n// overlapping the first labels.\n//\nvar labelLastNodes;\nvar labelFirstNodes;\n//\nvar nLabelOffsets = 3; // the number of offsets to use\nvar mouseX = -1;\nvar mouseY = -1;\nvar mouseXRel = -1;\nvar mouseYRel = -1;\n// tweening\n//\nvar progress = 0; // for tweening; goes from 0 to 1.\nvar progressLast = 0;\nvar tweenFactor = 0; // progress converted by a curve for a smoother effect.\nvar tweenLength = 850; // in ms\nvar tweenCurvature = 13;\n//\n// tweenMax is used to scale the sigmoid function so its range is [0,1] for the\n// domain [0,1]\n//\nvar tweenMax = 1 / (1 + Math.exp(-tweenCurvature / 2));\n//\nvar tweenStartTime;\n// for framerate debug\n//\nvar tweenFrames = 0;\nvar fpsDisplay = document.getElementById(\'frameRate\');\n// Arrays to translate xml attribute names into displayable attribute names\n//\nvar attributes = new Array();\n//\nvar magnitudeIndex; // the index of attribute arrays used for magnitude\nvar membersAssignedIndex;\nvar membersSummaryIndex;\n// For defining gradients\n//\nvar hueDisplayName;\nvar hueStopPositions;\nvar hueStopHues;\nvar hueStopText;\n// multiple datasets\n//\nvar currentDataset = 0;\nvar lastDataset = 0;\nvar datasets = 1;\nvar datasetNames;\nvar datasetSelectSize = 30;\nvar datasetAlpha = new Tween(0, 0);\nvar datasetWidths = new Array();\nvar datasetChanged;\nvar datasetSelectWidth = 50;\nwindow.onload = load;\nvar image;\nvar hiddenPattern;\nvar loadingImage;\nvar logoImage;\nfunction backingScale()\n{\n\tif (\'devicePixelRatio\' in window)\n\t{\n\t\tif (window.devicePixelRatio > 1)\n\t\t{\n\t\t\treturn window.devicePixelRatio;\n\t\t}\n\t}\n\t\n\treturn 1;\n}\nfunction resize()\n{\n\timageWidth = window.innerWidth;\n\timageHeight = window.innerHeight;\n\t\n\tif ( ! snapshotMode )\n\t{\n\t\tcontext.canvas.width = imageWidth * backingScale();\n\t\tcontext.canvas.height = imageHeight * backingScale();\n\t\tcontext.canvas.style.width = imageWidth + "px"\n\t\tcontext.canvas.style.height = imageHeight + "px"\n\t\tcontext.scale(backingScale(), backingScale());\n\t}\n\t\n\tif ( datasetDropDown )\n\t{\n\t\tvar ratio = \n\t\t\t(datasetDropDown.offsetTop + datasetDropDown.clientHeight) * 2 /\n\t\t\timageHeight;\n\t\t\n\t\tif ( ratio > 1 )\n\t\t{\n\t\t\tratio = 1;\n\t\t}\n\t\t\n\t\tratio = Math.sqrt(ratio);\n\t\t\n\t\tdatasetSelectWidth = \n\t\t\t(datasetDropDown.offsetLeft + datasetDropDown.clientWidth) * ratio;\n\t}\n\tvar leftMargin = datasets > 1 ? datasetSelectWidth + 30 : 0;\n\tvar minDimension = imageWidth - mapWidth - leftMargin > imageHeight ?\n\t\timageHeight :\n\t\timageWidth - mapWidth - leftMargin;\n\t\n\tmaxMapRadius = minDimension * .03;\n\tbuffer = minDimension * bufferFactor;\n\tmargin = minDimension * .015;\n\tcenterX = (imageWidth - mapWidth - leftMargin) / 2 + leftMargin;\n\tcenterY = imageHeight / 2;\n\tgRadius = minDimension / 2 - buffer;\n\t//context.font = \'11px sans-serif\';\n}\nfunction handleResize()\n{\n\tupdateViewNeeded = true;\n}\nfunction Attribute()\n{\n}\nfunction Tween(start, end)\n{\n\tthis.start = start;\n\tthis.end = end;\n\tthis.current = this.start;\n\t\n\tthis.current = function()\n\t{\n\t\tif ( progress == 1 || this.start == this.end )\n\t\t{\n\t\t\treturn this.end;\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn this.start + tweenFactor * (this.end - this.start);\n\t\t}\n\t};\n\t\n\tthis.setTarget = function(target)\n\t{\n\t\tthis.start = this.current();\n\t\tthis.end = target;\n\t}\n}\nfunction Node()\n{\n\tthis.id = currentNodeID;\n\tcurrentNodeID++;\n\tnodes[this.id] = this;\n\t\n\tthis.angleStart = new Tween(Math.PI, 0);\n\tthis.angleEnd = new Tween(Math.PI, 0);\n\tthis.radiusInner = new Tween(1, 1);\n\tthis.labelRadius = new Tween(1, 1);\n\tthis.labelWidth = new Tween(0, 0);\n\tthis.scale = new Tween(1, 1); // TEMP\n\tthis.radiusOuter = new Tween(1, 1);\n\t\n\tthis.r = new Tween(255, 255);\n\tthis.g = new Tween(255, 255);\n\tthis.b = new Tween(255, 255);\n\t\n\tthis.alphaLabel = new Tween(0, 1);\n\tthis.alphaLine = new Tween(0, 1);\n\tthis.alphaArc = new Tween(0, 0);\n\tthis.alphaWedge = new Tween(0, 1);\n\tthis.alphaOther = new Tween(0, 1);\n\tthis.alphaPattern = new Tween(0, 0);\n\tthis.children = Array();\n\tthis.parent = 0;\n\t\n\tthis.attributes = new Array(attributes.length);\n\t\n\tthis.addChild = function(child)\n\t{\n\t\tthis.children.push(child);\n\t};\n\t\n\tthis.addLabelNode = function(depth, labelOffset)\n\t{\n\t\tif ( labelHeadNodes[depth][labelOffset] == 0 )\n\t\t{\n\t\t\t// this will become the head node for this list\n\t\t\t\n\t\t\tlabelHeadNodes[depth][labelOffset] = this;\n\t\t\tthis.labelPrev = this;\n\t\t}\n\t\t\n\t\tvar head = labelHeadNodes[depth][labelOffset];\n\t\t\n\t\tthis.labelNext = head;\n\t\tthis.labelPrev = head.labelPrev;\n\t\thead.labelPrev.labelNext = this;\n\t\thead.labelPrev = this;\n\t}\n\t\n\tthis.canDisplayDepth = function()\n\t{\n\t\t// whether this node is at a depth that can be displayed, according\n\t\t// to the max absolute depth\n\t\t\n\t\treturn this.depth <= maxAbsoluteDepth;\n\t}\n\t\n\tthis.canDisplayHistory = function()\n\t{\n\t\tvar radiusInner;\n\t\t\n\t\tif ( compress )\n\t\t{\n\t\t\tradiusInner = compressedRadii[0];\n\t\t}\n\t\telse\n\t\t{\n\t\t\tradiusInner = nodeRadius;\n\t\t}\n\t\t\n\t\treturn (\n\t\t\t-this.labelRadius.end * gRadius +\n\t\t\thistorySpacingFactor * fontSize / 2 <\n\t\t\tradiusInner * gRadius\n\t\t\t);\n\t}\n\t\n\tthis.canDisplayLabelCurrent = function()\n\t{\n\t\treturn (\n\t\t\t(this.angleEnd.current() - this.angleStart.current()) *\n\t\t\t(this.radiusInner.current() * gRadius + gRadius) >=\n\t\t\tminWidth());\n\t}\n\t\n\tthis.checkHighlight = function()\n\t{\n\t\tif ( this.children.length == 0 && this == focusNode )\n\t\t{\n\t\t\t//return false;\n\t\t}\n\t\t\n\t\tif ( this.hide )\n\t\t{\n\t\t\treturn false;\n\t\t}\n\t\t\n\t\tif ( this.radiusInner.end == 1 )\n\t\t{\n\t\t\t// compressed to the outside; don\'t check\n\t\t\t\n\t\t\treturn false;\n\t\t}\n\t\t\n\t\tvar highlighted = false;\n\t\t\n\t\tvar angleStartCurrent = this.angleStart.current() + rotationOffset;\n\t\tvar angleEndCurrent = this.angleEnd.current() + rotationOffset;\n\t\tvar radiusInner = this.radiusInner.current() * gRadius;\n\t\t\n\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\thighlighted = this.children[i].checkHighlight();\n\t\t\t\n\t\t\tif ( highlighted )\n\t\t\t{\n\t\t\t\treturn true;\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( this.radial )\n\t\t{\n\t\t\tvar angleText = (angleStartCurrent + angleEndCurrent) / 2;\n\t\t\tvar radiusText = (gRadius + radiusInner) / 2;\n\t\t\t\n\t\t\tcontext.rotate(angleText);\n\t\t\tcontext.beginPath();\n\t\t\tcontext.moveTo(radiusText, -fontSize);\n\t\t\tcontext.lineTo(radiusText, fontSize);\n\t\t\tcontext.lineTo(radiusText + centerX, fontSize);\n\t\t\tcontext.lineTo(radiusText + centerX, -fontSize);\n\t\t\tcontext.closePath();\n\t\t\tcontext.rotate(-angleText);\n\t\t\t\n\t\t\tif ( context.isPointInPath(mouseXRel, mouseYRel) )\n\t\t\t{\n\t\t\t\tvar label = String(this.getPercentage()) + \'%\' + \'   \' + this.name;\n\t\t\t\t\n\t\t\t\tif ( this.searchResultChildren() )\n\t\t\t    {\n\t\t\t\t\tlabel += searchResultString(this.searchResultChildren());\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif\n\t\t\t\t(\n\t\t\t\t\tMath.sqrt((mouseXRel) * (mouseXRel) + (mouseYRel) * (mouseYRel)) / backingScale() <\n\t\t\t\t\tradiusText + measureText(label)\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\thighlighted = true;\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t    for ( var i = 0; i < this.hiddenLabels.length; i++ )\n\t\t    {\n\t\t        var hiddenLabel = this.hiddenLabels[i];\n\t\t        \n\t\t\t\tcontext.rotate(hiddenLabel.angle);\n\t\t\t\tcontext.beginPath();\n\t\t\t\tcontext.moveTo(gRadius, -fontSize);\n\t\t\t\tcontext.lineTo(gRadius, fontSize);\n\t\t\t\tcontext.lineTo(gRadius + centerX, fontSize);\n\t\t\t\tcontext.lineTo(gRadius + centerX, -fontSize);\n\t\t\t\tcontext.closePath();\n\t\t\t\tcontext.rotate(-hiddenLabel.angle);\n\t\t\t\t\n\t\t\t\tif ( context.isPointInPath(mouseXRel, mouseYRel) )\n\t\t\t\t{\n\t\t\t\t\tvar label = String(hiddenLabel.value) + \' more\';\n\t\t\t\t\t\n\t\t\t\t\tif ( hiddenLabel.search )\n\t\t\t\t    {\n\t\t\t\t\t\tlabel += searchResultString(hiddenLabel.search);\n\t\t\t\t\t}\n\t\t\t\t\t\n\t\t\t\t\tif\n\t\t\t\t\t(\n\t\t\t\t\t\tMath.sqrt((mouseXRel) * (mouseXRel) + (mouseYRel) * (mouseYRel)) / backingScale() <\n\t\t\t\t\t\tgRadius + fontSize + measureText(label)\n\t\t\t\t\t)\n\t\t\t\t\t{\n\t\t\t\t\t\thighlighted = true;\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( ! highlighted && this != selectedNode && ! this.getCollapse() )\n\t\t{\n\t\t\tcontext.beginPath();\n\t\t\tcontext.arc(0, 0, radiusInner, angleStartCurrent, angleEndCurrent, false);\n\t\t\tcontext.arc(0, 0, gRadius, angleEndCurrent, angleStartCurrent, true);\n\t\t\tcontext.closePath();\n\t\t\t\n\t\t\tif ( context.isPointInPath(mouseXRel, mouseYRel) )\n\t\t\t{\n\t\t\t\thighlighted = true;\n\t\t\t}\n\t\t\t\n\t\t\tif\n\t\t\t(\n\t\t\t\t! highlighted &&\n\t\t\t\t(angleEndCurrent - angleStartCurrent) *\n\t\t\t\t(radiusInner + gRadius) <\n\t\t\t\tminWidth() &&\n\t\t\t\tthis.getDepth() == selectedNode.getDepth() + 1\n\t\t\t)\n\t\t\t{\n\t\t\t\tif ( showKeys && this.checkHighlightKey() )\n\t\t\t\t{\n\t\t\t\t\thighlighted = true;\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( highlighted )\n\t\t{\n\t\t\tif ( this != highlightedNode )\n\t\t\t{\n\t\t\t//\tdocument.body.style.cursor=\'pointer\';\n\t\t\t}\n\t\t\t\n\t\t\thighlightedNode = this;\n\t\t}\n\t\t\n\t\treturn highlighted;\n\t}\n\t\n\tthis.checkHighlightCenter = function()\n\t{\n\t\tif ( ! this.canDisplayHistory() )\n\t\t{\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tvar cx = centerX;\n\t\tvar cy = centerY - this.labelRadius.end * gRadius;\n\t\t//var dim = context.measureText(this.name);\n\t\t\n\t\tvar width = this.nameWidth;\n\t\t\n\t\tif ( this.searchResultChildren() )\n\t\t{\n\t\t\tvar results = searchResultString(this.searchResultChildren());\n\t\t\tvar dim = context.measureText(results);\n\t\t\twidth += dim.width;\n\t\t}\n\t\t\n\t\tif\n\t\t(\n\t\t\tmouseX > cx - width / 2 &&\n\t\t\tmouseX < cx + width / 2 &&\n\t\t\tmouseY > cy - historySpacingFactor * fontSize / 2 &&\n\t\t\tmouseY < cy + historySpacingFactor * fontSize / 2\n\t\t)\n\t\t{\n\t\t\thighlightedNode = this;\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tif ( this.getParent() )\n\t\t{\n\t\t\tthis.getParent().checkHighlightCenter();\n\t\t}\n\t}\n\t\n\tthis.checkHighlightKey = function()\n\t{\n\t\tvar offset = keyOffset();\n\t\t\n\t\tvar xMin = imageWidth - keySize - margin - this.keyNameWidth - keyBuffer;\n\t\tvar xMax = imageWidth - margin;\n\t\tvar yMin = offset;\n\t\tvar yMax = offset + keySize;\n\t\t\n\t\tcurrentKey++;\n\t\t\n\t\treturn (\n\t\t\tmouseX > xMin &&\n\t\t\tmouseX < xMax &&\n\t\t\tmouseY > yMin &&\n\t\t\tmouseY < yMax);\n\t}\n\t\n\tthis.checkHighlightMap = function()\n\t{\n\t\tif ( this.parent )\n\t\t{\n\t\t\tthis.parent.checkHighlightMap();\n\t\t}\n\t\t\n\t\tif ( this.getCollapse() || this == focusNode )\n\t\t{\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tvar box = this.getMapPosition();\n\t\t\n\t\tif\n\t\t(\n\t\t\tmouseX > box.x - mapRadius &&\n\t\t\tmouseX < box.x + mapRadius &&\n\t\t\tmouseY > box.y - mapRadius &&\n\t\t\tmouseY < box.y + mapRadius\n\t\t)\n\t\t{\n\t\t\thighlightedNode = this;\n\t\t}\n\t}\n\t\n/*\tthis.collapse = function()\n\t{\n\t\tfor (var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\tthis.children[i] = this.children[i].collapse();\n\t\t}\n\t\t\n\t\tif\n\t\t(\n\t\t\tthis.children.length == 1 &&\n\t\t\tthis.children[0].magnitude == this.magnitude\n\t\t)\n\t\t{\n\t\t\tthis.children[0].parent = this.parent;\n\t\t\tthis.children[0].getDepth() = this.parent.getDepth() + 1;\n\t\t\treturn this.children[0];\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn this;\n\t\t}\n\t}\n*/\t\n\tthis.draw = function(labelMode, selected, searchHighlighted)\n\t{\n\t\tvar depth = this.getDepth() - selectedNode.getDepth() + 1;\n//\t\tvar hidden = false;\n\t\t\n\t\tif ( selectedNode == this )\n\t\t{\n\t\t\tselected = true;\n\t\t}\n\t\t\n\t\tvar angleStartCurrent = this.angleStart.current() + rotationOffset;\n\t\tvar angleEndCurrent = this.angleEnd.current() + rotationOffset;\n\t\tvar radiusInner = this.radiusInner.current() * gRadius;\n\t\tvar canDisplayLabelCurrent = this.canDisplayLabelCurrent();\n\t\tvar hiddenSearchResults = false;\n\t\t\n/*\t\tif ( ! this.hide )\n\t\t{\n\t\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t\t{\n\t\t\t\tif ( this.children[i].hide && this.children[i].searchResults )\n\t\t\t\t{\n\t\t\t\t\thiddenSearchResults = true;\n\t\t\t\t}\n\t\t\t}\n\t\t}\n*/\t\t\n\t\tvar drawChildren =\n\t\t\t( ! this.hide || ! this.hidePrev && progress < 1 ) &&\n\t\t\t( ! this.hideAlone || ! this.hideAlonePrev && progress < 1 );\n\t\t\n//\t\tif ( this.alphaWedge.current() > 0 || this.alphaLabel.current() > 0 )\n\t\t{\n\t\t\tvar lastChildAngleEnd = angleStartCurrent;\n\t\t\t\n\t\t\tif ( this.hasChildren() )//canDisplayChildren )\n\t\t\t{\n\t\t\t\tlastChildAngleEnd =\n\t\t\t\t\tthis.children[this.children.length - 1].angleEnd.current()\n\t\t\t\t\t+ rotationOffset;\n\t\t\t}\n\t\t\t\n\t\t\tif ( labelMode )\n\t\t\t{\n\t\t\t\tvar drawRadial =\n\t\t\t\t!(\n\t\t\t\t\tthis.parent &&\n\t\t\t\t\tthis.parent != selectedNode &&\n\t\t\t\t\tangleEndCurrent == this.parent.angleEnd.current() + rotationOffset\n\t\t\t\t);\n\t\t\t\t\n\t\t\t\t//if ( angleStartCurrent != angleEndCurrent )\n\t\t\t\t{\n\t\t\t\t\tthis.drawLines(angleStartCurrent, angleEndCurrent, radiusInner, drawRadial, selected);\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tvar alphaOtherCurrent = this.alphaOther.current();\n\t\t\t\tvar childRadiusInner;\n\t\t\t\t\n\t\t\t\tif ( this == selectedNode || alphaOtherCurrent )\n\t\t\t\t{\n\t\t\t\t\tchildRadiusInner =\n\t\t\t\t\t\tthis.children.length ?\n\t\t\t\t\t\t\tthis.children[this.children.length - 1].radiusInner.current() * gRadius\n\t\t\t\t\t\t: radiusInner\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif ( this == selectedNode )\n\t\t\t\t{\n\t\t\t\t\tthis.drawReferenceRings(childRadiusInner);\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif\n\t\t\t\t(\n\t\t\t\t\tselected &&\n\t\t\t\t\t! searchHighlighted &&\n\t\t\t\t\tthis != selectedNode &&\n\t\t\t\t\t(\n\t\t\t\t\t\tthis.isSearchResult ||\n\t\t\t\t\t\tthis.hideAlone && this.searchResultChildren() ||\n\t\t\t\t\t\tfalse\n//\t\t\t\t\t\tthis.hide &&\n//\t\t\t\t\t\tthis.containsSearchResult\n\t\t\t\t\t)\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\tcontext.globalAlpha = this.alphaWedge.current();\n\t\t\t\t\t\n\t\t\t\t\tdrawWedge\n\t\t\t\t\t(\n\t\t\t\t\t\tangleStartCurrent,\n\t\t\t\t\t\tangleEndCurrent,\n\t\t\t\t\t\tradiusInner,\n\t\t\t\t\t\tgRadius,\n\t\t\t\t\t\thighlightFill,\n\t\t\t\t\t\t0,\n\t\t\t\t\t\ttrue\n\t\t\t\t\t);\n\t\t\t\t\t\n\t\t\t\t\tif\n\t\t\t\t\t(\n\t\t\t\t\t\tthis.keyed &&\n\t\t\t\t\t\t! showKeys &&\n\t\t\t\t\t\tthis.searchResults &&\n\t\t\t\t\t\t! searchHighlighted &&\n\t\t\t\t\t\tthis != highlightedNode &&\n\t\t\t\t\t\tthis != focusNode\n\t\t\t\t\t)\n\t\t\t\t\t{\n\t\t\t\t\t\tvar angle = (angleEndCurrent + angleStartCurrent) / 2;\n\t\t\t\t\t\tthis.drawLabel(angle, true, false, true, true);\n\t\t\t\t\t}\n\t\t\t\t\t\n\t\t\t\t\t//this.drawHighlight(false);\n\t\t\t\t\tsearchHighlighted = true;\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif\n\t\t\t\t(\n\t\t\t\t\tthis == selectedNode ||\n//\t\t\t\t\ttrue\n\t\t\t\t\t//(canDisplayLabelCurrent) &&\n\t\t\t\t\tthis != highlightedNode &&\n\t\t\t\t\tthis != focusNode\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\tif ( this.radial != this.radialPrev && this.alphaLabel.end == 1 )\n\t\t\t\t\t{\n\t\t\t\t\t\tcontext.globalAlpha = tweenFactor;\n\t\t\t\t\t}\n\t\t\t\t\telse\n\t\t\t\t\t{\n\t\t\t\t\t\tcontext.globalAlpha = this.alphaLabel.current();\n\t\t\t\t\t}\n\t\t\t\t\t\n\t\t\t\t\tthis.drawLabel\n\t\t\t\t\t(\n\t\t\t\t\t\t(angleStartCurrent + angleEndCurrent) / 2,\n\t\t\t\t\t\tthis.hideAlone && this.searchResultChildren() ||\n\t\t\t\t\t\t(this.isSearchResult || hiddenSearchResults) && selected,\n\t\t\t\t\t\tthis == selectedNode && ! this.radial,\n\t\t\t\t\t\tselected,\n\t\t\t\t\t\tthis.radial\n\t\t\t\t\t);\n\t\t\t\t\t\n\t\t\t\t\tif ( this.radial != this.radialPrev && this.alphaLabel.start == 1 && progress < 1 )\n\t\t\t\t\t{\n\t\t\t\t\t\tcontext.globalAlpha = 1 - tweenFactor;\n\t\t\t\t\t\t\n\t\t\t\t\t\tthis.drawLabel\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\t(angleStartCurrent + angleEndCurrent) / 2,\n\t\t\t\t\t\t\t(this.isSearchResult || hiddenSearchResults) && selected,\n\t\t\t\t\t\t\tthis == selectedNodeLast && ! this.radialPrev,\n\t\t\t\t\t\t\tselected,\n\t\t\t\t\t\t\tthis.radialPrev\n\t\t\t\t\t\t);\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif\n\t\t\t\t(\n\t\t\t\t\talphaOtherCurrent &&\n\t\t\t\t\tlastChildAngleEnd != null\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\tif\n\t\t\t\t\t(\n\t\t\t\t\t\t(angleEndCurrent - lastChildAngleEnd) *\n\t\t\t\t\t\t(childRadiusInner + gRadius) >=\n\t\t\t\t\t\tminWidth()\n\t\t\t\t\t)\n\t\t\t\t\t{\n\t\t\t\t\t\t//context.font = fontNormal;\n\t\t\t\t\t\tcontext.globalAlpha = this.alphaOther.current();\n\t\t\t\t\t\t\n\t\t\t\t\t\tdrawTextPolar\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\tthis.getUnclassifiedText(),\n\t\t\t\t\t\t\tthis.getUnclassifiedPercentage(),\n\t\t\t\t\t\t\t(lastChildAngleEnd + angleEndCurrent) / 2,\n\t\t\t\t\t\t\t(childRadiusInner + gRadius) / 2,\n\t\t\t\t\t\t\ttrue,\n\t\t\t\t\t\t\tfalse,\n\t\t\t\t\t\t\tfalse,\n\t\t\t\t\t\t\t0,\n\t\t\t\t\t\t\t0\n\t\t\t\t\t\t);\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif ( this == selectedNode && this.keyUnclassified && showKeys )\n\t\t\t\t{\n\t\t\t\t\tthis.drawKey\n\t\t\t\t\t(\n\t\t\t\t\t\t(lastChildAngleEnd + angleEndCurrent) / 2,\n\t\t\t\t\t\tfalse,\n\t\t\t\t\t\tfalse\n\t\t\t\t\t);\n\t\t\t\t}\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tvar alphaWedgeCurrent = this.alphaWedge.current();\n\t\t\t\t\n\t\t\t\tif ( alphaWedgeCurrent || this.alphaOther.current() )\n\t\t\t\t{\n\t\t\t\t\tvar currentR = this.r.current();\n\t\t\t\t\tvar currentG = this.g.current();\n\t\t\t\t\tvar currentB = this.b.current();\n\t\t\t\t\t\t\n\t\t\t\t\tvar fill = rgbText(currentR, currentG, currentB);\n\t\t\t\t\t\n\t\t\t\t\tvar radiusOuter;\n\t\t\t\t\tvar lastChildAngle;\n\t\t\t\t\tvar truncateWedge =\n\t\t\t\t\t(\n\t\t\t\t\t\t(this.hasChildren() || this == selectedNode ) &&\n\t\t\t\t\t\t! this.keyed &&\n\t\t\t\t\t\t(compress || depth < maxDisplayDepth) &&\n\t\t\t\t\t\tdrawChildren\n\t\t\t\t\t);\n\t\t\t\t\t\n\t\t\t\t\tif ( truncateWedge )\n\t\t\t\t\t{\n\t\t\t\t\t\tradiusOuter = this.children.length ? this.children[0].radiusInner.current() * gRadius : radiusInner;\n\t\t\t\t\t}\n\t\t\t\t\telse\n\t\t\t\t\t{\n\t\t\t\t\t\tradiusOuter = gRadius;\n\t\t\t\t\t}\n\t\t\t\t\t/*\n\t\t\t\t\tif ( this.hasChildren() )\n\t\t\t\t\t{\n\t\t\t\t\t\tradiusOuter = this.children[0].getUncollapsed().radiusInner.current() * gRadius + 1;\n\t\t\t\t\t}\n\t\t\t\t\telse\n\t\t\t\t\t{ // TEMP\n\t\t\t\t\t\tradiusOuter = radiusInner + nodeRadius * gRadius;\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( radiusOuter > gRadius )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tradiusOuter = gRadius;\n\t\t\t\t\t\t}\n\t\t\t\t\t}\n\t\t\t\t\t*/\n\t\t\t\t\tcontext.globalAlpha = alphaWedgeCurrent;\n\t\t\t\t\t\n\t\t\t\t\tif ( radiusInner != radiusOuter || truncateWedge )\n\t\t\t\t\t{\n\t\t\t\t\t\tdrawWedge\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\tangleStartCurrent,\n\t\t\t\t\t\t\tangleEndCurrent,\n\t\t\t\t\t\t\tradiusInner,\n\t\t\t\t\t\t\tradiusOuter,//this.radiusOuter.current() * gRadius,\n\t\t\t\t\t\t\t//\'rgba(0, 200, 0, .1)\',\n\t\t\t\t\t\t\tfill,\n\t\t\t\t\t\t\tthis.alphaPattern.current()\n\t\t\t\t\t\t);\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( truncateWedge )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\t// fill in the extra space if the sum of our childrens\'\n\t\t\t\t\t\t\t// magnitudes is less than ours\n\t\t\t\t\t\t\t\n\t\t\t\t\t\t\tif ( lastChildAngleEnd < angleEndCurrent )//&& false) // TEMP\n\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\t\tif ( radiusOuter > 1 )\n\t\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\t\t\t// overlap slightly to hide the seam\n\t\t\t\t\t\t\t\t\t\n\t//\t\t\t\t\t\t\t\tradiusOuter -= 1;\n\t\t\t\t\t\t\t\t}\n\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t\t\tif ( alphaWedgeCurrent < 1 )\n\t\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\t\t\tcontext.globalAlpha = this.alphaOther.current();\n\t\t\t\t\t\t\t\t\tdrawWedge\n\t\t\t\t\t\t\t\t\t(\n\t\t\t\t\t\t\t\t\t\tlastChildAngleEnd,\n\t\t\t\t\t\t\t\t\t\tangleEndCurrent,\n\t\t\t\t\t\t\t\t\t\tradiusOuter,\n\t\t\t\t\t\t\t\t\t\tgRadius,\n\t\t\t\t\t\t\t\t\t\tcolorUnclassified,\n\t\t\t\t\t\t\t\t\t\t0\n\t\t\t\t\t\t\t\t\t);\n\t\t\t\t\t\t\t\t\tcontext.globalAlpha = alphaWedgeCurrent;\n\t\t\t\t\t\t\t\t}\n\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t\t\tdrawWedge\n\t\t\t\t\t\t\t\t(\n\t\t\t\t\t\t\t\t\tlastChildAngleEnd,\n\t\t\t\t\t\t\t\t\tangleEndCurrent,\n\t\t\t\t\t\t\t\t\tradiusOuter,\n\t\t\t\t\t\t\t\t\tgRadius,//this.radiusOuter.current() * gRadius,\n\t\t\t\t\t\t\t\t\t//\'rgba(200, 0, 0, .1)\',\n\t\t\t\t\t\t\t\t\tfill,\n\t\t\t\t\t\t\t\t\tthis.alphaPattern.current()\n\t\t\t\t\t\t\t\t);\n\t\t\t\t\t\t\t}\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( radiusOuter < gRadius )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\t// patch up the seam\n\t\t\t\t\t\t\t//\n\t\t\t\t\t\t\tcontext.beginPath();\n\t\t\t\t\t\t\tcontext.arc(0, 0, radiusOuter, angleStartCurrent/*lastChildAngleEnd*/, angleEndCurrent, false);\n\t\t\t\t\t\t\tcontext.strokeStyle = fill;\n\t\t\t\t\t\t\tcontext.lineWidth = 1;\n\t\t\t\t\t\t\tcontext.stroke();\n\t\t\t\t\t\t}\n\t\t\t\t\t}\n\t\t\t\t\t\n\t\t\t\t\tif ( this.keyed && selected && showKeys )//&& progress == 1 )\n\t\t\t\t\t{\n\t\t\t\t\t\tthis.drawKey\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\t(angleStartCurrent + angleEndCurrent) / 2,\n\t\t\t\t\t\t\t(\n\t\t\t\t\t\t\t\tthis == highlightedNode ||\n\t\t\t\t\t\t\t\tthis == focusNode ||\n\t\t\t\t\t\t\t\tthis.searchResults\n\t\t\t\t\t\t\t),\n\t\t\t\t\t\t\tthis == highlightedNode || this == focusNode\n\t\t\t\t\t\t);\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\t\n\t\tthis.hiddenLabels = Array();\n\t\t\n\t\tif ( drawChildren )\n\t\t{\n\t\t\t// draw children\n\t\t\t//\n\t\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t\t{\n\t\t\t\tif ( this.drawHiddenChildren(i, selected, labelMode, searchHighlighted) )\n\t\t\t\t{\n\t\t\t\t\ti = this.children[i].hiddenEnd;\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tthis.children[i].draw(labelMode, selected, searchHighlighted);\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t};\n\t\n\tthis.drawHiddenChildren = function\n\t(\n\t\tfirstHiddenChild,\n\t\tselected,\n\t\tlabelMode,\n\t\tsearchHighlighted\n\t)\n\t{\n\t\tvar firstChild = this.children[firstHiddenChild];\n\t\t\n\t\tif ( firstChild.hiddenEnd == null || firstChild.radiusInner.current() == 1 )\n\t\t{\n\t\t\treturn false;\n\t\t}\n\t\t\n\t\tfor ( var i = firstHiddenChild; i < firstChild.hiddenEnd; i++ )\n\t\t{\n\t\t\tif ( ! this.children[i].hide || ! this.children[i].hidePrev && progress < 1 )\n\t\t\t{\n\t\t\t\treturn false;\n\t\t\t}\n\t\t}\n\t\t\n\t\tvar angleStart = firstChild.angleStart.current() + rotationOffset;\n\t\tvar lastChild = this.children[firstChild.hiddenEnd];\n\t\tvar angleEnd = lastChild.angleEnd.current() + rotationOffset;\n\t\tvar radiusInner = gRadius * firstChild.radiusInner.current();\n\t\tvar hiddenChildren = firstChild.hiddenEnd - firstHiddenChild + 1;\n\t\t\n\t\tif ( labelMode )\n\t\t{\n\t\t\tvar hiddenSearchResults = 0;\n\t\t\t\n\t\t\tfor ( var i = firstHiddenChild; i <= firstChild.hiddenEnd; i++ )\n\t\t\t{\n\t\t\t\thiddenSearchResults += this.children[i].searchResults;\n\t\t\t\t\n\t\t\t\tif ( this.children[i].magnitude == 0 )\n\t\t\t\t{\n\t\t\t\t\thiddenChildren--;\n\t\t\t\t}\n\t\t\t}\n\t\t\t\n\t\t\tif\n\t\t\t(\n\t\t\t\tselected &&\n\t\t\t\t(angleEnd - angleStart) * \n\t\t\t\t(gRadius + gRadius) >=\n\t\t\t\tminWidth() ||\n\t\t\t\tthis == highlightedNode &&\n\t\t\t\thiddenChildren ||\n\t\t\t\thiddenSearchResults\n\t\t\t)\n\t\t\t{\n\t\t\t\tcontext.globalAlpha = this.alphaWedge.current();\n\t\t\t\t\n\t\t\t\tthis.drawHiddenLabel\n\t\t\t\t(\n\t\t\t\t\tangleStart,\n\t\t\t\t\tangleEnd,\n\t\t\t\t\thiddenChildren,\n\t\t\t\t\thiddenSearchResults\n\t\t\t\t);\n\t\t\t}\n\t\t}\n\t\t\n\t\tvar drawWedges = true;\n\t\t\n\t\tfor ( var i = firstHiddenChild; i <= firstChild.hiddenEnd; i++ )\n\t\t{\n\t\t\t// all hidden children must be completely hidden to draw together\n\t\t\t\n\t\t\tif ( this.children[i].alphaPattern.current() != this.children[i].alphaWedge.current() )\n\t\t\t{\n\t\t\t\tdrawWedges = false;\n\t\t\t\tbreak;\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( labelMode )\n\t\t{\n\t\t\tif ( drawWedges )\n\t\t\t{\n\t\t\t\tvar drawRadial = (angleEnd < this.angleEnd.current() + rotationOffset);\n\t\t\t\tthis.drawLines(angleStart, angleEnd, radiusInner, drawRadial);\n\t\t\t}\n\t\t\t\n\t\t\tif ( hiddenSearchResults && ! searchHighlighted )\n\t\t\t{\n\t\t\t\tdrawWedge\n\t\t\t\t(\n\t\t\t\t\tangleStart,\n\t\t\t\t\tangleEnd,\n\t\t\t\t\tradiusInner,\n\t\t\t\t\tgRadius,//this.radiusOuter.current() * gRadius,\n\t\t\t\t\thighlightFill,\n\t\t\t\t\t0,\n\t\t\t\t\ttrue\n\t\t\t\t);\n\t\t\t}\n\t\t}\n\t\telse if ( drawWedges )\n\t\t{\n\t\t\tcontext.globalAlpha = this.alphaWedge.current();\n\t\t\t\n\t\t\tvar fill = rgbText\n\t\t\t(\n\t\t\t\tfirstChild.r.current(),\n\t\t\t\tfirstChild.g.current(),\n\t\t\t\tfirstChild.b.current()\n\t\t\t);\n\t\t\t\n\t\t\tdrawWedge\n\t\t\t(\n\t\t\t\tangleStart,\n\t\t\t\tangleEnd,\n\t\t\t\tradiusInner,\n\t\t\t\tgRadius,//this.radiusOuter.current() * gRadius,\n\t\t\t\tfill,\n\t\t\t\tcontext.globalAlpha,\n\t\t\t\tfalse\n\t\t\t);\n\t\t}\n\t\t\n\t\treturn drawWedges;\n\t}\n\t\n\tthis.drawHiddenLabel = function(angleStart, angleEnd, value, hiddenSearchResults)\n\t{\n\t\tvar textAngle = (angleStart + angleEnd) / 2;\n\t\tvar labelRadius = gRadius + fontSize;//(radiusInner + radius) / 2;\n\t\t\n\t\tvar hiddenLabel = Array();\n\t\t\n\t\thiddenLabel.value = value;\n\t\thiddenLabel.angle = textAngle;\n\t\thiddenLabel.search = hiddenSearchResults;\n\t\t\n\t\tthis.hiddenLabels.push(hiddenLabel);\n\t\t\n\t\tdrawTick(gRadius - fontSize * .75, fontSize * 1.5, textAngle);\n\t\tdrawTextPolar\n\t\t(\n\t\t\tvalue.toString() + \' more\',\n\t\t\t0, // inner text\n\t\t\ttextAngle,\n\t\t\tlabelRadius,\n\t\t\ttrue, // radial\n\t\t\thiddenSearchResults, // bubble\n\t\t\tthis == highlightedNode || this == focusNode, // bold\n\t\t\tfalse,\n\t\t\thiddenSearchResults\n\t\t);\n\t}\n\t\n\tthis.drawHighlight = function(bold)\n\t{\n\t\tvar angleStartCurrent = this.angleStart.current() + rotationOffset;\n\t\tvar angleEndCurrent = this.angleEnd.current() + rotationOffset;\n\t\tvar radiusInner = this.radiusInner.current() * gRadius;\n\t\t\n\t\t//this.setHighlightStyle();\n\t\t\n\t\tif ( this == focusNode && this == highlightedNode && this.hasChildren() )\n\t\t{\n//\t\t\tcontext.fillStyle = "rgba(255, 255, 255, .3)";\n\t\t\tarrow\n\t\t\t(\n\t\t\t\tangleStartCurrent,\n\t\t\t\tangleEndCurrent,\n\t\t\t\tradiusInner\n\t\t\t);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tdrawWedge\n\t\t\t(\n\t\t\t\tangleStartCurrent,\n\t\t\t\tangleEndCurrent,\n\t\t\t\tradiusInner,\n\t\t\t\tgRadius,\n\t\t\t\thighlightFill,\n\t\t\t\t0,\n\t\t\t\ttrue\n\t\t\t);\n\t\t}\n\t\t\n\t\t// check if hidden children should be highlighted\n\t\t//\n\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\tif\n\t\t\t(\n\t\t\t\tthis.children[i].getDepth() - selectedNode.getDepth() + 1 <=\n\t\t\t\tmaxDisplayDepth &&\n\t\t\t\tthis.children[i].hiddenEnd != null\n\t\t\t)\n\t\t\t{\n\t\t\t\tvar firstChild = this.children[i];\n\t\t\t\tvar lastChild = this.children[firstChild.hiddenEnd];\n\t\t\t\tvar hiddenAngleStart = firstChild.angleStart.current() + rotationOffset;\n\t\t\t\tvar hiddenAngleEnd = lastChild.angleEnd.current() + rotationOffset;\n\t\t\t\tvar hiddenRadiusInner = gRadius * firstChild.radiusInner.current();\n\t\t\t\t\n\t\t\t\tdrawWedge\n\t\t\t\t(\n\t\t\t\t\thiddenAngleStart,\n\t\t\t\t\thiddenAngleEnd,\n\t\t\t\t\thiddenRadiusInner,\n\t\t\t\t\tgRadius,\n\t\t\t\t\t\'rgba(255, 255, 255, .3)\',\n\t\t\t\t\t0,\n\t\t\t\t\ttrue\n\t\t\t\t);\n\t\t\t\t\n\t\t\t\tif ( false && ! this.searchResults )\n\t\t\t\t{\n\t\t\t\t\tthis.drawHiddenLabel\n\t\t\t\t\t(\n\t\t\t\t\t\thiddenAngleStart,\n\t\t\t\t\t\thiddenAngleEnd,\n\t\t\t\t\t\tfirstChild.hiddenEnd - i + 1\n\t\t\t\t\t);\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\ti = firstChild.hiddenEnd;\n\t\t\t}\n\t\t}\n\t\t\n//\t\t\tcontext.strokeStyle = \'black\';\n\t\tcontext.fillStyle = \'black\';\n\t\t\n\t\tvar highlight = ! ( progress < 1 && zoomOut && this == selectedNodeLast );\n\t\t\n\t\tvar angle = (angleEndCurrent + angleStartCurrent) / 2;\n\t\t\n\t\tif ( ! (this.keyed && showKeys) )\n\t\t{\n\t\t\tthis.drawLabel(angle, true, bold, true, this.radial);\n\t\t}\n\t}\n\t\n\tthis.drawHighlightCenter = function()\n\t{\n\t\tif ( ! this.canDisplayHistory() )\n\t\t{\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tcontext.lineWidth = highlightLineWidth;\n\t\tcontext.strokeStyle = \'black\';\n\t\tcontext.fillStyle = "rgba(255, 255, 255, .6)";\n\t\t\n\t\tcontext.fillStyle = \'black\';\n\t\tthis.drawLabel(3 * Math.PI / 2, true, true, false);\n\t\tcontext.font = fontNormal;\n\t}\n\t\n\tthis.drawKey = function(angle, highlight, bold)\n\t{\n\t\tvar offset = keyOffset();\n\t\tvar color;\n\t\tvar colorText = this.magnitude == 0 ? \'gray\' : \'black\';\n\t\tvar patternAlpha = this.alphaPattern.end;\n\t\tvar boxLeft = imageWidth - keySize - margin;\n\t\tvar textY = offset + keySize / 2;\n\t\t\n\t\tvar label;\n\t\tvar keyNameWidth;\n\t\t\n\t\tif ( this == selectedNode )\n\t\t{\n\t\t\tcolor = colorUnclassified;\n\t\t\tlabel =\n\t\t\t\tthis.getUnclassifiedText() +\n\t\t\t\t\'   \' +\n\t\t\t\tthis.getUnclassifiedPercentage();\n\t\t\tkeyNameWidth = measureText(label, false);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tlabel = this.keyLabel;\n\t\t\tcolor = rgbText(this.r.end, this.g.end, this.b.end);\n\t\t\t\n\t\t\tif ( highlight )\n\t\t\t{\n\t\t\t\tif ( this.searchResultChildren() )\n\t\t\t\t{\n\t\t\t\t\tlabel = label + searchResultString(this.searchResultChildren());\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tkeyNameWidth = measureText(label, bold);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tkeyNameWidth = this.keyNameWidth;\n\t\t\t}\n\t\t}\n\t\t\n\t\tvar textLeft = boxLeft - keyBuffer - keyNameWidth - fontSize / 2;\n\t\tvar labelLeft = textLeft;\n\t\t\n\t\tif ( labelLeft > keyMinTextLeft - fontSize / 2 )\n\t\t{\n\t\t\tkeyMinTextLeft -= fontSize / 2;\n\t\t\t\n\t\t\tif ( keyMinTextLeft < centerX - gRadius + fontSize / 2 )\n\t\t\t{\n\t\t\t\tkeyMinTextLeft = centerX - gRadius + fontSize / 2;\n\t\t\t}\n\t\t\t\n\t\t\tlabelLeft = keyMinTextLeft;\n\t\t}\n\t\t\n\t\tvar lineX = new Array();\n\t\tvar lineY = new Array();\n\t\t\n\t\tvar bendRadius;\n\t\tvar keyAngle = Math.atan((textY - centerY) / (labelLeft - centerX));\n\t\tvar arcAngle;\n\t\t\n\t\tif ( keyAngle < 0 )\n\t\t{\n\t\t\tkeyAngle += Math.PI;\n\t\t}\n\t\t\n\t\tif ( keyMinAngle == 0 || angle < keyMinAngle )\n\t\t{\n\t\t\tkeyMinAngle = angle;\n\t\t}\n\t\t\n\t\tif ( angle > Math.PI && keyMinAngle > Math.PI )\n\t\t{\n\t\t\t// allow lines to come underneath the chart\n\t\t\t\n\t\t\tangle -= Math.PI * 2;\n\t\t}\n\t\t\n\t\tlineX.push(Math.cos(angle) * gRadius);\n\t\tlineY.push(Math.sin(angle) * gRadius);\n\t\t\n\t\tif ( angle < keyAngle && textY > centerY + Math.sin(angle) * (gRadius + buffer * (currentKey - 1) / (keys + 1) / 2 + buffer / 2) )\n\t\t{\n\t\t\tbendRadius = gRadius + buffer - buffer * currentKey / (keys + 1) / 2;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tbendRadius = gRadius + buffer * currentKey / (keys + 1) / 2 + buffer / 2;\n\t\t}\n\t\t\n\t\tvar outside =\n\t\t\tMath.sqrt\n\t\t\t(\n\t\t\t\tMath.pow(labelLeft - centerX, 2) +\n\t\t\t\tMath.pow(textY - centerY, 2)\n\t\t\t) > bendRadius;\n\t\t\n\t\tif ( ! outside )\n\t\t{\n\t\t\tarcAngle = Math.asin((textY - centerY) / bendRadius);\n\t\t\t\n\t\t\tkeyMinTextLeft = min(keyMinTextLeft, centerX + bendRadius * Math.cos(arcAngle) - fontSize / 2);\n\t\t\t\n\t\t\tif ( labelLeft < textLeft && textLeft > centerX + bendRadius * Math.cos(arcAngle) )\n\t\t\t{\n\t\t\t\tlineX.push(textLeft - centerX);\n\t\t\t\tlineY.push(textY - centerY);\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tkeyMinTextLeft = min(keyMinTextLeft, labelLeft - fontSize / 2);\n\t\t\t\n\t\t\tif ( angle < keyAngle )\n\t\t\t{\n\t\t\t\t// flip everything over y = x\n\t\t\t\t//\n\t\t\t\tarcAngle = Math.PI / 2 - keyLineAngle\n\t\t\t\t(\n\t\t\t\t\tMath.PI / 2 - angle,\n\t\t\t\t\tMath.PI / 2 - keyAngle,\n\t\t\t\t\tbendRadius,\n\t\t\t\t\ttextY - centerY,\n\t\t\t\t\tlabelLeft - centerX,\n\t\t\t\t\tlineY,\n\t\t\t\t\tlineX\n\t\t\t\t);\n\t\t\t\t\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tarcAngle = keyLineAngle\n\t\t\t\t(\n\t\t\t\t\tangle,\n\t\t\t\t\tkeyAngle,\n\t\t\t\t\tbendRadius,\n\t\t\t\t\tlabelLeft - centerX,\n\t\t\t\t\ttextY - centerY,\n\t\t\t\t\tlineX,\n\t\t\t\t\tlineY\n\t\t\t\t);\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( labelLeft > centerX + bendRadius * Math.cos(arcAngle) ||\n\t\ttextY > centerY + bendRadius * Math.sin(arcAngle) + .01)\n//\t\tif ( outside ||  )\n\t\t{\n\t\t\tlineX.push(labelLeft - centerX);\n\t\t\tlineY.push(textY - centerY);\n\t\t\t\n\t\t\tif ( textLeft != labelLeft )\n\t\t\t{\n\t\t\t\tlineX.push(textLeft - centerX);\n\t\t\t\tlineY.push(textY - centerY);\n\t\t\t}\n\t\t}\n\t\t\n\t\tcontext.globalAlpha = this.alphaWedge.current();\n\t\t\n\t\tif ( snapshotMode )\n\t\t{\n\t\t\tvar labelSVG;\n\t\t\t\n\t\t\tif ( this == selectedNode )\n\t\t\t{\n\t\t\t\tlabelSVG =\n\t\t\t\t\tthis.getUnclassifiedText() +\n\t\t\t\t\tspacer() +\n\t\t\t\t\tthis.getUnclassifiedPercentage();\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tlabelSVG = this.name + spacer() + this.getPercentage() + \'%\';\n\t\t\t}\n\t\t\t\n\t\t\tsvg +=\n\t\t\t\t\'<rect fill="\' + color + \'" \' +\n\t\t\t\t\'x="\' + boxLeft + \'" y="\' + offset +\n\t\t\t\t\'" width="\' + keySize + \'" height="\' + keySize + \'"/>\';\n\t\t\t\n\t\t\tif ( patternAlpha )\n\t\t\t{\n\t\t\t\tsvg +=\n\t\t\t\t\t\'<rect fill="url(#hiddenPattern)" style="stroke:none" \' +\n\t\t\t\t\t\'x="\' + boxLeft + \'" y="\' + offset +\n\t\t\t\t\t\'" width="\' + keySize + \'" height="\' + keySize + \'"/>\';\n\t\t\t}\n\t\t\t\n\t\t\tsvg +=\n\t\t\t\t\'<path class="line\' +\n\t\t\t\t(highlight ? \' highlight\' : \'\') +\n\t\t\t\t\'" d="M \' + (lineX[0] + centerX) + \',\' +\n\t\t\t\t(lineY[0] + centerY);\n\t\t\t\n\t\t\tif ( angle != arcAngle )\n\t\t\t{\n\t\t\t\tsvg +=\n\t\t\t\t\t\' L \' + (centerX + bendRadius * Math.cos(angle)) + \',\' +\n\t\t\t\t\t(centerY + bendRadius * Math.sin(angle)) +\n\t\t\t\t\t\' A \' + bendRadius + \',\' + bendRadius + \' 0 \' +\n\t\t\t\t\t\'0,\' + (angle > arcAngle ? \'0\' : \'1\') + \' \' +\n\t\t\t\t\t(centerX + bendRadius * Math.cos(arcAngle)) + \',\' +\n\t\t\t\t\t(centerY + bendRadius * Math.sin(arcAngle));\n\t\t\t}\n\t\t\t\n\t\t\tfor ( var i = 1; i < lineX.length; i++ )\n\t\t\t{\n\t\t\t\tsvg +=\n\t\t\t\t\t\' L \' + (centerX + lineX[i]) + \',\' +\n\t\t\t\t\t(centerY + lineY[i]);\n\t\t\t}\n\t\t\t\n\t\t\tsvg += \'"/>\';\n\t\t\t\n\t\t\tif ( highlight )\n\t\t\t{\n\t\t\t\tif ( this.searchResultChildren() )\n\t\t\t\t{\n\t\t\t\t\tlabelSVG = labelSVG + searchResultString(this.searchResultChildren());\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tdrawBubbleSVG\n\t\t\t\t(\n\t\t\t\t\tboxLeft - keyBuffer - keyNameWidth - fontSize / 2,\n\t\t\t\t\ttextY - fontSize,\n\t\t\t\t\tkeyNameWidth + fontSize,\n\t\t\t\t\tfontSize * 2,\n\t\t\t\t\tfontSize,\n\t\t\t\t\t0\n\t\t\t\t);\n\t\t\t\t\n\t\t\t\tif ( this.isSearchResult )\n\t\t\t\t{\n\t\t\t\t\tdrawSearchHighlights\n\t\t\t\t\t(\n\t\t\t\t\t\tlabel,\n\t\t\t\t\t\tboxLeft - keyBuffer - keyNameWidth,\n\t\t\t\t\t\ttextY,\n\t\t\t\t\t\t0\n\t\t\t\t\t)\n\t\t\t\t}\n\t\t\t}\n\t\t\t\n\t\t\tsvg += svgText(labelSVG, boxLeft - keyBuffer, textY, \'end\', bold, colorText);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tcontext.fillStyle = color;\n\t\t\tcontext.translate(-centerX, -centerY);\n\t\t\tcontext.strokeStyle = \'black\';\n\t\t\t\tcontext.globalAlpha = 1;//this.alphaWedge.current();\n\t\t\t\n\t\t\tcontext.fillRect(boxLeft, offset, keySize, keySize);\n\t\t\t\n\t\t\tif ( patternAlpha )\n\t\t\t{\n\t\t\t\tcontext.globalAlpha = patternAlpha;\n\t\t\t\tcontext.fillStyle = hiddenPattern;\n\t\t\t\t\n\t\t\t\t// make clipping box for Firefox performance\n\t\t\t\tcontext.beginPath();\n\t\t\t\tcontext.moveTo(boxLeft, offset);\n\t\t\t\tcontext.lineTo(boxLeft + keySize, offset);\n\t\t\t\tcontext.lineTo(boxLeft + keySize, offset + keySize);\n\t\t\t\tcontext.lineTo(boxLeft, offset + keySize);\n\t\t\t\tcontext.closePath();\n\t\t\t\tcontext.save();\n\t\t\t\tcontext.clip();\n\t\t\t\t\n\t\t\t\tcontext.fillRect(boxLeft, offset, keySize, keySize);\n\t\t\t\tcontext.fillRect(boxLeft, offset, keySize, keySize);\n\t\t\t\t\n\t\t\t\tcontext.restore(); // remove clipping region\n\t\t\t}\n\t\t\t\n\t\t\tif ( highlight )\n\t\t\t{\n\t\t\t\tthis.setHighlightStyle();\n\t\t\t\tcontext.fillRect(boxLeft, offset, keySize, keySize);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tcontext.lineWidth = thinLineWidth;\n\t\t\t}\n\t\t\t\n\t\t\tcontext.strokeRect(boxLeft, offset, keySize, keySize);\n\t\t\t\n\t\t\tif ( lineX.length )\n\t\t\t{\n\t\t\t\tcontext.beginPath();\n\t\t\t\tcontext.moveTo(lineX[0] + centerX, lineY[0] + centerY);\n\t\t\t\t\n\t\t\t\tcontext.arc(centerX, centerY, bendRadius, angle, arcAngle, angle > arcAngle);\n\t\t\t\t\n\t\t\t\tfor ( var i = 1; i < lineX.length; i++ )\n\t\t\t\t{\n\t\t\t\t\tcontext.lineTo(lineX[i] + centerX, lineY[i] + centerY);\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tcontext.globalAlpha = this == selectedNode ?\n\t\t\t\t\tthis.children[0].alphaWedge.current() :\n\t\t\t\t\tthis.alphaWedge.current();\n\t\t\t\tcontext.lineWidth = highlight ? highlightLineWidth : thinLineWidth;\n\t\t\t\tcontext.stroke();\n\t\t\t\tcontext.globalAlpha = 1;\n\t\t\t}\n\t\t\t\n\t\t\tif ( highlight )\n\t\t\t{\n\t\t\t\tdrawBubbleCanvas\n\t\t\t\t(\n\t\t\t\t\tboxLeft - keyBuffer - keyNameWidth - fontSize / 2,\n\t\t\t\t\ttextY - fontSize,\n\t\t\t\t\tkeyNameWidth + fontSize,\n\t\t\t\t\tfontSize * 2,\n\t\t\t\t\tfontSize,\n\t\t\t\t\t0\n\t\t\t\t);\n\t\t\t\t\n\t\t\t\tif ( this.isSearchResult )\n\t\t\t\t{\n\t\t\t\t\tdrawSearchHighlights\n\t\t\t\t\t(\n\t\t\t\t\t\tlabel,\n\t\t\t\t\t\tboxLeft - keyBuffer - keyNameWidth,\n\t\t\t\t\t\ttextY,\n\t\t\t\t\t\t0\n\t\t\t\t\t)\n\t\t\t\t}\n\t\t\t}\n\t\t\t\n\t\t\tdrawText(label, boxLeft - keyBuffer, offset + keySize / 2, 0, \'end\', bold, colorText);\n\t\t\t\n\t\t\tcontext.translate(centerX, centerY);\n\t\t}\n\t\t\n\t\tcurrentKey++;\n\t}\n\t\n\tthis.drawLabel = function(angle, bubble, bold, selected, radial)\n\t{\n\t\tif ( context.globalAlpha == 0 )\n\t\t{\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tvar innerText;\n\t\tvar label;\n\t\tvar radius;\n\t\t\n\t\tif ( radial )\n\t\t{\n\t\t\tradius = (this.radiusInner.current() + 1) * gRadius / 2;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tradius = this.labelRadius.current() * gRadius;\n\t\t}\n\t\t\n\t\tif ( radial && (selected || bubble ) )\n\t\t{\n\t\t\tvar percentage = this.getPercentage();\n\t\t\tinnerText = percentage + \'%\';\n\t\t}\n\t\t\n\t\tif\n\t\t(\n\t\t\t! radial &&\n\t\t\tthis != selectedNode &&\n\t\t\t! bubble &&\n\t\t\t( !zoomOut || this != selectedNodeLast)\n\t\t)\n\t\t{\n\t\t\tlabel = this.shortenLabel();\n\t\t}\n\t\telse\n\t\t{\n\t\t\tlabel = this.name;\n\t\t}\n\t\t\n\t\tvar flipped = drawTextPolar\n\t\t(\n\t\t\tlabel,\n\t\t\tinnerText,\n\t\t\tangle,\n\t\t\tradius,\n\t\t\tradial,\n\t\t\tbubble,\n\t\t\tbold,\n//\t\t\tthis.isSearchResult && this.shouldAddSearchResultsString() && (!selected || this == selectedNode || highlight),\n\t\t\tthis.isSearchResult && (!selected || this == selectedNode || bubble),\n\t\t\t(this.hideAlone || !selected || this == selectedNode ) ? this.searchResultChildren() : 0\n\t\t);\n\t\t\n\t\tvar depth = this.getDepth() - selectedNode.getDepth() + 1;\n\t\t\n\t\tif\n\t\t(\n\t\t\t! radial &&\n\t\t\t! bubble &&\n\t\t\tthis != selectedNode &&\n\t\t\tthis.angleEnd.end != this.angleStart.end &&\n\t\t\tnLabelOffsets[depth - 2] > 2 &&\n\t\t\tthis.labelWidth.current() > (this.angleEnd.end - this.angleStart.end) * Math.abs(radius) &&\n\t\t\t! ( zoomOut && this == selectedNodeLast ) &&\n\t\t\tthis.labelRadius.end > 0\n\t\t)\n\t\t{\n\t\t\t// name extends beyond wedge; draw tick mark towards the central\n\t\t\t// radius for easier identification\n\t\t\t\n\t\t\tvar radiusCenter = compress ?\n\t\t\t\t(compressedRadii[depth - 1] + compressedRadii[depth - 2]) / 2 :\n\t\t\t\t(depth - .5) * nodeRadius;\n\t\t\t\n\t\t\tif ( this.labelRadius.end > radiusCenter )\n\t\t\t{\n\t\t\t\tif ( flipped )\n\t\t\t\t{\n\t\t\t\t\tdrawTick(radius - tickLength * 1.4 , tickLength, angle);\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tdrawTick(radius - tickLength * 1.7, tickLength, angle);\n\t\t\t\t}\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tif ( flipped )\n\t\t\t\t{\n\t\t\t\t\tdrawTick(radius + tickLength * .7, tickLength, angle);\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tdrawTick(radius + tickLength * .4, tickLength, angle);\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t}\n\t\n\tthis.drawLines = function(angleStart, angleEnd, radiusInner, drawRadial, selected)\n\t{\n\t\tif ( snapshotMode )\n\t\t{\n\t\t\tif ( this != selectedNode)\n\t\t\t{\n\t\t\t\tif ( angleEnd == angleStart + Math.PI * 2 )\n\t\t\t\t{\n\t\t\t\t\t// fudge to prevent overlap, which causes arc ambiguity\n\t\t\t\t\t//\n\t\t\t\t\tangleEnd -= .1 / gRadius;\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tvar longArc = angleEnd - angleStart > Math.PI ? 1 : 0;\n\t\t\t\t\n\t\t\t\tvar x1 = centerX + radiusInner * Math.cos(angleStart);\n\t\t\t\tvar y1 = centerY + radiusInner * Math.sin(angleStart);\n\t\t\t\t\n\t\t\t\tvar x2 = centerX + gRadius * Math.cos(angleStart);\n\t\t\t\tvar y2 = centerY + gRadius * Math.sin(angleStart);\n\t\t\t\t\n\t\t\t\tvar x3 = centerX + gRadius * Math.cos(angleEnd);\n\t\t\t\tvar y3 = centerY + gRadius * Math.sin(angleEnd);\n\t\t\t\t\n\t\t\t\tvar x4 = centerX + radiusInner * Math.cos(angleEnd);\n\t\t\t\tvar y4 = centerY + radiusInner * Math.sin(angleEnd);\n\t\t\t\t\n\t\t\t\tif ( this.alphaArc.end )\n\t\t\t\t{\n\t\t\t\t\tvar dArray =\n\t\t\t\t\t[\n\t\t\t\t\t\t" M ", x4, ",", y4,\n\t\t\t\t\t\t" A ", radiusInner, ",", radiusInner, " 0 ", longArc,\n\t\t\t\t\t\t\t" 0 ", x1, ",", y1\n\t\t\t\t\t];\n\t\t\t\t\t\n\t\t\t\t\tsvg += \'<path class="line" d="\' + dArray.join(\'\') + \'"/>\';\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif ( drawRadial && this.alphaLine.end )\n\t\t\t\t{\n\t\t\t\t\tsvg += \'<line x1="\' + x3 + \'" y1="\' + y3 + \'" x2="\' + x4 + \'" y2="\' + y4 + \'"/>\';\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tcontext.lineWidth = thinLineWidth;\n\t\t\tcontext.strokeStyle = \'black\';\n\t\t\tcontext.beginPath();\n\t\t\tcontext.arc(0, 0, radiusInner, angleStart, angleEnd, false);\n\t\t\tcontext.globalAlpha = this.alphaArc.current();\n\t\t\tcontext.stroke();\n\t\t\t\n\t\t\tif ( drawRadial )\n\t\t\t{\n\t\t\t\tvar x1 = radiusInner * Math.cos(angleEnd);\n\t\t\t\tvar y1 = radiusInner * Math.sin(angleEnd);\n\t\t\t\tvar x2 = gRadius * Math.cos(angleEnd);\n\t\t\t\tvar y2 = gRadius * Math.sin(angleEnd);\n\t\t\t\t\n\t\t\t\tcontext.beginPath();\n\t\t\t\tcontext.moveTo(x1, y1);\n\t\t\t\tcontext.lineTo(x2, y2);\n\t\t\t\t\n//\t\t\t\tif ( this.getCollapse() )//( selected && this != selectedNode )\n\t\t\t\t{\n\t\t\t\t\tcontext.globalAlpha = this.alphaLine.current();\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tcontext.stroke();\n\t\t\t}\n\t\t}\n\t}\n\t\n\tthis.drawMap = function(child)\n\t{\n\t\tif ( this.parent )\n\t\t{\n\t\t\tthis.parent.drawMap(child);\n\t\t}\n\t\t\n\t\tif ( this.getCollapse() && this != child || this == focusNode )\n\t\t{\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tvar angleStart =\n\t\t\t(child.baseMagnitude - this.baseMagnitude) / this.magnitude * Math.PI * 2 +\n\t\t\trotationOffset;\n\t\tvar angleEnd =\n\t\t\t(child.baseMagnitude - this.baseMagnitude + child.magnitude) /\n\t\t\tthis.magnitude * Math.PI * 2 +\n\t\t\trotationOffset;\n\t\t\n\t\tvar box = this.getMapPosition();\n\t\t\n\t\tcontext.save();\n\t\tcontext.fillStyle = \'black\';\n\t\tcontext.textAlign = \'end\';\n\t\tcontext.textBaseline = \'middle\';\n\t\t\n\t\tvar textX = box.x - mapRadius - mapBuffer;\n\t\tvar percentage = getPercentage(child.magnitude / this.magnitude);\n\t\t\n\t\tvar highlight = this == selectedNode || this == highlightedNode;\n\t\t\n\t\tif ( highlight )\n\t\t{\n\t\t\tcontext.font = fontBold;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tcontext.font = fontNormal;\n\t\t}\n\t\t\n\t\tcontext.fillText(percentage + \'% of\', textX, box.y - mapRadius / 3);\n\t\tcontext.fillText(this.name, textX, box.y + mapRadius / 3);\n\t\t\n\t\tif ( highlight )\n\t\t{\n\t\t\tcontext.font = fontNormal;\n\t\t}\n\t\t\n\t\tif ( this == highlightedNode && this != selectedNode )\n\t\t{\n\t\t\tcontext.fillStyle = \'rgb(245, 245, 245)\';\n//\t\t\tcontext.fillStyle = \'rgb(200, 200, 200)\';\n\t\t}\n\t\telse\n\t\t{\n\t\t\tcontext.fillStyle = \'rgb(255, 255, 255)\';\n\t\t}\n\t\t\n\t\tcontext.beginPath();\n\t\tcontext.arc(box.x, box.y, mapRadius, 0, Math.PI * 2, true);\n\t\tcontext.closePath();\n\t\tcontext.fill();\n\t\t\n\t\tif ( this == selectedNode )\n\t\t{\n\t\t\tcontext.lineWidth = 1;\n\t\t\tcontext.fillStyle = \'rgb(100, 100, 100)\';\n\t\t}\n\t\telse\n\t\t{\n\t\t\tif ( this == highlightedNode )\n\t\t\t{\n\t\t\t\tcontext.lineWidth = .2;\n\t\t\t\tcontext.fillStyle = \'rgb(190, 190, 190)\';\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tcontext.lineWidth = .2;\n\t\t\t\tcontext.fillStyle = \'rgb(200, 200, 200)\';\n\t\t\t}\n\t\t}\n\t\t\n\t\tvar maxDepth = this.getMaxDepth();\n\t\t\n\t\tif ( ! compress && maxDepth > maxPossibleDepth + this.getDepth() - 1 )\n\t\t{\n\t\t\tmaxDepth = maxPossibleDepth + this.getDepth() - 1;\n\t\t}\n\t\t\n\t\tif ( this.getDepth() < selectedNode.getDepth() )\n\t\t{\n\t\t\tif ( child.getDepth() - 1 >= maxDepth )\n\t\t\t{\n\t\t\t\tmaxDepth = child.getDepth();\n\t\t\t}\n\t\t}\n\t\t\n\t\tvar radiusInner;\n\t\t\n\t\tif ( compress )\n\t\t{\n\t\t\tradiusInner = 0;\n//\t\t\t\tMath.atan(child.getDepth() - this.getDepth()) /\n//\t\t\t\tMath.PI * 2 * .9;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tradiusInner =\n\t\t\t\t(child.getDepth() - this.getDepth()) /\n\t\t\t\t(maxDepth - this.getDepth() + 1);\n\t\t}\n\t\t\n\t\tcontext.stroke();\n\t\tcontext.beginPath();\n\t\t\n\t\tif ( radiusInner == 0 )\n\t\t{\n\t\t\tcontext.moveTo(box.x, box.y);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tcontext.arc(box.x, box.y, mapRadius * radiusInner, angleEnd, angleStart, true);\n\t\t}\n\t\t\n\t\tcontext.arc(box.x, box.y, mapRadius, angleStart, angleEnd, false);\n\t\tcontext.closePath();\n\t\tcontext.fill();\n\t\t\n\t\tif ( this == highlightedNode && this != selectedNode )\n\t\t{\n\t\t\tcontext.lineWidth = 1;\n\t\t\tcontext.stroke();\n\t\t}\n\t\t\n\t\tcontext.restore();\n\t}\n\t\n\tthis.drawReferenceRings = function(childRadiusInner)\n\t{\n\t\tif ( snapshotMode )\n\t\t{\n\t\t\tsvg +=\n\t\t\t\t\'<circle cx="\' + centerX + \'" cy="\' + centerY +\n\t\t\t\t\'" r="\' + childRadiusInner + \'"/>\';\n\t\t\tsvg +=\n\t\t\t\t\'<circle cx="\' + centerX + \'" cy="\' + centerY +\n\t\t\t\t\'" r="\' + gRadius + \'"/>\';\n\t\t}\n\t\telse\n\t\t{\n\t\t\tcontext.globalAlpha = 1 - this.alphaLine.current();//this.getUncollapsed().alphaLine.current();\n\t\t\tcontext.beginPath();\n\t\t\tcontext.arc(0, 0, childRadiusInner, 0, Math.PI * 2, false);\n\t\t\tcontext.stroke();\n\t\t\tcontext.beginPath();\n\t\t\tcontext.arc(0, 0, gRadius, 0, Math.PI * 2, false);\n\t\t\tcontext.stroke();\n\t\t}\n\t}\n\t\n\tthis.getCollapse = function()\n\t{\n\t\treturn (\n\t\t\tcollapse &&\n\t\t\tthis.collapse &&\n\t\t\tthis.depth != maxAbsoluteDepth\n\t\t\t);\n\t}\n\t\n\tthis.getDepth = function()\n\t{\n\t\tif ( collapse )\n\t\t{\n\t\t\treturn this.depthCollapsed;\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn this.depth;\n\t\t}\n\t}\n\t\n\tthis.getMagnitude = function()\n\t{\n\t\treturn this.attributes[magnitudeIndex][currentDataset];\n\t}\n\t\n\tthis.getMapPosition = function()\n\t{\n\t\treturn {\n\t\t\tx : (details.offsetLeft + details.clientWidth - mapRadius),\n\t\t\ty : ((focusNode.getDepth() - this.getDepth()) *\n\t\t\t\t(mapBuffer + mapRadius * 2) - mapRadius) +\n\t\t\t\tdetails.clientHeight + details.offsetTop\n\t\t};\n\t}\n\t\n\tthis.getMaxDepth = function(limit)\n\t{\n\t\tvar max;\n\t\t\n\t\tif ( collapse )\n\t\t{\n\t\t\treturn this.maxDepthCollapsed;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tif ( this.maxDepth > maxAbsoluteDepth )\n\t\t\t{\n\t\t\t\treturn maxAbsoluteDepth;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\treturn this.maxDepth;\n\t\t\t}\n\t\t}\n\t}\n\t\n\tthis.getData = function(index, summary)\n\t{\n\t\tvar files = new Array();\n\t\t\n\t\tif\n\t\t(\n\t\t\tthis.attributes[index] != null &&\n\t\t\tthis.attributes[index][currentDataset] != null &&\n\t\t\tthis.attributes[index][currentDataset] != \'\'\n\t\t)\n\t\t{\n\t\t\tfiles.push\n\t\t\t(\n\t\t\t\tdocument.location +\n\t\t\t\t\'.files/\' +\n\t\t\t\tthis.attributes[index][currentDataset]\n\t\t\t);\n\t\t}\n\t\t\n\t\tif ( summary )\n\t\t{\n\t\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t\t{\n\t\t\t\tfiles = files.concat(this.children[i].getData(index, true));\n\t\t\t}\n\t\t}\n\t\t\n\t\treturn files;\n\t}\n\t\n\tthis.getList = function(index, summary)\n\t{\n\t\tvar list;\n\t\t\n\t\tif\n\t\t(\n\t\t\tthis.attributes[index] != null &&\n\t\t\tthis.attributes[index][currentDataset] != null\n\t\t)\n\t\t{\n\t\t\tlist = this.attributes[index][currentDataset];\n\t\t}\n\t\telse\n\t\t{\n\t\t\tlist = new Array();\n\t\t}\n\t\t\n\t\tif ( summary )\n\t\t{\n\t\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t\t{\n\t\t\t\tlist = list.concat(this.children[i].getList(index, true));\n\t\t\t}\n\t\t}\n\t\t\n\t\treturn list;\n\t}\n\t\n\tthis.getParent = function()\n\t{\n\t\t// returns parent, accounting for collapsing or 0 if doesn\'t exist\n\t\t\n\t\tvar parent = this.parent;\n\t\t\n\t\twhile ( parent != 0 && parent.getCollapse() )\n\t\t{\n\t\t\tparent = parent.parent;\n\t\t}\n\t\t\n\t\treturn parent;\n\t}\n\t\n\tthis.getPercentage = function()\n\t{\n\t\treturn getPercentage(this.magnitude / selectedNode.magnitude);\n\t}\n\t\n\tthis.getUnclassifiedPercentage = function()\n\t{\n\t\tif ( this.children.length )\n\t\t{\n\t\t\tvar lastChild = this.children[this.children.length - 1];\n\t\t\n\t\t\treturn getPercentage\n\t\t\t(\n\t\t\t\t(\n\t\t\t\t\tthis.baseMagnitude +\n\t\t\t\t\tthis.magnitude -\n\t\t\t\t\tlastChild.magnitude -\n\t\t\t\t\tlastChild.baseMagnitude\n\t\t\t\t) / this.magnitude\n\t\t\t) + \'%\';\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn \'100%\';\n\t\t}\n\t}\n\t\n\tthis.getUnclassifiedText = function()\n\t{\n\t\treturn \'[other \'+ this.name + \']\';\n\t}\n\t\n\tthis.getUncollapsed = function()\n\t{\n\t\t// recurse through collapsed children until uncollapsed node is found\n\t\t\n\t\tif ( this.getCollapse() )\n\t\t{\n\t\t\treturn this.children[0].getUncollapsed();\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn this;\n\t\t}\n\t}\n\t\n\tthis.hasChildren = function()\n\t{\n\t\treturn this.children.length && this.depth < maxAbsoluteDepth && this.magnitude;\n\t}\n\t\n\tthis.hasParent = function(parent)\n\t{\n\t\tif ( this.parent )\n\t\t{\n\t\t\tif ( this.parent == parent )\n\t\t\t{\n\t\t\t\treturn true;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\treturn this.parent.hasParent(parent);\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn false;\n\t\t}\n\t}\n\t\n\tthis.maxVisibleDepth = function(maxDepth)\n\t{\n\t\tvar childInnerRadius;\n\t\tvar depth = this.getDepth() - selectedNode.getDepth() + 1;\n\t\tvar currentMaxDepth = depth;\n\t\t\n\t\tif ( this.hasChildren() && depth < maxDepth)\n\t\t{\n\t\t\tvar lastChild = this.children[this.children.length - 1];\n\t\t\t\n\t\t\tif ( this.name == \'Pseudomonadaceae\' )\n\t\t\t{\n\t\t\t\tvar x = 3;\n\t\t\t}\n\t\t\t\n\t\t\tif\n\t\t\t(\n\t\t\t\tlastChild.baseMagnitude + lastChild.magnitude <\n\t\t\t\tthis.baseMagnitude + this.magnitude\n\t\t\t)\n\t\t\t{\n\t\t\t\tcurrentMaxDepth++;\n\t\t\t}\n\t\t\t\n\t\t\tif ( compress )\n\t\t\t{\n\t\t\t\tchildInnerRadius = compressedRadii[depth - 1];\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tchildInnerRadius = (depth) / maxDepth;\n\t\t\t}\n\t\t\t\n\t\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t\t{\n\t\t\t\tif\n\t\t\t\t(//true ||\n\t\t\t\t\tthis.children[i].magnitude *\n\t\t\t\t\tangleFactor *\n\t\t\t\t\t(childInnerRadius + 1) *\n\t\t\t\t\tgRadius >=\n\t\t\t\t\tminWidth()\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\tvar childMaxDepth = this.children[i].maxVisibleDepth(maxDepth);\n\t\t\t\t\t\n\t\t\t\t\tif ( childMaxDepth > currentMaxDepth )\n\t\t\t\t\t{\n\t\t\t\t\t\tcurrentMaxDepth = childMaxDepth;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\t\n\t\treturn currentMaxDepth;\n\t}\n\t\n\tthis.resetLabelWidth = function()\n\t{\n\t\tvar nameWidthOld = this.nameWidth;\n\t\t\n\t\tif ( true || ! this.radial )//&& fontSize != fontSizeLast )\n\t\t{\n\t\t\tvar dim = context.measureText(this.name);\n\t\t\tthis.nameWidth = dim.width;\n\t\t}\n\t\t\n\t\tif ( fontSize != fontSizeLast && this.labelWidth.end == nameWidthOld * labelWidthFudge )\n\t\t{\n\t\t\t// font size changed; adjust start of tween to match\n\t\t\t\n\t\t\tthis.labelWidth.start = this.nameWidth * labelWidthFudge;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tthis.labelWidth.start = this.labelWidth.current();\n\t\t}\n\t\t\n\t\tthis.labelWidth.end = this.nameWidth * labelWidthFudge;\n\t}\n\t\n\tthis.restrictLabelWidth = function(width)\n\t{\n\t\tif ( width < this.labelWidth.end )\n\t\t{\n\t\t\tthis.labelWidth.end = width;\n\t\t}\n\t}\n\t\n\tthis.search = function()\n\t{\n\t\tthis.isSearchResult = false;\n\t\tthis.searchResults = 0;\n\t\t\n\t\tif\n\t\t(\n\t\t\t! this.getCollapse() &&\n\t\t\tsearch.value != \'\' &&\n\t\t\tthis.name.toLowerCase().indexOf(search.value.toLowerCase()) != -1\n\t\t)\n\t\t{\n\t\t\tthis.isSearchResult = true;\n\t\t\tthis.searchResults = 1;\n\t\t\tnSearchResults++;\n\t\t}\n\t\t\n\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\tthis.searchResults += this.children[i].search();\n\t\t}\n\t\t\n\t\treturn this.searchResults;\n\t}\n\t\n\tthis.searchResultChildren = function()\n\t{\n\t\tif ( this.isSearchResult )\n\t\t{\n\t\t\treturn this.searchResults - 1;\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn this.searchResults;\n\t\t}\n\t}\n\t\n\tthis.setDepth = function(depth, depthCollapsed)\n\t{\n\t\tthis.depth = depth;\n\t\tthis.depthCollapsed = depthCollapsed;\n\t\t\n\t\tif\n\t\t(\n\t\t\tthis.children.length == 1 &&\n//\t\t\tthis.magnitude > 0 &&\n\t\t\tthis.children[0].magnitude == this.magnitude &&\n\t\t\t( head.children.length > 1 || this.children[0].children.length )\n\t\t)\n\t\t{\n\t\t\tthis.collapse = true;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tthis.collapse = false;\n\t\t\tdepthCollapsed++;\n\t\t}\n\t\t\n\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\tthis.children[i].setDepth(depth + 1, depthCollapsed);\n\t\t}\n\t}\n\t\n\tthis.setHighlightStyle = function()\n\t{\n\t\tcontext.lineWidth = highlightLineWidth;\n\t\t\n\t\tif ( this.hasChildren() || this != focusNode || this != highlightedNode )\n\t\t{\n\t\t\tcontext.strokeStyle = \'black\';\n\t\t\tcontext.fillStyle = "rgba(255, 255, 255, .3)";\n\t\t}\n\t\telse\n\t\t{\n\t\t\tcontext.strokeStyle = \'rgb(90,90,90)\';\n\t\t\tcontext.fillStyle = "rgba(155, 155, 155, .3)";\n\t\t}\n\t}\n\t\n\tthis.setLabelWidth = function(node)\n\t{\n\t\tif ( ! shorten || this.radial )\n\t\t{\n\t\t\treturn; // don\'t need to set width\n\t\t}\n\t\t\n\t\tif ( node.hide )\n\t\t{\n\t\t\talert(\'wtf\');\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tvar angle = (this.angleStart.end + this.angleEnd.end) / 2;\n\t\tvar a; // angle difference\n\t\t\n\t\tif ( node == selectedNode )\n\t\t{\n\t\t\ta = Math.abs(angle - node.angleOther);\n\t\t}\n\t\telse\n\t\t{\n\t\t\ta = Math.abs(angle - (node.angleStart.end + node.angleEnd.end) / 2);\n\t\t}\n\t\t\n\t\tif ( a == 0 )\n\t\t{\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tif ( a > Math.PI )\n\t\t{\n\t\t\ta = 2 * Math.PI - a;\n\t\t}\n\t\t\n\t\tif ( node.radial || node == selectedNode )\n\t\t{\n\t\t\tvar nodeLabelRadius;\n\t\t\t\n\t\t\tif ( node == selectedNode )\n\t\t\t{\n\t\t\t\t// radial \'other\' label\n\t\t\t\t\n\t\t\t\tnodeLabelRadius = (node.children[0].radiusInner.end + 1) / 2;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tnodeLabelRadius = (node.radiusInner.end + 1) / 2;\n\t\t\t}\n\t\t\t\n\t\t\tif ( a < Math.PI / 2 )\n\t\t\t{\n\t\t\t\tvar r = this.labelRadius.end * gRadius + .5 * fontSize\n\t\t\t\tvar hypotenuse = r / Math.cos(a);\n\t\t\t\tvar opposite = r * Math.tan(a);\n\t\t\t\tvar fontRadius = .8 * fontSize;\n\t\t\t\t\n\t\t\t\tif\n\t\t\t\t(\n\t\t\t\t\tnodeLabelRadius * gRadius < hypotenuse &&\n\t\t\t\t\tthis.labelWidth.end / 2 + fontRadius > opposite\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\tthis.labelWidth.end = 2 * (opposite - fontRadius);\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\telse if\n\t\t(\n\t\t\tthis.labelRadius.end == node.labelRadius.end &&\n\t\t\ta < Math.PI / 4\n\t\t)\n\t\t{\n\t\t\t// same radius with small angle; use circumferential approximation\n\t\t\t\n\t\t\tvar dist = a * this.labelRadius.end * gRadius - fontSize * (1 - a * 4 / Math.PI) * 1.3;\n\t\t\t\n\t\t\tif ( this.labelWidth.end < dist )\n\t\t\t{\n\t\t\t\tnode.restrictLabelWidth((dist - this.labelWidth.end / 2) * 2);\n\t\t\t}\n\t\t\telse if ( node.labelWidth.end < dist )\n\t\t\t{\n\t\t\t\tthis.restrictLabelWidth((dist - node.labelWidth.end / 2) * 2);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\t// both labels reach halfway point; restrict both\n\t\t\t\t\n\t\t\t\tthis.labelWidth.end = dist;\n\t\t\t\tnode.labelWidth.end = dist\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tvar r1 = this.labelRadius.end * gRadius;\n\t\t\tvar r2 = node.labelRadius.end * gRadius;\n\t\t\t\n\t\t\t// first adjust the radii to account for the height of the font by shifting them\n\t\t\t// toward each other\n\t\t\t//\n\t\t\tvar fontFudge = .35 * fontSize;\n\t\t\t//\n\t\t\tif ( this.labelRadius.end < node.labelRadius.end )\n\t\t\t{\n\t\t\t\tr1 += fontFudge;\n\t\t\t\tr2 -= fontFudge;\n\t\t\t}\n\t\t\telse if ( this.labelRadius.end > node.labelRadius.end )\n\t\t\t{\n\t\t\t\tr1 -= fontFudge;\n\t\t\t\tr2 += fontFudge;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tr1 -= fontFudge;\n\t\t\t\tr2 -= fontFudge;\n\t\t\t}\n\t\t\t\n\t\t\tvar r1s = r1 * r1;\n\t\t\tvar r2s = r2 * r2;\n\t\t\t\n\t\t\t// distance between the centers of the two labels\n\t\t\t//\n\t\t\tvar dist = Math.sqrt(r1s + r2s - 2 * r1 * r2 * Math.cos(a));\n\t\t\t\n\t\t\t// angle at our label center between our radius and the line to the other label center\n\t\t\t//\n\t\t\tvar b = Math.acos((r1s + dist * dist - r2s) / (2 * r1 * dist));\n\t\t\t\n\t\t\t// distance from our label center to the intersection of the two tangents\n\t\t\t//\n\t\t\tvar l1 = Math.sin(a + b - Math.PI / 2) * dist / Math.sin(Math.PI - a);\n\t\t\t\n\t\t\t// distance from other label center the the intersection of the two tangents\n\t\t\t//\n\t\t\tvar l2 = Math.sin(Math.PI / 2 - b) * dist / Math.sin(Math.PI - a);\n\t\t\t\n\t\t\tl1 = Math.abs(l1) - .4 * fontSize;\n\t\t\tl2 = Math.abs(l2) - .4 * fontSize;\n/*\t\t\t\n\t\t\t// amount to shorten the distances because of the height of the font\n\t\t\t//\n\t\t\tvar l3 = 0;\n\t\t\tvar fontRadius = fontSize * .55;\n\t\t\t//\n\t\t\tif ( l1 < 0 || l2 < 0 )\n\t\t\t{\n\t\t\t\tvar l4 = fontRadius / Math.tan(a);\n\t\t\tl1 = Math.abs(l1);\n\t\t\tl2 = Math.abs(l2);\n\t\t\t\n\t\t\t\tl1 -= l4;\n\t\t\t\tl2 -= l4;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tvar c = Math.PI - a;\n\t\t\t\t\n\t\t\t\tl3 = fontRadius * Math.tan(c / 2);\n\t\t\t}\n*/\t\t\t\n\t\t\tif ( this.labelWidth.end / 2 > l1 && node.labelWidth.end / 2 > l2 )\n\t\t\t{\n\t\t\t\t// shorten the farthest one from the intersection\n\t\t\t\t\n\t\t\t\tif ( l1 > l2 )\n\t\t\t\t{\n\t\t\t\t\tthis.restrictLabelWidth(2 * (l1));// - l3 - fontRadius));\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tnode.restrictLabelWidth(2 * (l2));// - l3 - fontRadius));\n\t\t\t\t}\n\t\t\t}/*\n\t\t\telse if ( this.labelWidth.end / 2 > l1 + l3 && node.labelWidth.end / 2 > l2 - l3 )\n\t\t\t{\n\t\t\t\tnode.restrictLabelWidth(2 * (l2 - l3));\n\t\t\t}\n\t\t\telse if ( this.labelWidth.end / 2 > l1 - l3 && node.labelWidth.end / 2 > l2 + l3 )\n\t\t\t{\n\t\t\t\tthis.restrictLabelWidth(2 * (l1 - l3));\n\t\t\t}*/\n\t\t}\n\t}\n\t\n\tthis.setMagnitudes = function(baseMagnitude)\n\t{\n\t\tthis.magnitude = this.getMagnitude();\n\t\tthis.baseMagnitude = baseMagnitude;\n\t\t\n\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\tthis.children[i].setMagnitudes(baseMagnitude);\n\t\t\tbaseMagnitude += this.children[i].magnitude;\n\t\t}\n\t\t\n\t\tthis.maxChildMagnitude = baseMagnitude;\n\t}\n\t\n\tthis.setMaxDepths = function()\n\t{\n\t\tthis.maxDepth = this.depth;\n\t\tthis.maxDepthCollapsed = this.depthCollapsed;\n\t\t\n\t\tfor ( i in this.children )\n\t\t{\n\t\t\tvar child = this.children[i];\n\t\t\t\n\t\t\tchild.setMaxDepths();\n\t\t\t\n\t\t\tif ( child.maxDepth > this.maxDepth )\n\t\t\t{\n\t\t\t\tthis.maxDepth = child.maxDepth;\n\t\t\t}\n\t\t\t\n\t\t\tif\n\t\t\t(\n\t\t\t\tchild.maxDepthCollapsed > this.maxDepthCollapsed &&\n\t\t\t\t(child.depth <= maxAbsoluteDepth || maxAbsoluteDepth == 0)\n\t\t\t)\n\t\t\t{\n\t\t\t\tthis.maxDepthCollapsed = child.maxDepthCollapsed;\n\t\t\t}\n\t\t}\n\t}\n\t\n\tthis.setTargetLabelRadius = function()\n\t{\n\t\tvar depth = this.getDepth() - selectedNode.getDepth() + 1;\n\t\tvar index = depth - 2;\n\t\tvar labelOffset = labelOffsets[index];\n\t\t\n\t\tif ( this.radial )\n\t\t{\n\t\t\t//this.labelRadius.setTarget((this.radiusInner.end + 1) / 2);\n\t\t\tvar max =\n\t\t\t\tdepth == maxDisplayDepth ?\n\t\t\t\t1 :\n\t\t\t\tcompressedRadii[index + 1];\n\t\t\t\n\t\t\tthis.labelRadius.setTarget((compressedRadii[index] + max) / 2);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tvar radiusCenter;\n\t\t\tvar width;\n\t\t\t\n\t\t\tif ( compress )\n\t\t\t{\n\t\t\t\tif ( nLabelOffsets[index] > 1 )\n\t\t\t\t{\n\t\t\t\t\tthis.labelRadius.setTarget\n\t\t\t\t\t(\n\t\t\t\t\t\tlerp\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\tlabelOffset + .75,\n\t\t\t\t\t\t\t0,\n\t\t\t\t\t\t\tnLabelOffsets[index] + .5,\n\t\t\t\t\t\t\tcompressedRadii[index],\n\t\t\t\t\t\t\tcompressedRadii[index + 1]\n\t\t\t\t\t\t)\n\t\t\t\t\t);\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tthis.labelRadius.setTarget((compressedRadii[index] + compressedRadii[index + 1]) / 2);\n\t\t\t\t}\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tradiusCenter =\n\t\t\t\t\tnodeRadius * (depth - 1) +\n\t\t\t\t\tnodeRadius / 2;\n\t\t\t\twidth = nodeRadius;\n\t\t\t\t\n\t\t\t\tthis.labelRadius.setTarget\n\t\t\t\t(\n\t\t\t\t\tradiusCenter + width * ((labelOffset + 1) / (nLabelOffsets[index] + 1) - .5)\n\t\t\t\t);\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( ! this.hide && ! this.keyed && nLabelOffsets[index] )\n\t\t{\n\t\t\t// check last and first labels in each track for overlap\n\t\t\t\n\t\t\tfor ( var i = 0; i < maxDisplayDepth - 1; i++ )\n\t\t\t{\n\t\t\t\tfor ( var j = 0; j <= nLabelOffsets[i]; j++ )\n\t\t\t\t{\n\t\t\t\t\tvar last = labelLastNodes[i][j];\n\t\t\t\t\tvar first = labelFirstNodes[i][j];\n\t\t\t\t\t\n\t\t\t\t\tif ( last )\n\t\t\t\t\t{\n\t\t\t\t\t\tif ( j == nLabelOffsets[i] )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\t// last is radial\n\t\t\t\t\t\t\tthis.setLabelWidth(last);\n\t\t\t\t\t\t}\n\t\t\t\t\t\telse\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tlast.setLabelWidth(this);\n\t\t\t\t\t\t}\n\t\t\t\t\t}\n\t\t\t\t\t\n\t\t\t\t\tif ( first )\n\t\t\t\t\t{\n\t\t\t\t\t\tif ( j == nLabelOffsets[i] )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tthis.setLabelWidth(first);\n\t\t\t\t\t\t}\n\t\t\t\t\t\telse\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tfirst.setLabelWidth(this);\n\t\t\t\t\t\t}\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\t\t\n\t\t\tif ( selectedNode.canDisplayLabelOther )\n\t\t\t{\n\t\t\t\tthis.setLabelWidth(selectedNode); // in case there is an \'other\' label\n\t\t\t}\n\t\t\t\n\t\t\tif ( this.radial )\n\t\t\t{\n\t\t\t\t// use the last \'track\' of this depth for radial\n\t\t\t\t\n\t\t\t\tlabelLastNodes[index][nLabelOffsets[index]] = this;\n\t\t\t\t\n\t\t\t\tif ( labelFirstNodes[index][nLabelOffsets[index]] == 0 )\n\t\t\t\t{\n\t\t\t\t\tlabelFirstNodes[index][nLabelOffsets[index]] = this;\n\t\t\t\t}\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tlabelLastNodes[index][labelOffset] = this;\n\t\t\t\t\n\t\t\t\t// update offset\n\t\t\t\t\n\t\t\t\tlabelOffsets[index] += 1;\n\t\t\t\t\n\t\t\t\tif ( labelOffsets[index] > nLabelOffsets[index] )\n\t\t\t\t{\n\t\t\t\t\tlabelOffsets[index] -= nLabelOffsets[index];\n\t\t\t\t\t\n\t\t\t\t\tif ( !(nLabelOffsets[index] & 1) )\n\t\t\t\t\t{\n\t\t\t\t\t\tlabelOffsets[index]--;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\telse if ( labelOffsets[index] == nLabelOffsets[index] )\n\t\t\t\t{\n\t\t\t\t\tlabelOffsets[index] -= nLabelOffsets[index];\n\t\t\t\t\t\n\t\t\t\t\tif ( false && !(nLabelOffsets[index] & 1) )\n\t\t\t\t\t{\n\t\t\t\t\t\tlabelOffsets[index]++;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif ( labelFirstNodes[index][labelOffset] == 0 )\n\t\t\t\t{\n\t\t\t\t\tlabelFirstNodes[index][labelOffset] = this;\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\telse if ( this.hide )\n\t\t{\n\t\t\tthis.labelWidth.end = 0;\n\t\t}\n\t}\n\t\n\tthis.setTargets = function()\n\t{\n\t\tif ( this == selectedNode )\n\t\t{\n\t\t\tthis.setTargetsSelected\n\t\t\t(\n\t\t\t\t0,\n\t\t\t\t1,\n\t\t\t\tlightnessBase,\n\t\t\t\tfalse,\n\t\t\t\tfalse\n\t\t\t);\n\t\t\treturn;\n\t\t}\n\t\t\n\t\tvar depthRelative = this.getDepth() - selectedNode.getDepth();\n\t\t\n\t\tvar parentOfSelected = selectedNode.hasParent(this);\n/*\t\t(\n//\t\t\t! this.getCollapse() &&\n\t\t\tthis.baseMagnitude <= selectedNode.baseMagnitude &&\n\t\t\tthis.baseMagnitude + this.magnitude >=\n\t\t\tselectedNode.baseMagnitude + selectedNode.magnitude\n\t\t);\n*/\t\t\n\t\tif ( parentOfSelected )\n\t\t{\n\t\t\tthis.resetLabelWidth();\n\t\t}\n\t\telse\n\t\t{\n\t\t\t//context.font = fontNormal;\n\t\t\tvar dim = context.measureText(this.name);\n\t\t\tthis.nameWidth = dim.width;\n\t\t\t//this.labelWidth.setTarget(this.labelWidth.end);\n\t\t\tthis.labelWidth.setTarget(0);\n\t\t}\n\t\t\n\t\t// set angles\n\t\t//\n\t\tif ( this.baseMagnitude <= selectedNode.baseMagnitude )\n\t\t{\n\t\t\tthis.angleStart.setTarget(0);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tthis.angleStart.setTarget(Math.PI * 2);\n\t\t}\n\t\t//\n\t\tif\n\t\t(\n\t\t\tparentOfSelected ||\n\t\t\tthis.baseMagnitude + this.magnitude >=\n\t\t\tselectedNode.baseMagnitude + selectedNode.magnitude\n\t\t)\n\t\t{\n\t\t\tthis.angleEnd.setTarget(Math.PI * 2);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tthis.angleEnd.setTarget(0);\n\t\t}\n\t\t\n\t\t// children\n\t\t//\n\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\tthis.children[i].setTargets();\n\t\t}\n\t\t\n\t\tif ( this.getDepth() <= selectedNode.getDepth() )\n\t\t{\n\t\t\t// collapse in\n\t\t\t\n\t\t\tthis.radiusInner.setTarget(0);\n\t\t\t\n\t\t\tif ( parentOfSelected )\n\t\t\t{\n\t\t\t\tthis.labelRadius.setTarget\n\t\t\t\t(\n\t\t\t\t\t(depthRelative) *\n\t\t\t\t\thistorySpacingFactor * fontSize / gRadius\n\t\t\t\t);\n\t\t\t\t//this.scale.setTarget(1 - (selectedNode.getDepth() - this.getDepth()) / 18); // TEMP\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.labelRadius.setTarget(0);\n\t\t\t\t//this.scale.setTarget(1); // TEMP\n\t\t\t}\n\t\t}\n\t\telse if ( depthRelative + 1 > maxDisplayDepth )\n\t\t{\n\t\t\t// collapse out\n\t\t\t\n\t\t\tthis.radiusInner.setTarget(1);\n\t\t\tthis.labelRadius.setTarget(1);\n\t\t\t//this.scale.setTarget(1); // TEMP\n\t\t}\n\t\telse\n\t\t{\n\t\t\t// don\'t collapse\n\t\t\t\n\t\t\tif ( compress )\n\t\t\t{\n\t\t\t\tthis.radiusInner.setTarget(compressedRadii[depthRelative - 1]);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.radiusInner.setTarget(nodeRadius * (depthRelative));\n\t\t\t}\n\t\t\t\n\t\t\t//this.scale.setTarget(1); // TEMP\n\t\t\t\n\t\t\tif ( this == selectedNode )\n\t\t\t{\n\t\t\t\tthis.labelRadius.setTarget(0);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tif ( compress )\n\t\t\t\t{\n\t\t\t\t\tthis.labelRadius.setTarget\n\t\t\t\t\t(\n\t\t\t\t\t\t(compressedRadii[depthRelative - 1] + compressedRadii[depthRelative]) / 2\n\t\t\t\t\t);\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tthis.labelRadius.setTarget(nodeRadius * (depthRelative) + nodeRadius / 2);\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\t\n//\t\tthis.r.start = this.r.end;\n//\t\tthis.g.start = this.g.end;\n//\t\tthis.b.start = this.b.end;\n\t\t\n\t\tthis.r.setTarget(255);\n\t\tthis.g.setTarget(255);\n\t\tthis.b.setTarget(255);\n\t\tthis.alphaLine.setTarget(0);\n\t\tthis.alphaArc.setTarget(0);\n\t\tthis.alphaWedge.setTarget(0);\n\t\tthis.alphaPattern.setTarget(0);\n\t\tthis.alphaOther.setTarget(0);\n\t\t\n\t\tif ( parentOfSelected && ! this.getCollapse() )\n\t\t{\n\t\t\tvar alpha =\n\t\t\t(\n\t\t\t\t1 -\n\t\t\t\t(selectedNode.getDepth() - this.getDepth()) /\n\t\t\t\t(Math.floor((compress ? compressedRadii[0] : nodeRadius) * gRadius / (historySpacingFactor * fontSize) - .5) + 1)\n\t\t\t);\n\t\t\t\n\t\t\tif ( alpha < 0 )\n\t\t\t{\n\t\t\t\talpha = 0;\n\t\t\t}\n\t\t\t\n\t\t\tthis.alphaLabel.setTarget(alpha);\n\t\t\tthis.radial = false;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tthis.alphaLabel.setTarget(0);\n\t\t}\n\t\t\n\t\tthis.hideAlonePrev = this.hideAlone;\n\t\tthis.hidePrev = this.hide;\n\t\t\n\t\tif ( parentOfSelected )\n\t\t{\n\t\t\tthis.hideAlone = false;\n\t\t\tthis.hide = false;\n\t\t}\n\t\t\n\t\tif ( this.getParent() == selectedNode.getParent() )\n\t\t{\n\t\t\tthis.hiddenEnd = null;\n\t\t}\n\t\t\n\t\tthis.radialPrev = this.radial;\n\t}\n\t\n\tthis.setTargetsSelected = function(hueMin, hueMax, lightness, hide, nextSiblingHidden)\n\t{\n\t\tvar collapse = this.getCollapse();\n\t\tvar depth = this.getDepth() - selectedNode.getDepth() + 1;\n\t\tvar canDisplayChildLabels = false;\n\t\tvar lastChild;\n\t\t\n\t\tif ( this.hasChildren() )//&& ! hide )\n\t\t{\n\t\t\tlastChild = this.children[this.children.length - 1];\n\t\t\tthis.hideAlone = true;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tthis.hideAlone = false;\n\t\t}\n\t\t\n\t\t// set child wedges\n\t\t//\n\t\tfor ( var i = 0; i < this.children.length; i++ )\n\t\t{\n\t\t\tthis.children[i].setTargetWedge();\n\t\t\t\n\t\t\tif\n\t\t\t(\n\t\t\t\t! this.children[i].hide &&\n\t\t\t\t( collapse || depth < maxDisplayDepth ) &&\n\t\t\t\tthis.depth < maxAbsoluteDepth\n\t\t\t)\n\t\t\t{\n\t\t\t\tcanDisplayChildLabels = true;\n\t\t\t\tthis.hideAlone = false;\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( this == selectedNode || lastChild && lastChild.angleEnd.end < this.angleEnd.end - .01)\n\t\t{\n\t\t\tthis.hideAlone = false;\n\t\t}\n\t\t\n\t\tif ( this.hideAlonePrev == undefined )\n\t\t{\n\t\t\tthis.hideAlonePrev = this.hideAlone;\n\t\t}\n\t\t\n\t\tif ( this == selectedNode )\n\t\t{\n\t\t\tvar otherArc = \n\t\t\t\tthis.children.length ?\n\t\t\t\t\tangleFactor *\n\t\t\t\t\t(\n\t\t\t\t\t\tthis.baseMagnitude + this.magnitude -\n\t\t\t\t\t\tlastChild.baseMagnitude - lastChild.magnitude\n\t\t\t\t\t)\n\t\t\t\t: this.baseMagnitude + this.magnitude;\n\t\t\tthis.canDisplayLabelOther =\n\t\t\t\tthis.children.length ?\n\t\t\t\t\totherArc *\n\t\t\t\t\t(this.children[0].radiusInner.end + 1) * gRadius >=\n\t\t\t\t\tminWidth()\n\t\t\t\t: true;\n\t\t\t\n\t\t\tthis.keyUnclassified = false;\n\t\t\t\n\t\t\tif ( this.canDisplayLabelOther )\n\t\t\t{\n\t\t\t\tthis.angleOther = Math.PI * 2 - otherArc / 2;\n\t\t\t}\n\t\t\telse if ( otherArc > 0.0000000001 )\n\t\t\t{\n\t\t\t\tthis.keyUnclassified = true;\n\t\t\t\tkeys++;\n\t\t\t}\n\t\t\t\n\t\t\tthis.angleStart.setTarget(0);\n\t\t\tthis.angleEnd.setTarget(Math.PI * 2);\n\t\t\t\n\t\t\tif ( this.children.length )\n\t\t\t{\n\t\t\t\tthis.radiusInner.setTarget(0);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.radiusInner.setTarget(compressedRadii[0]);\n\t\t\t}\n\t\t\t\n\t\t\tthis.hidePrev = this.hide;\n\t\t\tthis.hide = false;\n\t\t\tthis.hideAlonePrev = this.hideAlone;\n\t\t\tthis.hideAlone = false;\n\t\t\tthis.keyed = false;\n\t\t}\n\t\t\n\t\tif ( hueMax - hueMin > 1 / 12 )\n\t\t{\n\t\t\thueMax = hueMin + 1 / 12;\n\t\t}\n\t\t\n\t\t// set lightness\n\t\t//\n\t\tif ( ! ( hide || this.hideAlone ) )\n\t\t{\n\t\t\tif ( useHue() )\n\t\t\t{\n\t\t\t\tlightness = (lightnessBase + lightnessMax) / 2;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tlightness = lightnessBase + (depth - 1) * lightnessFactor;\n\t\t\t\t\n\t\t\t\tif ( lightness > lightnessMax )\n\t\t\t\t{\n\t\t\t\t\tlightness = lightnessMax;\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( hide )\n\t\t{\n\t\t\tthis.hide = true;\n\t\t}\n\t\t\n\t\tif ( this.hidePrev == undefined )\n\t\t{\n\t\t\tthis.hidePrev = this.hide;\n\t\t}\n\t\t\n\t\tvar hiddenStart = -1;\n\t\tvar hiddenHueNumer = 0;\n\t\tvar hiddenHueDenom = 0;\n\t\tvar i = 0;\n\t\t\n\t\tif ( ! this.hide )\n\t\t{\n\t\t\tthis.hiddenEnd = null;\n\t\t}\n\t\t\n\t\twhile ( true )\n\t\t{\n\t\t\tif ( ! this.hideAlone && ! hide && ( i == this.children.length || ! this.children[i].hide ) )\n\t\t\t{\n\t\t\t\t// reached a non-hidden child or the end; set targets for\n\t\t\t\t// previous group of hidden children (if any) using their\n\t\t\t\t// average hue\n\t\t\t\t\n\t\t\t\tif ( hiddenStart != -1 )\n\t\t\t\t{\n\t\t\t\t\tvar hiddenHue = hiddenHueDenom ? hiddenHueNumer / hiddenHueDenom : hueMin;\n\t\t\t\t\t\n\t\t\t\t\tfor ( var j = hiddenStart; j < i; j++ )\n\t\t\t\t\t{\n\t\t\t\t\t\tthis.children[j].setTargetsSelected\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\thiddenHue,\n\t\t\t\t\t\t\tnull,\n\t\t\t\t\t\t\tlightness,\n\t\t\t\t\t\t\tfalse,\n\t\t\t\t\t\t\tj < i - 1\n\t\t\t\t\t\t);\n\t\t\t\t\t\t\n\t\t\t\t\t\tthis.children[j].hiddenEnd = null;\n\t\t\t\t\t}\n\t\t\t\t\t\n\t\t\t\t\tthis.children[hiddenStart].hiddenEnd = i - 1;\n\t\t\t\t}\n\t\t\t}\n\t\t\t\n\t\t\tif ( i == this.children.length )\n\t\t\t{\n\t\t\t\tbreak;\n\t\t\t}\n\t\t\t\n\t\t\tvar child = this.children[i];\n\t\t\tvar childHueMin;\n\t\t\tvar childHueMax;\n\t\t\t\n\t\t\tif ( this.magnitude > 0 && ! this.hide && ! this.hideAlone )\n\t\t\t{\n\t\t\t\tif ( useHue() )\n\t\t\t\t{\n\t\t\t\t\tchildHueMin = child.hues[currentDataset];\n\t\t\t\t}\n\t\t\t\telse if ( this == selectedNode )\n\t\t\t\t{\n\t\t\t\t\tvar min = 0.0;\n\t\t\t\t\tvar max = 1.0;\n\t\t\t\t\t\n\t\t\t\t\tif ( this.children.length > 6 )\n\t\t\t\t\t{\n\t\t\t\t\t\tchildHueMin = lerp((1 - Math.pow(1 - i / this.children.length, 1.4)) * .95, 0, 1, min, max);\n\t\t\t\t\t\tchildHueMax = lerp((1 - Math.pow(1 - (i + .55) / this.children.length, 1.4)) * .95, 0, 1, min, max);\n\t\t\t\t\t}\n\t\t\t\t\telse\n\t\t\t\t\t{\n\t\t\t\t\t\tchildHueMin = lerp(i / this.children.length, 0, 1, min, max);\n\t\t\t\t\t\tchildHueMax = lerp((i + .55) / this.children.length, 0, 1, min, max);\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tchildHueMin = lerp\n\t\t\t\t\t(\n\t\t\t\t\t\tchild.baseMagnitude,\n\t\t\t\t\t\tthis.baseMagnitude, \n\t\t\t\t\t\tthis.baseMagnitude + this.magnitude,\n\t\t\t\t\t\thueMin,\n\t\t\t\t\t\thueMax\n\t\t\t\t\t);\n\t\t\t\t\tchildHueMax = lerp\n\t\t\t\t\t(\n\t\t\t\t\t\tchild.baseMagnitude + child.magnitude * .99,\n\t\t\t\t\t\tthis.baseMagnitude,\n\t\t\t\t\t\tthis.baseMagnitude + this.magnitude,\n\t\t\t\t\t\thueMin,\n\t\t\t\t\t\thueMax\n\t\t\t\t\t);\n\t\t\t\t}\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tchildHueMin = hueMin;\n\t\t\t\tchildHueMax = hueMax;\n\t\t\t}\n\t\t\t\n\t\t\tif ( ! this.hideAlone && ! hide && ! this.hide && child.hide )\n\t\t\t{\n\t\t\t\tif ( hiddenStart == -1 )\n\t\t\t\t{\n\t\t\t\t\thiddenStart = i;\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif ( useHue() )\n\t\t\t\t{\n\t\t\t\t\thiddenHueNumer += childHueMin * child.magnitude;\n\t\t\t\t\thiddenHueDenom += child.magnitude;\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\thiddenHueNumer += childHueMin;\n\t\t\t\t\thiddenHueDenom++;\n\t\t\t\t}\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\thiddenStart = -1;\n\t\t\t\t\n\t\t\t\tthis.children[i].setTargetsSelected\n\t\t\t\t(\n\t\t\t\t\tchildHueMin,\n\t\t\t\t\tchildHueMax,\n\t\t\t\t\tlightness,\n\t\t\t\t\thide || this.keyed || this.hideAlone || this.hide && ! collapse,\n\t\t\t\t\tfalse\n\t\t\t\t);\n\t\t\t}\n\t\t\t\n\t\t\ti++;\n\t\t}\n\t\t\n\t \tif ( this.hue && this.magnitude )\n\t \t{\n\t\t \tthis.hue.setTarget(this.hues[currentDataset]);\n\t\t\t\n\t\t\tif ( this.attributes[magnitudeIndex][lastDataset] == 0 )\n\t\t\t{\n\t\t\t\tthis.hue.start = this.hue.end;\n\t\t\t}\n\t\t}\n\t \t\n\t\tthis.radialPrev = this.radial;\n\t\t\n\t\tif ( this == selectedNode )\n\t\t{\n\t\t\tthis.resetLabelWidth();\n\t\t\tthis.labelWidth.setTarget(this.nameWidth * labelWidthFudge);\n\t\t\tthis.alphaWedge.setTarget(0);\n\t\t\tthis.alphaLabel.setTarget(1);\n\t\t\tthis.alphaOther.setTarget(1);\n\t\t\tthis.alphaArc.setTarget(0);\n\t\t\tthis.alphaLine.setTarget(0);\n\t\t\tthis.alphaPattern.setTarget(0);\n\t\t\tthis.r.setTarget(255);\n\t\t\tthis.g.setTarget(255);\n\t\t\tthis.b.setTarget(255);\n\t\t\tthis.radial = false;\n\t\t\tthis.labelRadius.setTarget(0);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tvar rgb = hslToRgb\n\t\t\t(\n\t\t\t\thueMin,\n\t\t\t\tsaturation,\n\t\t\t\tlightness\n\t\t\t);\n\t\t\t\n\t\t\tthis.r.setTarget(rgb.r);\n\t\t\tthis.g.setTarget(rgb.g);\n\t\t\tthis.b.setTarget(rgb.b);\n\t\t\tthis.alphaOther.setTarget(0);\n\t\t\t\n\t\t\tthis.alphaWedge.setTarget(1);\n\t\t\t\n\t\t\tif ( this.hide || this.hideAlone )\n\t\t\t{\n\t\t\t\tthis.alphaPattern.setTarget(1);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.alphaPattern.setTarget(0);\n\t\t\t}\n\t\t\t\n\t\t\t// set radial\n\t\t\t//\n\t\t\tif ( ! ( hide || this.hide ) )//&& ! this.keyed )\n\t\t\t{\n\t\t\t\tif ( this.hideAlone )\n\t\t\t\t{\n\t\t\t\t\tthis.radial = true;\n\t\t\t\t}\n\t\t\t\telse if ( false && canDisplayChildLabels )\n\t\t\t\t{\n\t\t\t\t\tthis.radial = false;\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tthis.radial = true;\n\t\t\t\t\t\n\t\t\t\t\tif ( this.hasChildren() && depth < maxDisplayDepth )\n\t\t\t\t\t{\n\t\t\t\t\t\tvar lastChild = this.children[this.children.length - 1];\n\t\t\t\t\t\t\n\t\t\t\t\t\tif\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\tlastChild.angleEnd.end == this.angleEnd.end ||\n\t\t\t\t\t\t\t(\n\t\t\t\t\t\t\t\t(this.angleStart.end + this.angleEnd.end) / 2 -\n\t\t\t\t\t\t\t\tlastChild.angleEnd.end\n\t\t\t\t\t\t\t) * (this.radiusInner.end + 1) * gRadius * 2 <\n\t\t\t\t\t\t\tminWidth()\n\t\t\t\t\t\t)\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tthis.radial = false;\n\t\t\t\t\t\t}\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\t\t\n\t\t\t// set alphaLabel\n\t\t\t//\n\t\t\tif\n\t\t\t(\n\t\t\t\tcollapse ||\n\t\t\t\thide ||\n\t\t\t\tthis.hide ||\n\t\t\t\tthis.keyed ||\n\t\t\t\tdepth > maxDisplayDepth ||\n\t\t\t\t! this.canDisplayDepth()\n\t\t\t)\n\t\t\t{\n\t\t\t\tthis.alphaLabel.setTarget(0);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tif\n\t\t\t\t(\n\t\t\t\t\t(this.radial || nLabelOffsets[depth - 2])\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\tthis.alphaLabel.setTarget(1);\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tthis.alphaLabel.setTarget(0);\n\t\t\t\t\t\n\t\t\t\t\tif ( this.radialPrev )\n\t\t\t\t\t{\n\t\t\t\t\t\tthis.alphaLabel.start = 0;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\t\t\n\t\t\t// set alphaArc\n\t\t\t//\n\t\t\tif\n\t\t\t(\n\t\t\t\tcollapse ||\n\t\t\t\thide ||\n\t\t\t\tdepth > maxDisplayDepth ||\n\t\t\t\t! this.canDisplayDepth()\n\t\t\t)\n\t\t\t{\n\t\t\t\tthis.alphaArc.setTarget(0);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.alphaArc.setTarget(1);\n\t\t\t}\n\t\t\t\n\t\t\t// set alphaLine\n\t\t\t//\n\t\t\tif\n\t\t\t(\n\t\t\t\thide ||\n\t\t\t\tthis.hide && nextSiblingHidden ||\n\t\t\t\tdepth > maxDisplayDepth ||\n\t\t\t\t! this.canDisplayDepth()\n\t\t\t)\n\t\t\t{\n\t\t\t\tthis.alphaLine.setTarget(0);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.alphaLine.setTarget(1);\n\t\t\t}\n\t\t\t\n\t\t\t//if (  ! this.radial )\n\t\t\t{\n\t\t\t\tthis.resetLabelWidth();\n\t\t\t}\n\t\t\t\n\t\t\t// set labelRadius target\n\t\t\t//\n\t\t\tif ( collapse )\n\t\t\t{\n\t\t\t\tthis.labelRadius.setTarget(this.radiusInner.end);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tif ( depth > maxDisplayDepth || ! this.canDisplayDepth() )\n\t\t\t\t{\n\t\t\t\t\tthis.labelRadius.setTarget(1);\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tthis.setTargetLabelRadius();\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t}\n\t\n\tthis.setTargetWedge = function()\n\t{\n\t\tvar depth = this.getDepth() - selectedNode.getDepth() + 1;\n\t\t\n\t\t// set angles\n\t\t//\n\t\tvar baseMagnitudeRelative = this.baseMagnitude - selectedNode.baseMagnitude;\n\t\t//\n\t\tthis.angleStart.setTarget(baseMagnitudeRelative * angleFactor);\n\t\tthis.angleEnd.setTarget((baseMagnitudeRelative + this.magnitude) * angleFactor);\n\t\t\n\t\t// set radiusInner\n\t\t//\n\t\tif ( depth > maxDisplayDepth || ! this.canDisplayDepth() )\n\t\t{\n\t\t\tthis.radiusInner.setTarget(1);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tif ( compress )\n\t\t\t{\n\t\t\t\tthis.radiusInner.setTarget(compressedRadii[depth - 2]);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.radiusInner.setTarget(nodeRadius * (depth - 1));\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( this.hide != undefined )\n\t\t{\n\t\t\tthis.hidePrev = this.hide;\n\t\t}\n\t\t\n\t\tif ( this.hideAlone != undefined )\n\t\t{\n\t\t\tthis.hideAlonePrev = this.hideAlone;\n\t\t}\n\t\t\n\t\t// set hide\n\t\t//\n\t\tif\n\t\t(\n\t\t\t(this.angleEnd.end - this.angleStart.end) *\n\t\t\t(this.radiusInner.end * gRadius + gRadius) <\n\t\t\tminWidth()\n\t\t)\n\t\t{\n\t\t\tif ( depth == 2 && ! this.getCollapse() && this.depth <= maxAbsoluteDepth )\n\t\t\t{\n\t\t\t\tthis.keyed = true;\n\t\t\t\tkeys++;\n\t\t\t\tthis.hide = false;\n\t\t\t\t\n\t\t\t\tvar percentage = this.getPercentage();\n\t\t\t\tthis.keyLabel = this.name + \'   \' + percentage + \'%\';\n\t\t\t\tvar dim = context.measureText(this.keyLabel);\n\t\t\t\tthis.keyNameWidth = dim.width;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tthis.keyed = false;\n\t\t\t\tthis.hide = depth > 2;\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tthis.keyed = false;\n\t\t\tthis.hide = false;\n\t\t}\n\t}\n\t\n\tthis.shortenLabel = function()\n\t{\n\t\tvar label = this.name;\n\t\t\n\t\tvar labelWidth = this.nameWidth;\n\t\tvar maxWidth = this.labelWidth.current();\n\t\tvar minEndLength = 0;\n\t\t\n\t\tif ( labelWidth > maxWidth && label.length > minEndLength * 2 )\n\t\t{\n\t\t\tvar endLength =\n\t\t\t\tMath.floor((label.length - 1) * maxWidth / labelWidth / 2);\n\t\t\t\n\t\t\tif ( endLength < minEndLength )\n\t\t\t{\n\t\t\t\tendLength = minEndLength;\n\t\t\t}\n\t\t\t\n\t\t\treturn (\n\t\t\t\tlabel.substring(0, endLength) +\n\t\t\t\t\'...\' +\n\t\t\t\tlabel.substring(label.length - endLength));\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn label;\n\t\t}\n\t}\n\t\n/*\tthis.shouldAddSearchResultsString = function()\n\t{\n\t\tif ( this.isSearchResult )\n\t\t{\n\t\t\treturn this.searchResults > 1;\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn this.searchResults > 0;\n\t\t}\n\t}\n*/\t\n\tthis.sort = function()\n\t{\n\t\tthis.children.sort(function(a, b){return b.getMagnitude() - a.getMagnitude()});\n\t\t\n\t\tfor (var i = 0; i < this.children.length; i++)\n\t\t{\n\t\t\tthis.children[i].sort();\n\t\t}\n\t}\n}\nvar options;\nfunction addOptionElement(position, innerHTML, title)\n{\n\tvar div = document.createElement("div");\n//\tdiv.style.position = \'absolute\';\n//\tdiv.style.top = position + \'px\';\n\tdiv.innerHTML = innerHTML;\n//\tdiv.style.display = \'block\';\n\tdiv.style.padding = \'2px\';\n\t\n\tif ( title )\n\t{\n\t\tdiv.title = title;\n\t}\n\t\n\toptions.appendChild(div);\n\tvar height = 0;//div.clientHeight;\n\treturn position + height;\n}\nfunction addOptionElements(hueName, hueDefault)\n{\n\toptions = document.createElement(\'div\');\n\toptions.style.position = \'absolute\';\n\toptions.style.top = \'0px\';\n\toptions.addEventListener(\'mousedown\', function(e) {mouseClick(e)}, false);\n//\toptions.onmouseup = function(e) {mouseUp(e)}\n\tdocument.body.appendChild(options);\n\t\n\tdocument.body.style.font = \'11px sans-serif\';\n\tvar position = 5;\n\t\n\tdetails = document.createElement(\'div\');\n\tdetails.style.position = \'absolute\';\n\tdetails.style.top = \'1%\';\n\tdetails.style.right = \'2%\';\n\tdetails.style.textAlign = \'right\';\n\tdocument.body.insertBefore(details, canvas);\n//\t\t<div id="details" style="position:absolute;top:1%;right:2%;text-align:right;">\n\tdetails.innerHTML = \'\\\n<span id="detailsName" style="font-weight:bold"></span>&nbsp;\\\n<input type="button" id="detailsExpand" onclick="expand(focusNode);"\\\nvalue="&harr;" title="Expand this wedge to become the new focus of the chart"/><br/>\\\n<div id="detailsInfo" style="float:right"></div>\';\n\tkeyControl = document.createElement(\'input\');\n\tkeyControl.type = \'button\';\n\tkeyControl.value = showKeys ? \'x\' : \'...\';\n\tkeyControl.style.position = \'\';\n\tkeyControl.style.position = \'fixed\';\n\tkeyControl.style.visibility = \'hidden\';\n\t\n\tdocument.body.insertBefore(keyControl, canvas);\n\t\n\tvar logoElement = document.getElementById(\'logo\');\n\t\n\tif ( logoElement )\n\t{\n\t\tlogoImage = logoElement.src;\n\t}\n\telse\n\t{\n\t\tlogoImage = \'http://marbl.github.io/Krona/img/logo-med.png\';\n\t}\n\t\n//\tdocument.getElementById(\'options\').style.fontSize = \'9pt\';\n\tposition = addOptionElement\n\t(\n\t\tposition,\n\'<a style="margin:2px" target="_blank" href="https://github.com/marbl/Krona/wiki"><img style="vertical-align:middle;width:108px;height:30px;" src="\' + logoImage + \'"/></a><input type="button" id="back" value="&larr;" title="Go back (Shortcut: &larr;)"/>\\\n<input type="button" id="forward" value="&rarr;" title="Go forward (Shortcut: &rarr;)"/> \\\n&nbsp;Search: <input type="text" id="search"/>\\\n<input id="searchClear" type="button" value="x" onclick="clearSearch()"/> \\\n<span id="searchResults"></span>\'\n\t);\n\t\n\tif ( datasets > 1 )\n\t{\n\t\tvar size = datasets < datasetSelectSize ? datasets : datasetSelectSize;\n\t\t\n\t\tvar select =\n\t\t\t\'<table style="border-collapse:collapse;padding:0px"><tr><td style="padding:0px">\' +\n\t\t\t\'<select id="datasets" style="min-width:100px" size="\' + size + \'" onchange="onDatasetChange()">\';\n\t\t\n\t\tfor ( var i = 0; i < datasetNames.length; i++ )\n\t\t{\n\t\t\tselect += \'<option>\' + datasetNames[i] + \'</option>\';\n\t\t}\n\t\t\n\t\tselect +=\n\t\t\t\'</select></td><td style="vertical-align:top;padding:1px;">\' +\n\t\t\t\'<input style="display:block" title="Previous dataset (Shortcut: &uarr;)" id="prevDataset" type="button" value="&uarr;" onclick="prevDataset()" disabled="true"/>\' +\n\t\t\t\'<input title="Next dataset (Shortcut: &darr;)" id="nextDataset" type="button" value="&darr;" onclick="nextDataset()"/><br/></td>\' +\n\t\t\t\'<td style="padding-top:1px;vertical-align:top"><input title="Switch to the last dataset that was viewed (Shortcut: TAB)" id="lastDataset" type="button" style="font:11px Times new roman" value="last" onclick="selectLastDataset()"/></td></tr></table>\';\n\t\t\n\t\tposition = addOptionElement(position + 5, select);\n\t\t\n\t\tdatasetDropDown = document.getElementById(\'datasets\');\n\t\tdatasetButtonLast = document.getElementById(\'lastDataset\');\n\t\tdatasetButtonPrev = document.getElementById(\'prevDataset\');\n\t\tdatasetButtonNext = document.getElementById(\'nextDataset\');\n\t\t\n\t\tposition += datasetDropDown.clientHeight;\n\t}\n\t\n\tposition = addOptionElement\n\t(\n\t\tposition + 5,\n\'<input type="button" id="maxAbsoluteDepthDecrease" value="-"/>\\\n<span id="maxAbsoluteDepth"></span>\\\n&nbsp;<input type="button" id="maxAbsoluteDepthIncrease" value="+"/> Max depth\',\n\'Maximum depth to display, counted from the top level \\\nand including collapsed wedges.\'\n\t);\n\t\n\tposition = addOptionElement\n\t(\n\t\tposition,\n\'<input type="button" id="fontSizeDecrease" value="-"/>\\\n<span id="fontSize"></span>\\\n&nbsp;<input type="button" id="fontSizeIncrease" value="+"/> Font size\'\n\t);\n\t\n\tposition = addOptionElement\n\t(\n\t\tposition,\n\'<input type="button" id="radiusDecrease" value="-"/>\\\n<input type="button" id="radiusIncrease" value="+"/> Chart size\'\n\t);\n\t\n\tif ( hueName )\n\t{\n\t\thueDisplayName = attributes[attributeIndex(hueName)].displayName;\n\t\t\n\t\tposition = addOptionElement\n\t\t(\n\t\t\tposition + 5,\n\t\t\t\'<input type="checkbox" id="useHue" style="float:left" \' +\n\t\t\t\'/><div>Color by<br/>\' + hueDisplayName +\n\t\t\t\'</div>\'\n\t\t);\n\t\t\n\t\tuseHueCheckBox = document.getElementById(\'useHue\');\n\t\tuseHueCheckBox.checked = hueDefault;\n\t\tuseHueCheckBox.onclick = handleResize;\n\t\tuseHueCheckBox.onmousedown = suppressEvent;\n\t}\n\t/*\n\tposition = addOptionElement\n\t(\n\t\tposition + 5,\n\t\t\'&nbsp;<input type="checkbox" id="shorten" checked="checked" />Shorten labels</div>\',\n\t\t\'Prevent labels from overlapping by shortening them\'\n\t);\n\t\n\tposition = addOptionElement\n\t(\n\t\tposition,\n\t\t\'&nbsp;<input type="checkbox" id="compress" checked="checked" />Compress\',\n\t\t\'Compress wedges if needed to show the entire depth\'\n\t);\n\t*/\n\tposition = addOptionElement\n\t(\n\t\tposition,\n\t\t\'<input type="checkbox" id="collapse" checked="checked" />Collapse\',\n\t\t\'Collapse wedges that are redundant (entirely composed of another wedge)\'\n\t);\n\t\n\tposition = addOptionElement\n\t(\n\t\tposition + 5,\n\t\t\'<input type="button" id="snapshot" value="Snapshot"/>\',\n\'Render the current view as SVG (Scalable Vector Graphics), a publication-\\\nquality format that can be printed and saved (see Help for browser compatibility)\'\n\t);\n\t\n\tposition = addOptionElement\n\t(\n\t\tposition + 5,\n\'<input type="button" id="linkButton" value="Link"/>\\\n<input type="text" size="30" id="linkText"/>\',\n\'Show a link to this view that can be copied for bookmarking or sharing\'\n\t);\n\t\n\tposition = addOptionElement\n\t(\n\t\tposition + 5,\n\'<input type="button" id="help" value="?"\\\nonclick="window.open(\\\'https://github.com/marbl/Krona/wiki/Browsing%20Krona%20charts\\\', \\\'help\\\')"/>\',\n\'Help\'\n\t);\n}\nfunction arrow(angleStart, angleEnd, radiusInner)\n{\n\tif ( context.globalAlpha == 0 )\n\t{\n\t\treturn;\n\t}\n\t\n\tvar angleCenter = (angleStart + angleEnd) / 2;\n\tvar radiusArrowInner = radiusInner - gRadius / 10;//nodeRadius * gRadius;\n\tvar radiusArrowOuter = gRadius * 1.1;//(1 + nodeRadius);\n\tvar radiusArrowCenter = (radiusArrowInner + radiusArrowOuter) / 2;\n\tvar pointLength = (radiusArrowOuter - radiusArrowInner) / 5;\n\t\n\tcontext.fillStyle = highlightFill;\n\tcontext.lineWidth = highlightLineWidth;\n\t\n\t// First, mask out the first half of the arrow.  This will prevent the tips\n\t// from superimposing if the arrow goes most of the way around the circle.\n\t// Masking is done by setting the clipping region to the inverse of the\n\t// half-arrow, which is defined by cutting the half-arrow out of a large\n\t// rectangle\n\t//\n\tcontext.beginPath();\n\tcontext.arc(0, 0, radiusInner, angleCenter, angleEnd, false);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowInner * Math.cos(angleEnd),\n\t\tradiusArrowInner * Math.sin(angleEnd)\n\t);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowCenter * Math.cos(angleEnd) - pointLength * Math.sin(angleEnd),\n\t\tradiusArrowCenter * Math.sin(angleEnd) + pointLength * Math.cos(angleEnd)\n\t);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowOuter * Math.cos(angleEnd),\n\t\tradiusArrowOuter * Math.sin(angleEnd)\n\t);\n\tcontext.arc(0, 0, gRadius, angleEnd, angleCenter, true);\n\tcontext.closePath();\n\tcontext.moveTo(-imageWidth, -imageHeight);\n\tcontext.lineTo(imageWidth, -imageHeight);\n\tcontext.lineTo(imageWidth, imageHeight);\n\tcontext.lineTo(-imageWidth, imageHeight);\n\tcontext.closePath();\n\tcontext.save();\n\tcontext.clip();\n\t\n\t// Next, draw the other half-arrow with the first half masked out\n\t//\n\tcontext.beginPath();\n\tcontext.arc(0, 0, radiusInner, angleCenter, angleStart, true);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowInner * Math.cos(angleStart),\n\t\tradiusArrowInner * Math.sin(angleStart)\n\t);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowCenter * Math.cos(angleStart) + pointLength * Math.sin(angleStart),\n\t\tradiusArrowCenter * Math.sin(angleStart) - pointLength * Math.cos(angleStart)\n\t);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowOuter * Math.cos(angleStart),\n\t\tradiusArrowOuter * Math.sin(angleStart)\n\t);\n\tcontext.arc(0, 0, gRadius, angleStart, angleCenter, false);\n\tcontext.fill();\n\tcontext.stroke();\n\t\n\t// Finally, remove the clipping region and draw the first half-arrow.  This\n\t// half is extended slightly to fill the seam.\n\t//\n\tcontext.restore();\n\tcontext.beginPath();\n\tcontext.arc(0, 0, radiusInner, angleCenter - 2 / (2 * Math.PI * radiusInner), angleEnd, false);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowInner * Math.cos(angleEnd),\n\t\tradiusArrowInner * Math.sin(angleEnd)\n\t);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowCenter * Math.cos(angleEnd) - pointLength * Math.sin(angleEnd),\n\t\tradiusArrowCenter * Math.sin(angleEnd) + pointLength * Math.cos(angleEnd)\n\t);\n\tcontext.lineTo\n\t(\n\t\tradiusArrowOuter * Math.cos(angleEnd),\n\t\tradiusArrowOuter * Math.sin(angleEnd)\n\t);\n\tcontext.arc(0, 0, gRadius, angleEnd, angleCenter - 2 / (2 * Math.PI * gRadius), true);\n\tcontext.fill();\n\tcontext.stroke();\n}\nfunction attributeIndex(aname)\n{\n\tfor ( var i = 0 ; i < attributes.length; i++ )\n\t{\n\t\tif ( aname == attributes[i].name )\n\t\t{\n\t\t\treturn i;\n\t\t}\n\t}\n\t\n\treturn null;\n}\nfunction checkHighlight()\n{\n\tvar lastHighlightedNode = highlightedNode;\n\tvar lastHighlightingHidden = highlightingHidden;\n\t\n\thighlightedNode = selectedNode;\n\tresetKeyOffset();\n\t\n\tif ( progress == 1 )\n\t{\n\t\tselectedNode.checkHighlight();\n\t\tif ( selectedNode.getParent() )\n\t\t{\n\t\t\tselectedNode.getParent().checkHighlightCenter();\n\t\t}\n\t\t\n\t\tfocusNode.checkHighlightMap();\n\t}\n\t\n\tif ( highlightedNode != selectedNode )\n\t{\n\t\tif ( highlightedNode == focusNode )\n\t\t{\n//\t\t\tcanvas.style.display=\'none\';\n//\t\t\twindow.resizeBy(1,0);\n//\t\t\tcanvas.style.cursor=\'ew-resize\';\n//\t\t\twindow.resizeBy(-1,0);\n//\t\t\tcanvas.style.display=\'inline\';\n\t\t}\n\t\telse\n\t\t{\n//\t\t\tcanvas.style.cursor=\'pointer\';\n\t\t}\n\t}\n\telse\n\t{\n//\t\tcanvas.style.cursor=\'auto\';\n\t}\n\t\n\tif\n\t(\n\t\t(\n\t\t\ttrue ||\n\t\t\thighlightedNode != lastHighlightedNode ||\n\t\t\thighlightingHidden != highlightingHiddenLast\n\t\t) &&\n\t\tprogress == 1\n\t)\n\t{\n\t\tdraw(); // TODO: handle in update()\n\t}\n}\nfunction checkSelectedCollapse()\n{\n\tvar newNode = selectedNode;\n\t\n\twhile ( newNode.getCollapse() )\n\t{\n\t\tnewNode = newNode.children[0];\n\t}\n\t\n\tif ( newNode.children.length == 0 && newNode.getParent() )\n\t{\n\t\tnewNode = newNode.getParent();\n\t}\n\t\n\tif ( newNode != selectedNode )\n\t{\n\t\tselectNode(newNode);\n\t}\n}\nfunction clearSearch()\n{\n\tif ( search.value != \'\' )\n\t{\n\t\tsearch.value = \'\';\n\t\tonSearchChange();\n\t}\n}\nfunction createSVG()\n{\n\tsvgNS = "http://www.w3.org/2000/svg";\n\tvar SVG = {};\n\tSVG.xlinkns = "http://www.w3.org/1999/xlink";\n\t\n\tvar newSVG = document.createElementNS(svgNS, "svg:svg");\n\t\n\tnewSVG.setAttribute("id", "canvas");\n\t// How big is the canvas in pixels\n\tnewSVG.setAttribute("width", \'100%\');\n\tnewSVG.setAttribute("height", \'100%\');\n\t// Set the coordinates used by drawings in the canvas\n//\tnewSVG.setAttribute("viewBox", "0 0 " + imageWidth + " " + imageHeight);\n\t// Define the XLink namespace that SVG uses\n\tnewSVG.setAttributeNS\n\t(\n\t\t"http://www.w3.org/2000/xmlns/",\n\t\t"xmlns:xlink",\n\t\tSVG.xlinkns\n\t);\n\t\n\treturn newSVG;\n}\nfunction degrees(radians)\n{\n\treturn radians * 180 / Math.PI;\n}\nfunction draw()\n{\n\ttweenFrames++;\n\t//resize();\n//\tcontext.fillRect(0, 0, imageWidth, imageHeight);\n\tcontext.clearRect(0, 0, imageWidth, imageHeight);\n\t\n\tcontext.font = fontNormal;\n\tcontext.textBaseline = \'middle\';\n\t\n\t//context.strokeStyle = \'rgba(0, 0, 0, 0.3)\';\n\tcontext.translate(centerX, centerY);\n\t\n\tresetKeyOffset();\n\t\n\thead.draw(false, false); // draw pie slices\n\thead.draw(true, false); // draw labels\n\t\n\tvar pathRoot = selectedNode;\n\t\n\tif ( focusNode != 0 && focusNode != selectedNode )\n\t{\n\t\tcontext.globalAlpha = 1;\n\t\tfocusNode.drawHighlight(true);\n\t\tpathRoot = focusNode;\n\t}\n\t\n\tif\n\t(\n\t\thighlightedNode &&\n\t\thighlightedNode.getDepth() >= selectedNode.getDepth() &&\n\t\thighlightedNode != focusNode\n\t)\n\t{\n\t\tif\n\t\t(\n\t\t\tprogress == 1 &&\n\t\t\thighlightedNode != selectedNode &&\n\t\t\t(\n\t\t\t\thighlightedNode != focusNode ||\n\t\t\t\tfocusNode.children.length > 0\n\t\t\t)\n\t\t)\n\t\t{\n\t\t\tcontext.globalAlpha = 1;\n\t\t\thighlightedNode.drawHighlight(true);\n\t\t}\n\t\t\n\t\t//pathRoot = highlightedNode;\n\t}\n\telse if\n\t(\n\t\tprogress == 1 &&\n\t\thighlightedNode.getDepth() < selectedNode.getDepth()\n\t)\n\t{\n\t\tcontext.globalAlpha = 1;\n\t\thighlightedNode.drawHighlightCenter();\n\t}\n\t\n\tif ( quickLook && false) // TEMP\n\t{\n\t\tcontext.globalAlpha = 1 - progress / 2;\n\t\tselectedNode.drawHighlight(true);\n\t}\n\telse if ( progress < 1 )//&& zoomOut() )\n\t{\n\t\tif ( !zoomOut)//() )\n\t\t{\n\t\t\tcontext.globalAlpha = selectedNode.alphaLine.current();\n\t\t\tselectedNode.drawHighlight(true);\n\t\t}\n\t\telse if ( selectedNodeLast )\n\t\t{\n\t\t\tcontext.globalAlpha = 1 - 4 * Math.pow(progress - .5, 2);\n\t\t\tselectedNodeLast.drawHighlight(false);\n\t\t}\n\t}\n\t\n\tdrawDatasetName();\n\t\n\t//drawHistory();\n\t\n\tcontext.translate(-centerX, -centerY);\n\tcontext.globalAlpha = 1;\n\t\n\tmapRadius =\n\t\t(imageHeight / 2 - details.clientHeight - details.offsetTop) /\n\t\t(pathRoot.getDepth() - 1) * 3 / 4 / 2;\n\t\n\tif ( mapRadius > maxMapRadius )\n\t{\n\t\tmapRadius = maxMapRadius;\n\t}\n\t\n\tmapBuffer = mapRadius / 2;\n\t\n\t//context.font = fontNormal;\n\tpathRoot.drawMap(pathRoot);\n\t\n\tif ( hueDisplayName && useHue() )\n\t{\n\t\tdrawLegend();\n\t}\n}\nfunction drawBubble(angle, radius, width, radial, flip)\n{\n\tvar height = fontSize * 2;\n\tvar x;\n\tvar y;\n\t\n\twidth = width + fontSize;\n\t\n\tif ( radial )\n\t{\n\t\ty = -fontSize;\n\t\t\n\t\tif ( flip )\n\t\t{\n\t\t\tx = radius - width + fontSize / 2;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tx = radius - fontSize / 2;\n\t\t}\n\t}\n\telse\n\t{\n\t\tx = -width / 2;\n\t\ty = -radius - fontSize;\n\t}\n\t\n\tif ( snapshotMode )\n\t{\n\t\tdrawBubbleSVG(x + centerX, y + centerY, width, height, fontSize, angle);\n\t}\n\telse\n\t{\n\t\tdrawBubbleCanvas(x, y, width, height, fontSize, angle);\n\t}\n}\nfunction drawBubbleCanvas(x, y, width, height, radius, rotation)\n{\n\tcontext.strokeStyle = \'black\';\n\tcontext.lineWidth = highlightLineWidth;\n\tcontext.fillStyle = \'rgba(255, 255, 255, .75)\';\n\tcontext.rotate(rotation);\n\troundedRectangle(x, y, width, fontSize * 2, fontSize);\n\tcontext.fill();\n\tcontext.stroke();\n\tcontext.rotate(-rotation);\n}\nfunction drawBubbleSVG(x, y, width, height, radius, rotation)\n{\n\tsvg +=\n\t\t\'<rect x="\' + x + \'" y="\' + y +\n\t\t\'" width="\' + width +\n\t\t\'" height="\' + height +\n\t\t\'" rx="\' + radius +\n\t\t\'" ry="\' + radius +\n\t\t\'" fill="rgba(255, 255, 255, .75)\' +\n\t\t\'" class="highlight" \' +\n\t\t\'transform="rotate(\' +\n\t\tdegrees(rotation) + \',\' + centerX + \',\' + centerY +\n\t\t\')"/>\';\n}\nfunction drawDatasetName()\n{\n\tvar alpha = datasetAlpha.current();\n\t\n\tif ( alpha > 0 )\n\t{\n\t\tvar radius = gRadius * compressedRadii[0] / -2;\n\t\t\n\t\tif ( alpha > 1 )\n\t\t{\n\t\t\talpha = 1;\n\t\t}\n\t\t\n\t\tcontext.globalAlpha = alpha;\n\t\t\n\t\tdrawBubble(0, -radius, datasetWidths[currentDataset], false, false);\n\t\tdrawText(datasetNames[currentDataset], 0, radius, 0, \'center\', true);\n\t}\n}\nfunction drawHistory()\n{\n\tvar alpha = 1;\n\tcontext.textAlign = \'center\';\n\t\n\tfor ( var i = 0; i < nodeHistoryPosition && alpha > 0; i++ )\n\t{\n\t\t\n\t\tcontext.globalAlpha = alpha - historyAlphaDelta * tweenFactor;\n\t\tcontext.fillText\n\t\t(\n\t\t\tnodeHistory[nodeHistoryPosition - i - 1].name,\n\t\t\t0,\n\t\t\t(i + tweenFactor) * historySpacingFactor * fontSize - 1\n\t\t);\n\t\t\n\t\tif ( alpha > 0 )\n\t\t{\n\t\t\talpha -= historyAlphaDelta;\n\t\t}\n\t}\n\t\n\tcontext.globalAlpha = 1;\n}\nfunction drawLegend()\n{\n\tvar left = imageWidth * .01;\n\tvar width = imageHeight * .0265;\n\tvar height = imageHeight * .15;\n\tvar top = imageHeight - fontSize * 3.5 - height;\n\tvar textLeft = left + width + fontSize / 2;\n\t\n\tcontext.fillStyle = \'black\';\n\tcontext.textAlign = \'start\';\n\tcontext.font = fontNormal;\n//\tcontext.fillText(valueStartText, textLeft, top + height);\n//\tcontext.fillText(valueEndText, textLeft, top);\n\tcontext.fillText(hueDisplayName, left, imageHeight - fontSize * 1.5);\n\t\n\tvar gradient = context.createLinearGradient(0, top + height, 0, top);\n\t\n\tfor ( var i = 0; i < hueStopPositions.length; i++ )\n\t{\n\t\tgradient.addColorStop(hueStopPositions[i], hueStopHsl[i]);\n\t\t\n\t\tvar textY = top + (1 - hueStopPositions[i]) * height;\n\t\t\n\t\tif\n\t\t(\n\t\t\ti == 0 ||\n\t\t\ti == hueStopPositions.length - 1 ||\n\t\t\ttextY > top + fontSize && textY < top + height - fontSize\n\t\t)\n\t\t{\n\t\t\tcontext.fillText(hueStopText[i], textLeft, textY);\n\t\t}\n\t}\n\t\n\tcontext.fillStyle = gradient;\n\tcontext.fillRect(left, top, width, height);\n\tcontext.lineWidth = thinLineWidth;\n\tcontext.strokeRect(left, top, width, height);\n}\nfunction drawLegendSVG()\n{\n\tvar left = imageWidth * .01;\n\tvar width = imageHeight * .0265;\n\tvar height = imageHeight * .15;\n\tvar top = imageHeight - fontSize * 3.5 - height;\n\tvar textLeft = left + width + fontSize / 2;\n\tvar text = \'\';\n\t\n\ttext += svgText(hueDisplayName, left, imageHeight - fontSize * 1.5);\n\t\n\tvar svgtest = \'<linearGradient id="gradient" x1="0%" y1="100%" x2="0%" y2="0%">\';\n\t\n\tfor ( var i = 0; i < hueStopPositions.length; i++ )\n\t{\n\t\tsvgtest +=\n\t\t\t\'<stop offset="\' + round(hueStopPositions[i] * 100) +\n\t\t\t\'%" style="stop-color:\' + hueStopHsl[i] + \'"/>\';\n\t\t\n\t\tvar textY = top + (1 - hueStopPositions[i]) * height;\n\t\t\n\t\tif\n\t\t(\n\t\t\ti == 0 ||\n\t\t\ti == hueStopPositions.length - 1 ||\n\t\t\ttextY > top + fontSize && textY < top + height - fontSize\n\t\t)\n\t\t{\n\t\t\ttext += svgText(hueStopText[i], textLeft, textY);\n\t\t}\n\t}\n\t\n\tsvgtest += \'</linearGradient>\';\n\t//alert(svgtest);\n\tsvg += svgtest;\n\tsvg +=\n\t\t\'<rect style="fill:url(#gradient)" x="\' + left + \'" y="\' + top +\n\t\t\'" width="\' + width + \'" height="\' + height + \'"/>\';\n\t\n\tsvg += text;\n}\nfunction drawSearchHighlights(label, bubbleX, bubbleY, rotation, center)\n{\n\tvar index = -1;\n\tvar labelLength = label.length;\n\t\n\tbubbleX -= fontSize / 4;\n\t\n\tdo\n\t{\n\t\tindex = label.toLowerCase().indexOf(search.value.toLowerCase(), index + 1);\n\t\t\n\t\tif ( index != -1 && index < labelLength )\n\t\t{\n\t\t\tvar dim = context.measureText(label.substr(0, index));\n\t\t\tvar x = bubbleX + dim.width;\n\t\t\t\n\t\t\tdim = context.measureText(label.substr(index, search.value.length));\n\t\t\t\n\t\t\tvar y = bubbleY - fontSize * 3 / 4;\n\t\t\tvar width = dim.width + fontSize / 2;\n\t\t\tvar height = fontSize * 3 / 2;\n\t\t\tvar radius = fontSize / 2;\n\t\t\t\n\t\t\tif ( snapshotMode )\n\t\t\t{\n\t\t\t\tif ( center )\n\t\t\t\t{\n\t\t\t\t\tx += centerX;\n\t\t\t\t\ty += centerY;\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tsvg +=\n\t\t\t\t\t\'<rect x="\' + x + \'" y="\' + y +\n\t\t\t\t\t\'" width="\' + width +\n\t\t\t\t\t\'" height="\' + height +\n\t\t\t\t\t\'" rx="\' + radius +\n\t\t\t\t\t\'" ry="\' + radius +\n\t\t\t\t\t\'" class="searchHighlight\' +\n\t\t\t\t\t\'" transform="rotate(\' +\n\t\t\t\t\tdegrees(rotation) + \',\' + centerX + \',\' + centerY +\n\t\t\t\t\t\')"/>\';\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tcontext.fillStyle = \'rgb(255, 255, 100)\';\n\t\t\t\tcontext.rotate(rotation);\n\t\t\t\troundedRectangle(x, y, width, height, radius);\n\t\t\t\tcontext.fill();\n\t\t\t\tcontext.rotate(-rotation);\n\t\t\t}\n\t\t}\n\t}\n\twhile ( index != -1 && index < labelLength );\n}\nfunction drawText(text, x, y, angle, anchor, bold, color)\n{\n\tif ( color == undefined )\n\t{\n\t\tcolor = \'black\';\n\t}\n\t\n\tif ( snapshotMode )\n\t{\n\t\tsvg +=\n\t\t\t\'<text x="\' + (centerX + x) + \'" y="\' + (centerY + y) +\n\t\t\t\'" text-anchor="\' + anchor + \'" style="font-color:\' + color + \';font-weight:\' + (bold ? \'bold\' : \'normal\') +\n\t\t\t\'" transform="rotate(\' + degrees(angle) + \',\' + centerX + \',\' + centerY + \')">\' +\n\t\t\ttext + \'</text>\';\n\t}\n\telse\n\t{\n\t\tcontext.fillStyle = color;\n\t\tcontext.textAlign = anchor;\n\t\tcontext.font = bold ? fontBold : fontNormal;\n\t\tcontext.rotate(angle);\n\t\tcontext.fillText(text, x, y);\n\t\tcontext.rotate(-angle);\n\t}\n}\nfunction drawTextPolar\n(\n\ttext,\n\tinnerText,\n\tangle,\n\tradius,\n\tradial,\n\tbubble,\n\tbold, \n\tsearchResult,\n\tsearchResults\n)\n{\n\tvar anchor;\n\tvar textX;\n\tvar textY;\n\tvar spacer;\n\tvar totalText = text;\n\tvar flip;\n\t\n\tif ( snapshotMode )\n\t{\n\t\tspacer = \'&#160;&#160;&#160;\';\n\t}\n\telse\n\t{\n\t\tspacer = \'   \';\n\t}\n\t\n\tif ( radial )\n\t{\n\t\tflip = angle < 3 * Math.PI / 2;\n\t\t\n\t\tif ( flip )\n\t\t{\n\t\t\tangle -= Math.PI;\n\t\t\tradius = -radius;\n\t\t\tanchor = \'end\';\n\t\t\t\n\t\t\tif ( innerText )\n\t\t\t{\n\t\t\t\ttotalText = text + spacer + innerText;\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tanchor = \'start\';\n\t\t\t\n\t\t\tif ( innerText )\n\t\t\t{\n\t\t\t\ttotalText = innerText + spacer + text;\n\t\t\t}\n\t\t}\n\t\t\n\t\ttextX = radius;\n\t\ttextY = 0;\n\t}\n\telse\n\t{\n\t\tflip = angle < Math.PI || angle > 2 * Math.PI;\n\t\tvar label;\n\t\t\n\t\tanchor = snapshotMode ? \'middle\' : \'center\';\n\t\t\n\t\tif ( flip )\n\t\t{\n\t\t\tangle -= Math.PI;\n\t\t\tradius = -radius;\n\t\t}\n\t\t\n\t\tangle += Math.PI / 2;\n\t\ttextX = 0;\n\t\ttextY = -radius;\n\t}\n\t\n\tif ( bubble )\n\t{\n\t\tvar textActual = totalText;\n\t\t\n\t\tif ( innerText && snapshotMode )\n\t\t{\n\t\t\tif ( flip )\n\t\t\t{\n\t\t\t\ttextActual = text + \'   \' + innerText;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\ttextActual = innerText + \'   \' + text;\n\t\t\t}\n\t\t}\n\t\t\n\t\tif ( searchResults )\n\t\t{\n\t\t\ttextActual = textActual + searchResultString(searchResults);\n\t\t}\n\t\t\n\t\tvar textWidth = measureText(textActual, bold);\n\t\t\n\t\tvar x = textX;\n\t\t\n\t\tif ( anchor == \'end\' )\n\t\t{\n\t\t\tx -= textWidth;\n\t\t}\n\t\telse if ( anchor != \'start\' )\n\t\t{\n\t\t\t// centered\n\t\t\tx -= textWidth / 2;\n\t\t}\n\t\t\n\t\tdrawBubble(angle, radius, textWidth, radial, flip);\n\t\t\n\t\tif ( searchResult )\n\t\t{\n\t\t\tdrawSearchHighlights\n\t\t\t(\n\t\t\t\ttextActual,\n\t\t\t\tx,\n\t\t\t\ttextY,\n\t\t\t\tangle,\n\t\t\t\ttrue\n\t\t\t)\n\t\t}\n\t}\n\t\n\tif ( searchResults )\n\t{\n\t\ttotalText = totalText + searchResultString(searchResults);\n\t}\n\t\n\tdrawText(totalText, textX, textY, angle, anchor, bold);\n\t\n\treturn flip;\n}\nfunction drawTick(start, length, angle)\n{\n\tif ( snapshotMode )\n\t{\n\t\tsvg +=\n\t\t\t\'<line x1="\' + (centerX + start) +\n\t\t\t\'" y1="\' + centerY +\n\t\t\t\'" x2="\' + (centerX + start + length) +\n\t\t\t\'" y2="\' + centerY +\n\t\t\t\'" class="tick" transform="rotate(\' +\n\t\t\tdegrees(angle) + \',\' + centerX + \',\' + centerY +\n\t\t\t\')"/>\';\n\t}\n\telse\n\t{\n\t\tcontext.rotate(angle);\n\t\tcontext.beginPath();\n\t\tcontext.moveTo(start, 0);\n\t\tcontext.lineTo(start + length, 0);\n\t\tcontext.lineWidth = thinLineWidth * 2;\n\t\tcontext.stroke();\n\t\tcontext.rotate(-angle);\n\t}\n}\nfunction drawWedge\n(\n\tangleStart,\n\tangleEnd,\n\tradiusInner,\n\tradiusOuter,\n\tcolor,\n\tpatternAlpha,\n\thighlight\n)\n{\n\tif ( context.globalAlpha == 0 )\n\t{\n\t\treturn;\n\t}\n\t\n\tif ( snapshotMode )\n\t{\n\t\tif ( angleEnd == angleStart + Math.PI * 2 )\n\t\t{\n\t\t\t// fudge to prevent overlap, which causes arc ambiguity\n\t\t\t//\n\t\t\tangleEnd -= .1 / gRadius;\n\t\t}\n\t\t\n\t\tvar longArc = angleEnd - angleStart > Math.PI ? 1 : 0;\n\t\t\n\t\tvar x1 = centerX + radiusInner * Math.cos(angleStart);\n\t\tvar y1 = centerY + radiusInner * Math.sin(angleStart);\n\t\t\n\t\tvar x2 = centerX + gRadius * Math.cos(angleStart);\n\t\tvar y2 = centerY + gRadius * Math.sin(angleStart);\n\t\t\n\t\tvar x3 = centerX + gRadius * Math.cos(angleEnd);\n\t\tvar y3 = centerY + gRadius * Math.sin(angleEnd);\n\t\t\n\t\tvar x4 = centerX + radiusInner * Math.cos(angleEnd);\n\t\tvar y4 = centerY + radiusInner * Math.sin(angleEnd);\n\t\t\n\t\tvar dArray =\n\t\t[\n\t\t\t" M ", x1, ",", y1,\n\t\t\t" L ", x2, ",", y2,\n\t\t\t" A ", gRadius, ",", gRadius, " 0 ", longArc, ",1 ", x3, ",", y3,\n\t\t\t" L ", x4, ",", y4,\n\t\t\t" A ", radiusInner, ",", radiusInner, " 0 ", longArc, " 0 ", x1, ",", y1,\n\t\t\t" Z "\n\t\t];\n\t\t\n\t\tsvg +=\n\t\t\t\'<path class="\'+ (highlight ? \'highlight\' : \'wedge\') + \'" fill="\' + color +\n\t\t\t\'" d="\' + dArray.join(\'\') + \'"/>\';\n\t\t\n\t\tif ( patternAlpha > 0 )\n\t\t{\n\t\t\tsvg +=\n\t\t\t\t\'<path class="wedge" fill="url(#hiddenPattern)" d="\' +\n\t\t\t\tdArray.join(\'\') + \'"/>\';\n\t\t}\n\t}\n\telse\n\t{\n\t\t// fudge to prevent seams during animation\n\t\t//\n\t\tangleEnd += 1 / gRadius;\n\t\t\n\t\tcontext.fillStyle = color;\n\t\tcontext.beginPath();\n\t\tcontext.arc(0, 0, radiusInner, angleStart, angleEnd, false);\n\t\tcontext.arc(0, 0, radiusOuter, angleEnd, angleStart, true);\n\t\tcontext.closePath();\n\t\tcontext.fill();\n\t\t\n\t\tif ( patternAlpha > 0 )\n\t\t{\n\t\t\tcontext.save();\n\t\t\tcontext.clip();\n\t\t\tcontext.globalAlpha = patternAlpha;\n\t\t\tcontext.fillStyle = hiddenPattern;\n\t\t\tcontext.fill();\n\t\t\tcontext.restore();\n\t\t}\n\t\t\n\t\tif ( highlight )\n\t\t{\n\t\t\tcontext.lineWidth = highlight ? highlightLineWidth : thinLineWidth;\n\t\t\tcontext.strokeStyle = \'black\';\n\t\t\tcontext.stroke();\n\t\t}\n\t}\n}\nfunction expand(node)\n{\n\tselectNode(node);\n\tupdateView();\n}\nfunction focusLost()\n{\n\tmouseX = -1;\n\tmouseY = -1;\n\tcheckHighlight();\n\tdocument.body.style.cursor = \'auto\';\n}\nfunction fontSizeDecrease()\n{\n\tif ( fontSize > 1 )\n\t{\n\t\tfontSize--;\n\t\tupdateViewNeeded = true;\n\t}\n}\nfunction fontSizeIncrease()\n{\n\tfontSize++;\n\tupdateViewNeeded = true;\n}\nfunction getGetString(name, value, bool)\n{\n\treturn name + \'=\' + (bool ? value ? \'true\' : \'false\' : value);\n}\nfunction hideLink()\n{\n\thide(linkText);\n\tshow(linkButton);\n}\nfunction show(object)\n{\n\tobject.style.display = \'inline\';\n}\nfunction hide(object)\n{\n\tobject.style.display = \'none\';\n}\nfunction showLink()\n{\n\tvar urlHalves = String(document.location).split(\'?\');\n\tvar newGetVariables = new Array();\n\t\n\tnewGetVariables.push\n\t(\n\t\tgetGetString(\'dataset\', currentDataset, false),\n\t\tgetGetString(\'node\', selectedNode.id, false),\n\t\tgetGetString(\'collapse\', collapse, true),\n\t\tgetGetString(\'color\', useHue(), true),\n\t\tgetGetString(\'depth\', maxAbsoluteDepth - 1, false),\n\t\tgetGetString(\'font\', fontSize, false),\n\t\tgetGetString(\'key\', showKeys, true)\n\t);\n\t\n\thide(linkButton);\n\tshow(linkText);\n\tlinkText.value = urlHalves[0] + \'?\' + getVariables.concat(newGetVariables).join(\'&\');\n\t//linkText.disabled = false;\n\tlinkText.focus();\n\tlinkText.select();\n\t//linkText.disabled = true;\n//\tdocument.location = urlHalves[0] + \'?\' + getVariables.join(\'&\');\n}\nfunction getFirstChild(element)\n{\n\telement = element.firstChild;\n\t\n\tif ( element && element.nodeType != 1 )\n\t{\n\t\telement = getNextSibling(element);\n\t}\n\t\n\treturn element;\n}\nfunction getNextSibling(element)\n{\n\tdo\n\t{\n\t\telement = element.nextSibling;\n\t}\n\twhile ( element && element.nodeType != 1 );\n\t\n\treturn element;\n}\nfunction getPercentage(fraction)\n{\n\treturn round(fraction * 100);\n}\nfunction hslText(hue)\n{\n\tif ( 1 || snapshotMode )\n\t{\n\t\t// Safari doesn\'t seem to allow hsl() in SVG\n\t\t\n\t\tvar rgb = hslToRgb(hue, saturation, (lightnessBase + lightnessMax) / 2);\n\t\t\n\t\treturn rgbText(rgb.r, rgb.g, rgb.b);\n\t}\n\telse\n\t{\n\t\tvar hslArray =\n\t\t[\n\t\t\t\'hsl(\',\n\t\t\tMath.floor(hue * 360),\n\t\t\t\',\',\n\t\t\tMath.floor(saturation * 100),\n\t\t\t\'%,\',\n\t\t\tMath.floor((lightnessBase + lightnessMax) * 50),\n\t\t\t\'%)\'\n\t\t];\n\t\t\n\t\treturn hslArray.join(\'\');\n\t}\n}\nfunction hslToRgb(h, s, l)\n{\n\tvar m1, m2;\n\tvar r, g, b;\n\t\n\tif (s == 0)\n\t{\n\t\tr = g = b = Math.floor((l * 255));\n\t}\n\telse\n\t{\n\t\tif (l <= 0.5)\n\t\t{\n\t\t\tm2 = l * (s + 1);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tm2 = l + s - l * s;\n\t\t}\n\t\t\n\t\tm1 = l * 2 - m2;\n\t\t\n\t\tr = Math.floor(hueToRgb(m1, m2, h + 1 / 3));\n\t\tg = Math.floor(hueToRgb(m1, m2, h));\n\t\tb = Math.floor(hueToRgb(m1, m2, h - 1/3));\n\t}\n\t\n\treturn {r: r, g: g, b: b};\n}\nfunction hueToRgb(m1, m2, hue)\n{\n\tvar v;\n\t\n\twhile (hue < 0)\n\t{\n\t\thue += 1;\n\t}\n\t\n\twhile (hue > 1)\n\t{\n\t\thue -= 1;\n\t}\n\t\n\tif (6 * hue < 1)\n\t\tv = m1 + (m2 - m1) * hue * 6;\n\telse if (2 * hue < 1)\n\t\tv = m2;\n\telse if (3 * hue < 2)\n\t\tv = m1 + (m2 - m1) * (2/3 - hue) * 6;\n\telse\n\t\tv = m1;\n\treturn 255 * v;\n}\nfunction interpolateHue(hueStart, hueEnd, valueStart, valueEnd)\n{\n\t// since the gradient will be RGB based, we need to add stops to hit all the\n\t// colors in the hue spectrum\n\t\n\thueStopPositions = new Array();\n\thueStopHsl = new Array();\n\thueStopText = new Array();\n\t\n\thueStopPositions.push(0);\n\thueStopHsl.push(hslText(hueStart));\n\thueStopText.push(round(valueStart));\n\t\n\tfor\n\t(\n\t\tvar i = (hueStart > hueEnd ? 5 / 6 : 1 / 6);\n\t\t(hueStart > hueEnd ? i > 0 : i < 1);\n\t\ti += (hueStart > hueEnd ? -1 : 1) / 6\n\t)\n\t{\n\t\tif\n\t\t(\n\t\t\thueStart > hueEnd ?\n\t\t\t\ti > hueEnd && i < hueStart :\n\t\t\t\ti > hueStart && i < hueEnd\n\t\t)\n\t\t{\n\t\t\thueStopPositions.push(lerp(i, hueStart, hueEnd, 0, 1));\n\t\t\thueStopHsl.push(hslText(i));\n\t\t\thueStopText.push(round(lerp\n\t\t\t(\n\t\t\t\ti,\n\t\t\t\thueStart,\n\t\t\t\thueEnd,\n\t\t\t\tvalueStart,\n\t\t\t\tvalueEnd\n\t\t\t)));\n\t\t}\n\t}\n\t\n\thueStopPositions.push(1);\n\thueStopHsl.push(hslText(hueEnd));\n\thueStopText.push(round(valueEnd));\n}\nfunction keyLineAngle(angle, keyAngle, bendRadius, keyX, keyY, pointsX, pointsY)\n{\n\tif ( angle < Math.PI / 2 && keyY < bendRadius * Math.sin(angle) \n\t|| angle > Math.PI / 2 && keyY < bendRadius)\n\t{\n\t\treturn Math.asin(keyY / bendRadius);\n\t}\n\telse\n\t{\n\t\t// find the angle of the normal to a tangent line that goes to\n\t\t// the label\n\t\t\n\t\tvar textDist = Math.sqrt\n\t\t(\n\t\t\tMath.pow(keyX, 2) +\n\t\t\tMath.pow(keyY, 2)\n\t\t);\n\t\t\n\t\tvar tanAngle = Math.acos(bendRadius / textDist) + keyAngle;\n\t\t\n\t\tif ( angle < tanAngle || angle < Math.PI / 2 )//|| labelLeft < centerX )\n\t\t{\n\t\t\t// angle doesn\'t reach far enough for tangent; collapse and\n\t\t\t// connect directly to label\n\t\t\t\n\t\t\tif ( keyY / Math.tan(angle) > 0 )\n\t\t\t{\n\t\t\t\tpointsX.push(keyY / Math.tan(angle));\n\t\t\t\tpointsY.push(keyY);\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tpointsX.push(bendRadius * Math.cos(angle));\n\t\t\t\tpointsY.push(bendRadius * Math.sin(angle));\n\t\t\t}\n\t\t\t\n\t\t\treturn angle;\n\t\t}\n\t\telse\n\t\t{\n\t\t\treturn tanAngle;\n\t\t}\n\t}\n}\nfunction keyOffset()\n{\n\treturn imageHeight - (keys - currentKey + 1) * (keySize + keyBuffer) + keyBuffer - margin;\n}\nfunction lerp(value, fromStart, fromEnd, toStart, toEnd)\n{\n\treturn (value - fromStart) *\n\t\t(toEnd - toStart) /\n\t\t(fromEnd - fromStart) +\n\t\ttoStart;\n}\nfunction createCanvas()\n{\n\tcanvas = document.createElement(\'canvas\');\n\tdocument.body.appendChild(canvas);\n\tcontext = canvas.getContext(\'2d\');\n}\nfunction load()\n{\n\tdocument.body.style.overflow = "hidden";\n\tdocument.body.style.margin = 0;\n\t\n\tcreateCanvas();\n\t\n\tif ( context == undefined )\n\t{\n\t\tdocument.body.innerHTML = \'\\\n<br/>This browser does not support HTML5 (see \\\n<a href="https://github.com/marbl/Krona/wiki/Browser%20support">Browser support</a>).\\\n\t\';\n\t\treturn;\n\t}\n\tif ( typeof context.fillText != \'function\' )\n\t{\n\t\tdocument.body.innerHTML = \'\\\n<br/>This browser does not support HTML5 canvas text (see \\\n<a href="https://github.com/marbl/Krona/wiki/Browser%20support">Browser support</a>).\\\n\t\';\n\t\treturn;\n\t}\n\t\n\tresize();\n\t\n\tvar kronaElement = document.getElementsByTagName(\'krona\')[0];\n\t\n\tvar magnitudeName;\n\tvar hueName;\n\tvar hueDefault;\n\tvar hueStart;\n\tvar hueEnd;\n\tvar valueStart;\n\tvar valueEnd;\n\t\n\tif ( kronaElement.getAttribute(\'collapse\') != undefined )\n\t{\n\t\tcollapse = kronaElement.getAttribute(\'collapse\') == \'true\';\n\t}\n\t\n\tif ( kronaElement.getAttribute(\'key\') != undefined )\n\t{\n\t\tshowKeys = kronaElement.getAttribute(\'key\') == \'true\';\n\t}\n\t\n\tfor\n\t(\n\t\tvar element = getFirstChild(kronaElement);\n\t\telement;\n\t\telement = getNextSibling(element)\n\t)\n\t{\n\t\tswitch ( element.tagName.toLowerCase() )\n\t\t{\n\t\t\tcase \'attributes\':\n\t\t\t\tmagnitudeName = element.getAttribute(\'magnitude\');\n\t\t\t\t//\n\t\t\t\tfor\n\t\t\t\t(\n\t\t\t\t\tvar attributeElement = getFirstChild(element);\n\t\t\t\t\tattributeElement;\n\t\t\t\t\tattributeElement = getNextSibling(attributeElement)\n\t\t\t\t)\n\t\t\t\t{\n\t\t\t\t\tvar tag = attributeElement.tagName.toLowerCase();\n\t\t\t\t\t\n\t\t\t\t\tif ( tag == \'attribute\' )\n\t\t\t\t\t{\n\t\t\t\t\t\tvar attribute = new Attribute();\n\t\t\t\t\t\tattribute.name = attributeElement.firstChild.nodeValue.toLowerCase();\n\t\t\t\t\t\tattribute.displayName = attributeElement.getAttribute(\'display\');\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attributeElement.getAttribute(\'hrefBase\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.hrefBase = attributeElement.getAttribute(\'hrefBase\');\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attributeElement.getAttribute(\'target\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.target = attributeElement.getAttribute(\'target\');\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attribute.name == magnitudeName )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tmagnitudeIndex = attributes.length;\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attributeElement.getAttribute(\'listAll\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.listAll = attributeElement.getAttribute(\'listAll\').toLowerCase();\n\t\t\t\t\t\t}\n\t\t\t\t\t\telse if ( attributeElement.getAttribute(\'listNode\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.listNode = attributeElement.getAttribute(\'listNode\').toLowerCase();\n\t\t\t\t\t\t}\n\t\t\t\t\t\telse if ( attributeElement.getAttribute(\'dataAll\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.dataAll = attributeElement.getAttribute(\'dataAll\').toLowerCase();\n\t\t\t\t\t\t}\n\t\t\t\t\t\telse if ( attributeElement.getAttribute(\'dataNode\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.dataNode = attributeElement.getAttribute(\'dataNode\').toLowerCase();\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attributeElement.getAttribute(\'postUrl\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.postUrl = attributeElement.getAttribute(\'postUrl\');\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attributeElement.getAttribute(\'postVar\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.postVar = attributeElement.getAttribute(\'postVar\');\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attributeElement.getAttribute(\'mono\') )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\tattribute.mono = true;\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tattributes.push(attribute);\n\t\t\t\t\t}\n\t\t\t\t\telse if ( tag == \'list\' )\n\t\t\t\t\t{\n\t\t\t\t\t\tvar attribute = new Attribute();\n\t\t\t\t\t\t\n\t\t\t\t\t\tattribute.name = attributeElement.firstChild.nodeValue;\n\t\t\t\t\t\tattribute.list = true;\n\t\t\t\t\t\tattributes.push(attribute);\n\t\t\t\t\t}\n\t\t\t\t\telse if ( tag == \'data\' )\n\t\t\t\t\t{\n\t\t\t\t\t\tvar attribute = new Attribute();\n\t\t\t\t\t\t\n\t\t\t\t\t\tattribute.name = attributeElement.firstChild.nodeValue;\n\t\t\t\t\t\tattribute.data = true;\n\t\t\t\t\t\tattributes.push(attribute);\n\t\t\t\t\t\t\n\t\t\t\t\t\tvar enableScript = document.createElement(\'script\');\n\t\t\t\t\t\tvar date = new Date();\n\t\t\t\t\t\tenableScript.src =\n\t\t\t\t\t\t\tattributeElement.getAttribute(\'enable\') + \'?\' +\n\t\t\t\t\t\t\tdate.getTime();\n\t\t\t\t\t\tdocument.body.appendChild(enableScript);\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\tbreak;\n\t\t\t\n\t\t\tcase \'color\':\n\t\t\t\thueName = element.getAttribute(\'attribute\');\n\t\t\t\thueStart = Number(element.getAttribute(\'hueStart\')) / 360;\n\t\t\t\thueEnd = Number(element.getAttribute(\'hueEnd\')) / 360;\n\t\t\t\tvalueStart = Number(element.getAttribute(\'valueStart\'));\n\t\t\t\tvalueEnd = Number(element.getAttribute(\'valueEnd\'));\n\t\t\t\t//\n\t\t\t\tinterpolateHue(hueStart, hueEnd, valueStart, valueEnd);\n\t\t\t\t//\n\t\t\t\tif ( element.getAttribute(\'default\') == \'true\' )\n\t\t\t\t{\n\t\t\t\t\thueDefault = true;\n\t\t\t\t}\n\t\t\t\tbreak;\n\t\t\t\n\t\t\tcase \'datasets\':\n\t\t\t\tdatasetNames = new Array();\n\t\t\t\t//\n\t\t\t\tfor ( j = getFirstChild(element); j; j = getNextSibling(j) )\n\t\t\t\t{\n\t\t\t\t\tdatasetNames.push(j.firstChild.nodeValue);\n\t\t\t\t}\n\t\t\t\tdatasets = datasetNames.length;\n\t\t\t\tbreak;\n\t\t\t\n\t\t\tcase \'node\':\n\t\t\t\thead = loadTreeDOM\n\t\t\t\t(\n\t\t\t\t\telement,\n\t\t\t\t\tmagnitudeName,\n\t\t\t\t\thueName,\n\t\t\t\t\thueStart,\n\t\t\t\t\thueEnd,\n\t\t\t\t\tvalueStart,\n\t\t\t\t\tvalueEnd\n\t\t\t\t);\n\t\t\t\tbreak;\n\t\t}\n\t}\n\t\n\t// get GET options\n\t//\n\tvar urlHalves = String(document.location).split(\'?\');\n\tvar datasetDefault = 0;\n\tvar maxDepthDefault;\n\tvar nodeDefault = 0;\n\t//\n\tif ( urlHalves[1] )\n\t{\n\t\tvar vars = urlHalves[1].split(\'&\');\n\t\t\n\t\tfor ( i = 0; i < vars.length; i++ )\n\t\t{\n\t\t\tvar pair = vars[i].split(\'=\');\n\t\t\t\n\t\t\tswitch ( pair[0] )\n\t\t\t{\n\t\t\t\tcase \'collapse\':\n\t\t\t\t\tcollapse = pair[1] == \'true\';\n\t\t\t\t\tbreak;\n\t\t\t\t\n\t\t\t\tcase \'color\':\n\t\t\t\t\thueDefault = pair[1] == \'true\';\n\t\t\t\t\tbreak;\n\t\t\t\t\n\t\t\t\tcase \'dataset\':\n\t\t\t\t\tdatasetDefault = Number(pair[1]);\n\t\t\t\t\tbreak;\n\t\t\t\t\t\n\t\t\t\tcase \'depth\':\n\t\t\t\t\tmaxDepthDefault = Number(pair[1]) + 1;\n\t\t\t\t\tbreak;\n\t\t\t\t\n\t\t\t\tcase \'key\':\n\t\t\t\t\tshowKeys = pair[1] == \'true\';\n\t\t\t\t\tbreak;\n\t\t\t\t\n\t\t\t\tcase \'font\':\n\t\t\t\t\tfontSize = Number(pair[1]);\n\t\t\t\t\tbreak;\n\t\t\t\t\n\t\t\t\tcase \'node\':\n\t\t\t\t\tnodeDefault = Number(pair[1]);\n\t\t\t\t\tbreak;\n\t\t\t\t\n\t\t\t\tdefault:\n\t\t\t\t\tgetVariables.push(pair[0] + \'=\' + pair[1]);\n\t\t\t\t\tbreak;\n\t\t\t}\n\t\t}\n\t}\n\t\n\taddOptionElements(hueName, hueDefault);\n\tsetCallBacks();\n\t\n\thead.sort();\n\tmaxAbsoluteDepth = 0;\n\tselectDataset(datasetDefault);\n\t\n\tif ( maxDepthDefault && maxDepthDefault < head.maxDepth )\n\t{\n\t\tmaxAbsoluteDepth = maxDepthDefault;\n\t}\n\telse\n\t{\n\t\tmaxAbsoluteDepth = head.maxDepth;\n\t}\n\t\n\tselectNode(nodes[nodeDefault]);\n\t\n\tsetInterval(update, 20);\n\t\n\twindow.onresize = handleResize;\n\tupdateMaxAbsoluteDepth();\n\tupdateViewNeeded = true;\n}\nfunction loadTreeDOM\n(\n\tdomNode,\n\tmagnitudeName,\n\thueName,\n\thueStart,\n\thueEnd,\n\tvalueStart,\n\tvalueEnd\n)\n{\n\tvar newNode = new Node();\n\t\n\tnewNode.name = domNode.getAttribute(\'name\');\n\t\n\tif ( domNode.getAttribute(\'href\') )\n\t{\n\t\tnewNode.href = domNode.getAttribute(\'href\');\n\t}\n\t\n\tif ( hueName )\n\t{\n\t\tnewNode.hues = new Array();\n\t}\n\t\n\tfor ( var i = getFirstChild(domNode); i; i = getNextSibling(i) )\n\t{\n\t\tswitch ( i.tagName.toLowerCase() )\n\t\t{\n\t\tcase \'node\': \n\t\t\tvar newChild = loadTreeDOM\n\t\t\t(\n\t\t\t\ti,\n\t\t\t\tmagnitudeName,\n\t\t\t\thueName,\n\t\t\t\thueStart,\n\t\t\t\thueEnd,\n\t\t\t\tvalueStart,\n\t\t\t\tvalueEnd\n\t\t\t);\n\t\t\tnewChild.parent = newNode;\n\t\t\tnewNode.children.push(newChild);\n\t\t\tbreak;\n\t\t\t\n\t\tdefault:\n\t\t\tvar attributeName = i.tagName.toLowerCase();\n\t\t\tvar index = attributeIndex(attributeName);\n\t\t\t//\n\t\t\tnewNode.attributes[index] = new Array();\n\t\t\t//\n\t\t\tfor ( var j = getFirstChild(i); j; j = getNextSibling(j) )\n\t\t\t{\n\t\t\t\tif ( attributes[index] == undefined )\n\t\t\t\t{\n\t\t\t\t\tvar x = 5;\n\t\t\t\t}\n\t\t\t\tif ( attributes[index].list )\n\t\t\t\t{\n\t\t\t\t\tnewNode.attributes[index].push(new Array());\n\t\t\t\t\t\n\t\t\t\t\tfor ( var k = getFirstChild(j); k; k = getNextSibling(k) )\n\t\t\t\t\t{\n\t\t\t\t\t\tnewNode.attributes[index][newNode.attributes[index].length - 1].push(k.firstChild.nodeValue);\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\telse\n\t\t\t\t{\n\t\t\t\t\tvar value = j.firstChild ? j.firstChild.nodeValue : \'\';\n\t\t\t\t\t\n\t\t\t\t\tif ( j.getAttribute(\'href\') )\n\t\t\t\t\t{\n\t\t\t\t\t\tvar target;\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( attributes[index].target )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\ttarget = \' target="\' + attributes[index].target + \'"\';\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tvalue = \'<a href="\' + attributes[index].hrefBase + j.getAttribute(\'href\') + \'"\' + target + \'>\' + value + \'</a>\';\n\t\t\t\t\t}\n\t\t\t\t\t\n\t\t\t\t\tnewNode.attributes[index].push(value);\n\t\t\t\t}\n\t\t\t}\n\t\t\t//\n\t\t\tif ( attributeName == magnitudeName || attributeName == hueName )\n\t\t\t{\n\t\t\t\tfor ( j = 0; j < datasets; j++ )\n\t\t\t\t{\n\t\t\t\t\tvar value = newNode.attributes[index][j] == undefined ? 0 : Number(newNode.attributes[index][j]);\n\t\t\t\t\t\n\t\t\t\t\tnewNode.attributes[index][j] = value;\n\t\t\t\t\t\n\t\t\t\t\tif ( attributeName == hueName )\n\t\t\t\t\t{\n\t\t\t\t\t\tvar hue = lerp\n\t\t\t\t\t\t(\n\t\t\t\t\t\t\tvalue,\n\t\t\t\t\t\t\tvalueStart,\n\t\t\t\t\t\t\tvalueEnd,\n\t\t\t\t\t\t\thueStart,\n\t\t\t\t\t\t\thueEnd\n\t\t\t\t\t\t);\n\t\t\t\t\t\t\n\t\t\t\t\t\tif ( hue < hueStart == hueStart < hueEnd )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\thue = hueStart;\n\t\t\t\t\t\t}\n\t\t\t\t\t\telse if ( hue > hueEnd == hueStart < hueEnd )\n\t\t\t\t\t\t{\n\t\t\t\t\t\t\thue = hueEnd;\n\t\t\t\t\t\t}\n\t\t\t\t\t\t\n\t\t\t\t\t\tnewNode.hues[j] = hue;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\tif ( attributeName == hueName )\n\t\t\t\t{\n\t\t\t\t\tnewNode.hue = new Tween(newNode.hues[0], newNode.hues[0]);\n\t\t\t\t}\n\t\t\t}\n\t\t\tbreak;\n\t\t}\n\t}\n\t\n\treturn newNode;\n}\nfunction maxAbsoluteDepthDecrease()\n{\n\tif ( maxAbsoluteDepth > 2 )\n\t{\n\t\tmaxAbsoluteDepth--;\n\t\thead.setMaxDepths();\n\t\thandleResize();\n\t}\n}\nfunction maxAbsoluteDepthIncrease()\n{\n\tif ( maxAbsoluteDepth < head.maxDepth )\n\t{\n\t\tmaxAbsoluteDepth++;\n\t\thead.setMaxDepths();\n\t\thandleResize();\n\t}\n}\nfunction measureText(text, bold)\n{\n\tcontext.font = bold ? fontBold : fontNormal;\n\tvar dim = context.measureText(text);\n\treturn dim.width;\n}\nfunction min(a, b)\n{\n\treturn a < b ? a : b;\n}\nfunction minWidth()\n{\n\t// Min wedge width (at center) for displaying a node (or for displaying a\n\t// label if it\'s at the highest level being viewed, multiplied by 2 to make\n\t// further calculations simpler\n\t\n\treturn (fontSize * 2.3);\n}\nfunction mouseMove(e)\n{\n\tmouseX = e.pageX;\n\tmouseY = e.pageY - headerHeight;\n\tmouseXRel = (mouseX - centerX) * backingScale()\n\tmouseYRel = (mouseY - centerY) * backingScale()\n\t\n\tif ( head && ! quickLook )\n\t{\n\t\tcheckHighlight();\n\t}\n}\nfunction mouseClick(e)\n{\n\tif ( highlightedNode == focusNode && focusNode != selectedNode || selectedNode.hasParent(highlightedNode) )\n\t{\n\t\tif ( highlightedNode.hasChildren() )\n\t\t{\n\t\t\texpand(highlightedNode);\n\t\t}\n\t}\n\telse if ( progress == 1 )//( highlightedNode != selectedNode )\n\t{\n\t\tsetFocus(highlightedNode);\n//\t\tdocument.body.style.cursor=\'ew-resize\';\n\t\tdraw();\n\t\tcheckHighlight();\n\t\tvar date = new Date();\n\t\tmouseDownTime = date.getTime();\n\t\tmouseDown = true;\n\t}\n}\nfunction mouseUp(e)\n{\n\tif ( quickLook )\n\t{\n\t\tnavigateBack();\n\t\tquickLook = false;\n\t}\n\t\n\tmouseDown = false;\n}\nfunction navigateBack()\n{\n\tif ( nodeHistoryPosition > 0 )\n\t{\n\t\tnodeHistory[nodeHistoryPosition] = selectedNode;\n\t\tnodeHistoryPosition--;\n\t\t\n\t\tif ( nodeHistory[nodeHistoryPosition].collapse )\n\t\t{\n\t\t\tcollapseCheckBox.checked = collapse = false;\n\t\t}\n\t\t\n\t\tsetSelectedNode(nodeHistory[nodeHistoryPosition]);\n\t\tupdateDatasetButtons();\n\t\tupdateView();\n\t}\n}\nfunction navigateUp()\n{\n\tif ( selectedNode.getParent() )\n\t{\n\t\tselectNode(selectedNode.getParent());\n\t\tupdateView();\n\t}\n}\nfunction navigateForward()\n{\n\tif ( nodeHistoryPosition < nodeHistory.length - 1 )\n\t{\n\t\tnodeHistoryPosition++;\n\t\tvar newNode = nodeHistory[nodeHistoryPosition];\n\t\t\n\t\tif ( newNode.collapse )\n\t\t{\n\t\t\tcollapseCheckBox.checked = collapse = false;\n\t\t}\n\t\t\n\t\tif ( nodeHistoryPosition == nodeHistory.length - 1 )\n\t\t{\n\t\t\t// this will ensure the forward button is disabled\n\t\t\t\n\t\t\tnodeHistory.length = nodeHistoryPosition;\n\t\t}\n\t\t\n\t\tsetSelectedNode(newNode);\n\t\tupdateDatasetButtons();\n\t\tupdateView();\n\t}\n}\nfunction nextDataset()\n{\n\tvar newDataset = currentDataset;\n\t\n\tdo\n\t{\n\t\tif ( newDataset == datasets - 1 )\n\t\t{\n\t\t\tnewDataset = 0;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tnewDataset++;\n\t\t}\n\t}\n\twhile ( datasetDropDown.options[newDataset].disabled )\n\t\n\tselectDataset(newDataset);\n}\nfunction onDatasetChange()\n{\n\tselectDataset(datasetDropDown.selectedIndex);\n}\nfunction onKeyDown(event)\n{\n\tif\n\t(\n\t\tevent.keyCode == 37 &&\n\t\tdocument.activeElement.id != \'search\' &&\n\t\tdocument.activeElement.id != \'linkText\'\n\t)\n\t{\n\t\tnavigateBack();\n\t\tevent.preventDefault();\n\t}\n\telse if\n\t(\n\t\tevent.keyCode == 39 &&\n\t\tdocument.activeElement.id != \'search\' &&\n\t\tdocument.activeElement.id != \'linkText\'\n\t)\n\t{\n\t\tnavigateForward();\n\t\tevent.preventDefault();\n\t}\n\telse if ( event.keyCode == 38 && datasets > 1 )\n\t{\n\t\tprevDataset();\n\t\t\n\t\t//if ( document.activeElement.id == \'datasets\' )\n\t\t{\n\t\t\tevent.preventDefault();\n\t\t}\n\t}\n\telse if ( event.keyCode == 40 && datasets > 1 )\n\t{\n\t\tnextDataset();\n\t\t\n\t\t//if ( document.activeElement.id == \'datasets\' )\n\t\t{\n\t\t\tevent.preventDefault();\n\t\t}\n\t}\n\telse if ( event.keyCode == 9 && datasets > 1 )\n\t{\n\t\tselectLastDataset();\n\t\tevent.preventDefault();\n\t}\n\telse if ( event.keyCode == 83 )\n\t{\n\t\tprogress += .2;\n\t}\n\telse if ( event.keyCode == 66 )\n\t{\n\t\tprogress -= .2;\n\t}\n\telse if ( event.keyCode == 70 )\n\t{\n\t\tprogress = 1;\n\t}\n}\nfunction onKeyPress(event)\n{\n\tif ( event.keyCode == 38 && datasets > 1 )\n\t{\n//\t\tprevDataset();\n\t\t\n\t\t//if ( document.activeElement.id == \'datasets\' )\n\t\t{\n\t\t\tevent.preventDefault();\n\t\t}\n\t}\n\telse if ( event.keyCode == 40 && datasets > 1 )\n\t{\n//\t\tnextDataset();\n\t\t\n\t\t//if ( document.activeElement.id == \'datasets\' )\n\t\t{\n\t\t\tevent.preventDefault();\n\t\t}\n\t}\n}\nfunction onKeyUp(event)\n{\n\tif ( event.keyCode == 27 && document.activeElement.id == \'search\' )\n\t{\n\t\tsearch.value = \'\';\n\t\tonSearchChange();\n\t}\n\telse if ( event.keyCode == 38 && datasets > 1 )\n\t{\n//\t\tprevDataset();\n\t\t\n\t\t//if ( document.activeElement.id == \'datasets\' )\n\t\t{\n\t\t\tevent.preventDefault();\n\t\t}\n\t}\n\telse if ( event.keyCode == 40 && datasets > 1 )\n\t{\n//\t\tnextDataset();\n\t\t\n\t\t//if ( document.activeElement.id == \'datasets\' )\n\t\t{\n\t\t\tevent.preventDefault();\n\t\t}\n\t}\n}\nfunction onSearchChange()\n{\n\tnSearchResults = 0;\n\thead.search();\n\t\n\tif ( search.value == \'\' )\n\t{\n\t\tsearchResults.innerHTML = \'\';\n\t}\n\telse\n\t{\n\t\tsearchResults.innerHTML = nSearchResults + \' results\';\n\t}\n\t\n\tsetFocus(selectedNode);\n\tdraw();\n}\nfunction post(url, variable, value, postWindow)\n{\n\tvar form = document.createElement(\'form\');\n\tvar input = document.createElement(\'input\');\n\tvar inputDataset = document.createElement(\'input\');\n\t\n\tform.appendChild(input);\n\tform.appendChild(inputDataset);\n\t\n\tform.method = "POST";\n\tform.action = url;\n\t\n\tif ( postWindow == undefined )\n\t{\n\t\tform.target = \'_blank\';\n\t\tpostWindow = window;\n\t}\n\t\n\tinput.type = \'hidden\';\n\tinput.name = variable;\n\tinput.value = value;\n\t\n\tinputDataset.type = \'hidden\';\n\tinputDataset.name = \'dataset\';\n\tinputDataset.value = currentDataset;\n\t\n\tpostWindow.document.body.appendChild(form);\n\tform.submit();\n}\nfunction prevDataset()\n{\n\tvar newDataset = currentDataset;\n\t\n\tdo\n\t{\n\t\tif ( newDataset == 0 )\n\t\t{\n\t\t\tnewDataset = datasets - 1;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tnewDataset--;\n\t\t}\n\t}\n\twhile ( datasetDropDown.options[newDataset].disabled );\n\t\n\tselectDataset(newDataset);\n}\nfunction radiusDecrease()\n{\n\tif ( bufferFactor < .309 )\n\t{\n\t\tbufferFactor += .03;\n\t\tupdateViewNeeded = true;\n\t}\n}\nfunction radiusIncrease()\n{\n\tif ( bufferFactor > .041 )\n\t{\n\t\tbufferFactor -= .03;\n\t\tupdateViewNeeded = true;\n\t}\n}\nfunction resetKeyOffset()\n{\n\tcurrentKey = 1;\n\tkeyMinTextLeft = centerX + gRadius + buffer - buffer / (keys + 1) / 2 + fontSize / 2;\n\tkeyMinAngle = 0;\n}\nfunction rgbText(r, g, b)\n{\n\tvar rgbArray =\n\t[\n\t\t"rgb(",\n\t\tMath.floor(r),\n\t\t",",\n\t\tMath.floor(g),\n\t\t",",\n\t\tMath.floor(b),\n\t\t")"\n\t];\n\t\n\treturn rgbArray.join(\'\');\n}\nfunction round(number)\n{\n\tif ( number >= 1 || number <= -1 )\n\t{\n\t\treturn number.toFixed(0);\n\t}\n\telse\n\t{\n\t\treturn number.toPrecision(1);\n\t}\n}\nfunction roundedRectangle(x, y, width, height, radius)\n{\n\tif ( radius * 2 > width )\n\t{\n\t\tradius = width / 2;\n\t}\n\t\n\tif ( radius * 2 > height )\n\t{\n\t\tradius = height / 2;\n\t}\n\t\n\tcontext.beginPath();\n\tcontext.arc(x + radius, y + radius, radius, Math.PI, Math.PI * 3 / 2, false);\n\tcontext.lineTo(x + width - radius, y);\n\tcontext.arc(x + width - radius, y + radius, radius, Math.PI * 3 / 2, Math.PI * 2, false);\n\tcontext.lineTo(x + width, y + height - radius);\n\tcontext.arc(x + width - radius, y + height - radius, radius, 0, Math.PI / 2, false);\n\tcontext.lineTo(x + radius, y + height);\n\tcontext.arc(x + radius, y + height - radius, radius, Math.PI / 2, Math.PI, false);\n\tcontext.lineTo(x, y + radius);\n}\nfunction passClick(e)\n{\n\tmouseClick(e);\n}\nfunction searchResultString(results)\n{\n\tvar searchResults = this.searchResults;\n\t\n\tif ( this.isSearchResult )\n\t{\n\t\t// don\'t count ourselves\n\t\tsearchResults--;\n\t}\n\t\n\treturn \' - \' + results + (results > 1 ? \' results\' : \' result\');\n}\nfunction setCallBacks()\n{\n\tcanvas.onselectstart = function(){return false;} // prevent unwanted highlighting\n\toptions.onselectstart = function(){return false;} // prevent unwanted highlighting\n\tdocument.onmousemove = mouseMove;\n\twindow.onblur = focusLost;\n\twindow.onmouseout = focusLost;\n\tdocument.onkeyup = onKeyUp;\n\tdocument.onkeydown = onKeyDown;\n\tcanvas.onmousedown = mouseClick;\n\tdocument.onmouseup = mouseUp;\n\tkeyControl.onclick = toggleKeys;\n\tcollapseCheckBox = document.getElementById(\'collapse\');\n\tcollapseCheckBox.checked = collapse;\n\tcollapseCheckBox.onclick = handleResize;\n\tcollapseCheckBox.onmousedown = suppressEvent;\n\tmaxAbsoluteDepthText = document.getElementById(\'maxAbsoluteDepth\');\n\tmaxAbsoluteDepthButtonDecrease = document.getElementById(\'maxAbsoluteDepthDecrease\');\n\tmaxAbsoluteDepthButtonIncrease = document.getElementById(\'maxAbsoluteDepthIncrease\');\n\tmaxAbsoluteDepthButtonDecrease.onclick = maxAbsoluteDepthDecrease;\n\tmaxAbsoluteDepthButtonIncrease.onclick = maxAbsoluteDepthIncrease;\n\tmaxAbsoluteDepthButtonDecrease.onmousedown = suppressEvent;\n\tmaxAbsoluteDepthButtonIncrease.onmousedown = suppressEvent;\n\tfontSizeText = document.getElementById(\'fontSize\');\n\tfontSizeButtonDecrease = document.getElementById(\'fontSizeDecrease\');\n\tfontSizeButtonIncrease = document.getElementById(\'fontSizeIncrease\');\n\tfontSizeButtonDecrease.onclick = fontSizeDecrease;\n\tfontSizeButtonIncrease.onclick = fontSizeIncrease;\n\tfontSizeButtonDecrease.onmousedown = suppressEvent;\n\tfontSizeButtonIncrease.onmousedown = suppressEvent;\n\tradiusButtonDecrease = document.getElementById(\'radiusDecrease\');\n\tradiusButtonIncrease = document.getElementById(\'radiusIncrease\');\n\tradiusButtonDecrease.onclick = radiusDecrease;\n\tradiusButtonIncrease.onclick = radiusIncrease;\n\tradiusButtonDecrease.onmousedown = suppressEvent;\n\tradiusButtonIncrease.onmousedown = suppressEvent;\n\tmaxAbsoluteDepth = 0;\n\tbackButton = document.getElementById(\'back\');\n\tbackButton.onclick = navigateBack;\n\tbackButton.onmousedown = suppressEvent;\n\tforwardButton = document.getElementById(\'forward\');\n\tforwardButton.onclick = navigateForward;\n\tforwardButton.onmousedown = suppressEvent;\n\tsnapshotButton = document.getElementById(\'snapshot\');\n\tsnapshotButton.onclick = snapshot;\n\tsnapshotButton.onmousedown = suppressEvent;\n\tdetailsName = document.getElementById(\'detailsName\');\n\tdetailsExpand = document.getElementById(\'detailsExpand\');\n\tdetailsInfo = document.getElementById(\'detailsInfo\');\n\tsearch = document.getElementById(\'search\');\n\tsearch.onkeyup = onSearchChange;\n\tsearch.onmousedown = suppressEvent;\n\tsearchResults = document.getElementById(\'searchResults\');\n\tuseHueDiv = document.getElementById(\'useHueDiv\');\n\tlinkButton = document.getElementById(\'linkButton\');\n\tlinkButton.onclick = showLink;\n\tlinkButton.onmousedown = suppressEvent;\n\tlinkText = document.getElementById(\'linkText\');\n\tlinkText.onblur = hideLink;\n\tlinkText.onmousedown = suppressEvent;\n\thide(linkText);\n\tvar helpButton = document.getElementById(\'help\');\n\thelpButton.onmousedown = suppressEvent;\n\tvar searchClear = document.getElementById(\'searchClear\');\n\tsearchClear.onmousedown = suppressEvent;\n\tif ( datasets > 1 )\n\t{\n\t\tdatasetDropDown.onmousedown = suppressEvent;\n\t\tvar prevDatasetButton = document.getElementById(\'prevDataset\');\n\t\tprevDatasetButton.onmousedown = suppressEvent;\n\t\tvar nextDatasetButton = document.getElementById(\'nextDataset\');\n\t\tnextDatasetButton.onmousedown = suppressEvent;\n\t\tvar lastDatasetButton = document.getElementById(\'lastDataset\');\n\t\tlastDatasetButton.onmousedown = suppressEvent;\n\t}\n\t\n\timage = document.getElementById(\'hiddenImage\');\n\t\n\tif ( image.complete )\n\t{\n\t\thiddenPattern = context.createPattern(image, \'repeat\');\n\t}\n\telse\n\t{\n\t\timage.onload = function()\n\t\t{\n\t\t\thiddenPattern = context.createPattern(image, \'repeat\');\n\t\t}\n\t}\n\t\n\tvar loadingImageElement = document.getElementById(\'loadingImage\');\n\t\n\tif ( loadingImageElement )\n\t{\n\t\tloadingImage = loadingImageElement.src;\n\t}\n}\nfunction selectDataset(newDataset)\n{\n\tlastDataset = currentDataset;\n\tcurrentDataset = newDataset\n\tif ( datasets > 1 )\n\t{\n\t\tdatasetDropDown.selectedIndex = currentDataset;\n\t\tupdateDatasetButtons();\n\t\tdatasetAlpha.start = 1.5;\n\t\tdatasetChanged = true;\n\t}\n\thead.setMagnitudes(0);\n\thead.setDepth(1, 1);\n\thead.setMaxDepths();\n\thandleResize();\n}\nfunction selectLastDataset()\n{\n\tselectDataset(lastDataset);\n\thandleResize();\n}\nfunction selectNode(newNode)\n{\n\tif ( selectedNode != newNode )\n\t{\n\t\t// truncate history at current location to create a new branch\n\t\t//\n\t\tnodeHistory.length = nodeHistoryPosition;\n\t\t\n\t\tif ( selectedNode != 0 )\n\t\t{\n\t\t\tnodeHistory.push(selectedNode);\n\t\t\tnodeHistoryPosition++;\n\t\t}\n\t\t\n\t\tsetSelectedNode(newNode);\n\t\t//updateView();\n\t}\n\t\n\tupdateDatasetButtons();\n}\nfunction setFocus(node)\n{\n\tif ( node == focusNode )\n\t{\n//\t\treturn;\n\t}\n\t\n\tfocusNode = node;\n\t\n\tif ( node.href )\n\t{\n\t\tdetailsName.innerHTML =\n\t\t\t\'<a target="_blank" href="\' + node.href + \'">\' + node.name + \'</a>\';\n\t}\n\telse\n\t{\n\t\tdetailsName.innerHTML = node.name;\n\t}\n\t\n\tvar table = \'<table>\';\n\t\n\ttable += \'<tr><td></td></tr>\';\n\t\n\tfor ( var i = 0; i < node.attributes.length; i++ )\n\t{\n\t\tif ( attributes[i].displayName && node.attributes[i] != undefined )\n\t\t{\n\t\t\tvar index = node.attributes[i].length == 1 && attributes[i].mono ? 0 : currentDataset;\n\t\t\t\n\t\t\tif ( typeof node.attributes[i][currentDataset] == \'number\' || node.attributes[i][index] != undefined && node.attributes[i][currentDataset] != \'\' )\n\t\t\t{\n\t\t\t\tvar value = node.attributes[i][index];\n\t\t\t\t\n\t\t\t\tif ( attributes[i].listNode != undefined )\n\t\t\t\t{\n\t\t\t\t\tvalue =\n\t\t\t\t\t\t\'<a href="" onclick="showList(\' +\n\t\t\t\t\t\tattributeIndex(attributes[i].listNode) + \',\' + i +\n\t\t\t\t\t\t\',false);return false;" title="Show list">\' +\n\t\t\t\t\t\tvalue + \'</a>\';\n\t\t\t\t}\n\t\t\t\telse if ( attributes[i].listAll != undefined )\n\t\t\t\t{\n\t\t\t\t\tvalue =\n\t\t\t\t\t\t\'<a href="" onclick="showList(\' +\n\t\t\t\t\t\tattributeIndex(attributes[i].listAll) + \',\' + i +\n\t\t\t\t\t\t\',true);return false;" title="Show list">\' +\n\t\t\t\t\t\tvalue + \'</a>\';\n\t\t\t\t}\n\t\t\t\telse if ( attributes[i].dataNode != undefined && dataEnabled )\n\t\t\t\t{\n\t\t\t\t\tvalue =\n\t\t\t\t\t\t\'<a href="" onclick="showData(\' +\n\t\t\t\t\t\tattributeIndex(attributes[i].dataNode) + \',\' + i +\n\t\t\t\t\t\t\',false);return false;" title="Show data">\' +\n\t\t\t\t\t\tvalue + \'</a>\';\n\t\t\t\t}\n\t\t\t\telse if ( attributes[i].dataAll != undefined && dataEnabled )\n\t\t\t\t{\n\t\t\t\t\tvalue =\n\t\t\t\t\t\t\'<a href="" onclick="showData(\' +\n\t\t\t\t\t\tattributeIndex(attributes[i].dataAll) + \',\' + i +\n\t\t\t\t\t\t\',true);return false;" title="Show data">\' +\n\t\t\t\t\t\tvalue + \'</a>\';\n\t\t\t\t}\n\t\t\t\t\n\t\t\t\ttable +=\n\t\t\t\t\t\'<tr><td><strong>\' + attributes[i].displayName + \':</strong></td><td>\' +\n\t\t\t\t\tvalue + \'</td></tr>\';\n\t\t\t}\n\t\t}\n\t}\n\t\n\ttable += \'</table>\';\n\tdetailsInfo.innerHTML = table;\n\t\n\tdetailsExpand.disabled = !focusNode.hasChildren() || focusNode == selectedNode;\n}\nfunction setSelectedNode(newNode)\n{\n\tif ( selectedNode && selectedNode.hasParent(newNode) )\n\t{\n\t\tzoomOut = true;\n\t}\n\telse\n\t{\n\t\tzoomOut = false;\n\t}\n\t\n\tselectedNodeLast = selectedNode;\n\tselectedNode = newNode;\n\t\n\t//if ( focusNode != selectedNode )\n\t{\n\t\tsetFocus(selectedNode);\n\t}\n}\nfunction waitForData(dataWindow, target, title, time, postUrl, postVar)\n{\n\tif ( nodeData.length == target )\n\t{\n\t\tif ( postUrl != undefined )\n\t\t{\n\t\t\tfor ( var i = 0; i < nodeData.length; i++ )\n\t\t\t{\n\t\t\t\tnodeData[i] = nodeData[i].replace(/\\n/g, \',\');\n\t\t\t}\n\t\t\t\n\t\t\tvar postString = nodeData.join(\'\');\n\t\t\tpostString = postString.slice(0, -1);\n\t\t\t\n\t\t\tdataWindow.document.body.removeChild(dataWindow.document.getElementById(\'loading\'));\n\t\t\tdocument.body.removeChild(document.getElementById(\'data\'));\n\t\t\t\n\t\t\tpost(postUrl, postVar, postString, dataWindow);\n\t\t}\n\t\telse\n\t\t{\n\t\t\t//dataWindow.document.body.removeChild(dataWindow.document.getElementById(\'loading\'));\n\t\t\t//document.body.removeChild(document.getElementById(\'data\'));\n\t\t\t\n\t\t\tdataWindow.document.open();\n\t\t\tdataWindow.document.write(\'<pre>\' + nodeData.join(\'\') + \'</pre>\');\n\t\t\tdataWindow.document.close();\n\t\t}\n\t\t\n\t\tdataWindow.document.title = title; // replace after document.write()\n\t}\n\telse\n\t{\n\t\tvar date = new Date();\n\t\t\n\t\tif ( date.getTime() - time > 10000 )\n\t\t{\n\t\t\tdataWindow.document.body.removeChild(dataWindow.document.getElementById(\'loading\'));\n\t\t\tdocument.body.removeChild(document.getElementById(\'data\'));\n\t\t\tdataWindow.document.body.innerHTML =\n\t\t\t\t\'Timed out loading supplemental files for:<br/>\' + document.location;\n\t\t}\n\t\telse\n\t\t{\n\t\t\tsetTimeout(function() {waitForData(dataWindow, target, title, time, postUrl, postVar);}, 100);\n\t\t}\n\t}\n}\nfunction data(newData)\n{\n\tnodeData.push(newData);\n}\nfunction enableData()\n{\n\tdataEnabled = true;\n}\nfunction showData(indexData, indexAttribute, summary)\n{\n\tvar dataWindow = window.open(\'\', \'_blank\');\n\tvar title = \'Krona - \' + attributes[indexAttribute].displayName + \' - \' + focusNode.name;\n\tdataWindow.document.title = title;\n\t\n\tnodeData = new Array();\n\t\n\tif ( dataWindow && dataWindow.document && dataWindow.document.body != null )\n\t{\n\t\t//var loadImage = document.createElement(\'img\');\n\t\t//loadImage.src = "file://localhost/Users/ondovb/Krona/KronaTools/img/loading.gif";\n\t\t//loadImage.id = "loading";\n\t\t//loadImage.alt = "Loading...";\n\t\t//dataWindow.document.body.appendChild(loadImage);\n\t\tdataWindow.document.body.innerHTML =\n\t\t\t\'<img id="loading" src="\' + loadingImage + \'" alt="Loading..."></img>\';\n\t}\n\t\n\tvar scripts = document.createElement(\'div\');\n\tscripts.id = \'data\';\n\tdocument.body.appendChild(scripts);\n\t\n\tvar files = focusNode.getData(indexData, summary);\n\t\n\tvar date = new Date();\n\tvar time = date.getTime();\n\t\n\tfor ( var i = 0; i < files.length; i++ )\n\t{\n\t\tvar script = document.createElement(\'script\');\n\t\tscript.src = files[i] + \'?\' + time;\n\t\tscripts.appendChild(script);\n\t}\n\t\n\twaitForData(dataWindow, files.length, title, time, attributes[indexAttribute].postUrl, attributes[indexAttribute].postVar);\n\t\n\treturn false;\n}\nfunction showList(indexList, indexAttribute, summary)\n{\n\tvar list = focusNode.getList(indexList, summary);\n\t\n\tif ( attributes[indexAttribute].postUrl != undefined )\n\t{\n\t\tpost(attributes[indexAttribute].postUrl, attributes[indexAttribute].postVar, list.join(\',\'));\n\t}\n\telse\n\t{\n\t\tvar dataWindow = window.open(\'\', \'_blank\');\n\t\t\n\t\tif ( true || navigator.appName == \'Microsoft Internet Explorer\' ) // :(\n\t\t{\n\t\t\tdataWindow.document.open();\n\t\t\tdataWindow.document.write(\'<pre>\' + list.join(\'\\n\') + \'</pre>\');\n\t\t\tdataWindow.document.close();\n\t\t}\n\t\telse\n\t\t{\n\t\t\tvar pre = document.createElement(\'pre\');\n\t\t\tdataWindow.document.body.appendChild(pre);\n\t\t\tpre.innerHTML = list;\n\t\t}\n\t\t\n\t\tdataWindow.document.title = \'Krona - \' + attributes[indexAttribute].displayName + \' - \' + focusNode.name;\n\t}\n}\nfunction snapshot()\n{\n\tsvg = svgHeader();\n\t\n\tresetKeyOffset();\n\t\n\tsnapshotMode = true;\n\t\n\tselectedNode.draw(false, true);\n\tselectedNode.draw(true, true);\n\t\n\tif ( focusNode != 0 && focusNode != selectedNode )\n\t{\n\t\tcontext.globalAlpha = 1;\n\t\tfocusNode.drawHighlight(true);\n\t}\n\t\n\tif ( hueDisplayName && useHue() )\n\t{\n\t\tdrawLegendSVG();\n\t}\n\t\n\tsnapshotMode = false;\n\t\n\tsvg += svgFooter();\n\t\n\tsnapshotWindow = window.open\n\t(\n\t\t\'data:image/svg+xml;charset=utf-8,\' + encodeURIComponent(svg),\n\t\t\'_blank\'\n\t);\n/*\tvar data = window.open(\'data:text/plain;charset=utf-8,hello\', \'_blank\');\n\tvar data = window.open(\'\', \'_blank\');\n\tdata.document.open(\'text/plain\');\n\tdata.document.write(\'hello\');\n\tdata.document.close();\n\tvar button = document.createElement(\'input\');\n\tbutton.type = \'button\';\n\tbutton.value = \'save\';\n\tbutton.onclick = save;\n\tdata.document.body.appendChild(button);\n//\tsnapshotWindow.document.write(svg);\n//\tsnapshotWindow.document.close();\n*/\t\n}\nfunction save()\n{\n\talert(document.body.innerHTML);\n}\nfunction spacer()\n{\n\tif ( snapshotMode )\n\t{\n\t\treturn \'&#160;&#160;&#160;\';\n\t}\n\telse\n\t{\n\t\treturn \'   \';\n\t}\n}\nfunction suppressEvent(e)\n{\n\te.cancelBubble = true;\n\tif (e.stopPropagation) e.stopPropagation();\n}\nfunction svgFooter()\n{\n\treturn \'</svg>\';\n}\nfunction svgHeader()\n{\n\tvar patternWidth = fontSize * .6;//radius / 50;\n\t\n\treturn \'\\\n<?xml version="1.0" standalone="no"?>\\\n<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN" \\\n\t"http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">\\\n<svg width="\' + imageWidth + \'" height="\' + imageHeight + \'" version="1.1"\\\n\txmlns="http://www.w3.org/2000/svg">\\\n<title>Krona (snapshot) - \' +\n(datasets > 1 ? datasetNames[currentDataset] + \' - \' : \'\') + selectedNode.name +\n\'</title>\\\n<defs>\\\n\t<style type="text/css">\\\n\ttext {font-size: \' + fontSize + \'px; font-family: \' + fontFamily + \'; dominant-baseline:central}\\\n\tpath {stroke-width:\' + thinLineWidth * fontSize / 12 + \';}\\\n\tpath.wedge {stroke:none}\\\n\tpath.line {fill:none;stroke:black;}\\\n\tline {stroke:black;stroke-width:\' + thinLineWidth * fontSize / 12 + \';}\\\n\tline.tick {stroke-width:\' + thinLineWidth * fontSize / 6 + \';}\\\n\tline.pattern {stroke-width:\' + thinLineWidth * fontSize / 18 + \';}\\\n\tcircle {fill:none;stroke:black;stroke-width:\' + thinLineWidth * fontSize / 12 + \';}\\\n\trect {stroke:black;stroke-width:\' + thinLineWidth * fontSize / 12 + \';}\\\n\t.highlight {stroke:black;stroke-width:\'+ highlightLineWidth * fontSize / 12 + \';}\\\n\t.searchHighlight {fill:rgb(255, 255, 100);stroke:none;}\\\n\t</style>\\\n<pattern id="hiddenPattern" patternUnits="userSpaceOnUse" \\\nx="0" y="0" width="\' + patternWidth + \'" height="\' + patternWidth + \'">\\\n<line class="pattern" x1="0" y1="0" x2="\' + patternWidth / 2 + \'" y2="\' + patternWidth / 2 + \'"/>\\\n<line class="pattern" x1="\' + patternWidth / 2 + \'" y1="\' + patternWidth +\n\'" x2="\' + patternWidth + \'" y2="\' + patternWidth / 2 + \'"/>\\\n</pattern>\\\n</defs>\\\n\';\n}\nfunction svgText(text, x, y, anchor, bold, color)\n{\n\tif ( typeof(anchor) == \'undefined\' )\n\t{\n\t\tanchor = \'start\';\n\t}\n\t\n\tif ( color == undefined )\n\t{\n\t\tcolor = \'black\';\n\t}\n\t\n\treturn \'<text x="\' + x + \'" y="\' + y +\n\t\t\'" style="font-color:\' + color + \';font-weight:\' + (bold ? \'bold\' : \'normal\') +\n\t\t\'" text-anchor="\' + anchor + \'">\' + text + \'</text>\';\n}\nfunction toggleKeys()\n{\n\tif ( showKeys )\n\t{\n\t\tkeyControl.value = \'...\';\n\t\tshowKeys = false;\n\t}\n\telse\n\t{\n\t\tkeyControl.value = \'x\';\n\t\tshowKeys = true;\n\t}\n\t\n\tupdateKeyControl();\n\t\n\tif ( progress == 1 )\n\t{\n\t\tdraw();\n\t}\n}\nfunction update()\n{\n\tif ( ! head )\n\t{\n\t\treturn;\n\t}\n\t\n\tif ( mouseDown && focusNode != selectedNode )\n\t{\n\t\tvar date = new Date();\n\t\t\n\t\tif ( date.getTime() - mouseDownTime > quickLookHoldLength )\n\t\t{\n\t\t\tif ( focusNode.hasChildren() )\n\t\t\t{\n\t\t\t\texpand(focusNode);\n\t\t\t\tquickLook = true;\n\t\t\t}\n\t\t}\n\t}\n\t\n\tif ( updateViewNeeded )\n\t{\n\t\tresize();\n\t\tmouseX = -1;\n\t\tmouseY = -1;\n\t\t\n\t\tcollapse = collapseCheckBox.checked;\n\t\tcompress = true;//compressCheckBox.checked;\n\t\tshorten = true;//shortenCheckBox.checked;\n\t\t\n\t\tcheckSelectedCollapse();\n\t\tupdateMaxAbsoluteDepth();\n\t\t\n\t\tif ( focusNode.getCollapse() || focusNode.depth > maxAbsoluteDepth )\n\t\t{\n\t\t\tsetFocus(selectedNode);\n\t\t}\n\t\telse\n\t\t{\n\t\t\tsetFocus(focusNode);\n\t\t}\n\t\t\n\t\tupdateView();\n\t\t\n\t\tupdateViewNeeded = false;\n\t}\n\t\n\tvar date = new Date();\n\tprogress = (date.getTime() - tweenStartTime) / tweenLength;\n//\tprogress += .01;\n\t\n\tif ( progress >= 1 )\n\t{\n\t\tprogress = 1;\n\t}\n\t\n\tif ( progress != progressLast )\n\t{\n\t\ttweenFactor =// progress;\n\t\t\t(1 / (1 + Math.exp(-tweenCurvature * (progress - .5))) - .5) /\n\t\t\t(tweenMax - .5) / 2 + .5;\n\t\t\n\t\tif ( progress == 1 )\n\t\t{\n\t\t\tsnapshotButton.disabled = false;\n\t\t\tzoomOut = false;\n\t\t\t\n\t\t\t//updateKeyControl();\n\t\t\t\n\t\t\tif ( ! quickLook )\n\t\t\t{\n\t\t\t\t//checkHighlight();\n\t\t\t}\n\t\t\t\n\t\t\t\n\t\t\tif ( fpsDisplay )\n\t\t\t{\n\t\t\t\tfpsDisplay.innerHTML = \'fps: \' + Math.round(tweenFrames * 1000 / tweenLength);\n\t\t\t}\n\t\t}\n\t\t\n\t\tdraw();\n\t}\n\t\n\tprogressLast = progress;\n}\nfunction updateDatasetButtons()\n{\n\tif ( datasets == 1 )\n\t{\n\t\treturn;\n\t}\n\t\n\tvar node = selectedNode ? selectedNode : head;\n\t\n\tdatasetButtonLast.disabled =\n\t\tnode.attributes[magnitudeIndex][lastDataset] == 0;\n\t\n\tdatasetButtonPrev.disabled = true;\n\tdatasetButtonNext.disabled = true;\n\t\n\tfor ( var i = 0; i < datasets; i++ )\n\t{\n\t\tvar disable = node.attributes[magnitudeIndex][i] == 0;\n\t\t\n\t\tdatasetDropDown.options[i].disabled = disable;\n\t\t\n\t\tif ( ! disable )\n\t\t{\n\t\t\tif ( i != currentDataset )\n\t\t\t{\n\t\t\t\tdatasetButtonPrev.disabled = false;\n\t\t\t\tdatasetButtonNext.disabled = false;\n\t\t\t}\n\t\t}\n\t}\n}\nfunction updateDatasetWidths()\n{\n\tif ( datasets > 1 )\n\t{\n\t\tfor ( var i = 0; i < datasets; i++ )\n\t\t{\n\t\t\tcontext.font = fontBold;\n\t\t\tvar dim = context.measureText(datasetNames[i]);\n\t\t\tdatasetWidths[i] = dim.width;\n\t\t}\n\t}\n}\nfunction updateKeyControl()\n{\n\tif ( keys == 0 )//|| progress != 1 )\n\t{\n\t\tkeyControl.style.visibility = \'hidden\';\n\t}\n\telse\n\t{\n\t\tkeyControl.style.visibility = \'visible\';\n\t\tkeyControl.style.right = margin + \'px\';\n\t\t\n\t\tif ( showKeys )\n\t\t{\n\t\t\tkeyControl.style.top =\n\t\t\t\timageHeight -\n\t\t\t\t(\n\t\t\t\t\tkeys * (keySize + keyBuffer) -\n\t\t\t\t\tkeyBuffer +\n\t\t\t\t\tmargin +\n\t\t\t\t\tkeyControl.clientHeight * 1.5\n\t\t\t\t) + \'px\';\n\t\t}\n\t\telse\n\t\t{\n\t\t\tkeyControl.style.top =\n\t\t\t\t(imageHeight - margin - keyControl.clientHeight) + \'px\';\n\t\t}\n\t}\n}\nfunction updateView()\n{\n\tif ( selectedNode.depth > maxAbsoluteDepth - 1 )\n\t{\n\t\tmaxAbsoluteDepth = selectedNode.depth + 1;\n\t}\n\t\n\thighlightedNode = selectedNode;\n\t\n\tangleFactor = 2 * Math.PI / (selectedNode.magnitude);\n\t\n\tmaxPossibleDepth = Math.floor(gRadius / (fontSize * minRingWidthFactor));\n\t\n\tif ( maxPossibleDepth < 4 )\n\t{\n\t\tmaxPossibleDepth = 4;\n\t}\n\t\n\tvar minRadiusInner = fontSize * 8 / gRadius;\n\tvar minRadiusFirst = fontSize * 6 / gRadius;\n\tvar minRadiusOuter = fontSize * 5 / gRadius;\n\t\n\tif ( .25 < minRadiusInner )\n\t{\n\t\tminRadiusInner = .25;\n\t}\n\t\n\tif ( .15 < minRadiusFirst )\n\t{\n\t\tminRadiusFirst = .15;\n\t}\n\t\n\tif ( .15 < minRadiusOuter )\n\t{\n\t\tminRadiusOuter = .15;\n\t}\n\t\n\t// visibility of nodes depends on the depth they are displayed at,\n\t// so we need to set the max depth assuming they can all be displayed\n\t// and iterate it down based on the deepest child node we can display\n\t//\n\tvar maxDepth;\n\tvar newMaxDepth = selectedNode.getMaxDepth() - selectedNode.getDepth() + 1;\n\t//\n\tdo\n\t{\n\t\tmaxDepth = newMaxDepth;\n\t\t\n\t\tif ( ! compress && maxDepth > maxPossibleDepth )\n\t\t{\n\t\t\tmaxDepth = maxPossibleDepth;\n\t\t}\n\t\t\n\t\tif ( compress )\n\t\t{\n\t\t\tcompressedRadii = new Array(maxDepth);\n\t\t\t\n\t\t\tcompressedRadii[0] = minRadiusInner;\n\t\t\t\n\t\t\tvar offset = 0;\n\t\t\t\n\t\t\twhile\n\t\t\t(\n\t\t\t\tlerp\n\t\t\t\t(\n\t\t\t\t\tMath.atan(offset + 2),\n\t\t\t\t\tMath.atan(offset + 1),\n\t\t\t\t\tMath.atan(maxDepth + offset - 1),\n\t\t\t\t\tminRadiusInner,\n\t\t\t\t\t1 - minRadiusOuter\n\t\t\t\t) - minRadiusInner > minRadiusFirst &&\n\t\t\t\toffset < 10\n\t\t\t)\n\t\t\t{\n\t\t\t\toffset++;\n\t\t\t}\n\t\t\t\n\t\t\toffset--;\n\t\t\t\n\t\t\tfor ( var i = 1; i < maxDepth; i++ )\n\t\t\t{\n\t\t\t\tcompressedRadii[i] = lerp\n\t\t\t\t(\n\t\t\t\t\tMath.atan(i + offset),\n\t\t\t\t\tMath.atan(offset),\n\t\t\t\t\tMath.atan(maxDepth + offset - 1),\n\t\t\t\t\tminRadiusInner,\n\t\t\t\t\t1 - minRadiusOuter\n\t\t\t\t)\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tnodeRadius = 1 / maxDepth;\n\t\t}\n\t\t\n\t\tnewMaxDepth = selectedNode.maxVisibleDepth(maxDepth);\n\t\t\n\t\tif ( compress )\n\t\t{\n\t\t\tif ( newMaxDepth <= maxPossibleDepth )\n\t\t\t{\n//\t\t\t\tcompress\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tif ( newMaxDepth > maxPossibleDepth )\n\t\t\t{\n\t\t\t\tnewMaxDepth = maxPossibleDepth;\n\t\t\t}\n\t\t}\n\t}\n\twhile ( newMaxDepth < maxDepth );\n\t\n\tmaxDisplayDepth = maxDepth;\n\t\n\tlightnessFactor = (lightnessMax - lightnessBase) / (maxDepth > 8 ? 8 : maxDepth);\n\tkeys = 0;\n\t\n\tnLabelOffsets = new Array(maxDisplayDepth - 1);\n\tlabelOffsets = new Array(maxDisplayDepth - 1);\n\tlabelLastNodes = new Array(maxDisplayDepth - 1);\n\tlabelFirstNodes = new Array(maxDisplayDepth - 1);\n\t\n\tfor ( var i = 0; i < maxDisplayDepth - 1; i++ )\n\t{\n\t\tif ( compress )\n\t\t{\n\t\t\tif ( i == maxDisplayDepth - 1 )\n\t\t\t{\n\t\t\t\tnLabelOffsets[i] = 0;\n\t\t\t}\n\t\t\telse\n\t\t\t{\n\t\t\t\tvar width =\n\t\t\t\t\t(compressedRadii[i + 1] - compressedRadii[i]) *\n\t\t\t\t\tgRadius;\n\t\t\t\t\n\t\t\t\tnLabelOffsets[i] = Math.floor(width / fontSize / 1.2);\n\t\t\t\t\n\t\t\t\tif ( nLabelOffsets[i] > 2 )\n\t\t\t\t{\n\t\t\t\t\tnLabelOffsets[i] = min\n\t\t\t\t\t(\n\t\t\t\t\t\tMath.floor(width / fontSize / 1.75),\n\t\t\t\t\t\t5\n\t\t\t\t\t);\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\telse\n\t\t{\n\t\t\tnLabelOffsets[i] = Math.max\n\t\t\t(\n\t\t\t\tMath.floor(Math.sqrt((nodeRadius * gRadius / fontSize)) * 1.5),\n\t\t\t\t3\n\t\t\t);\n\t\t}\n\t\t\n\t\tlabelOffsets[i] = Math.floor((nLabelOffsets[i] - 1) / 2);\n\t\tlabelLastNodes[i] = new Array(nLabelOffsets[i] + 1);\n\t\tlabelFirstNodes[i] = new Array(nLabelOffsets[i] + 1);\n\t\t\n\t\tfor ( var j = 0; j <= nLabelOffsets[i]; j++ )\n\t\t{\n\t\t\t// these arrays will allow nodes with neighboring labels to link to\n\t\t\t// each other to determine max label length\n\t\t\t\n\t\t\tlabelLastNodes[i][j] = 0;\n\t\t\tlabelFirstNodes[i][j] = 0;\n\t\t}\n\t}\n\t\n\tfontSizeText.innerHTML = fontSize;\n\tfontNormal = fontSize + \'px \' + fontFamily;\n\tcontext.font = fontNormal;\n\tfontBold = \'bold \' + fontSize + \'px \' + fontFamily;\n\ttickLength = fontSize * .7;\n\t\n\thead.setTargets(0);\n\t\n\tkeySize = ((imageHeight - margin * 3) * 1 / 2) / keys * 3 / 4;\n\t\n\tif ( keySize > fontSize * maxKeySizeFactor )\n\t{\n\t\tkeySize = fontSize * maxKeySizeFactor;\n\t}\n\t\n\tkeyBuffer = keySize / 3;\n\t\n\tfontSizeLast = fontSize;\n\t\n\tif ( datasetChanged )\n\t{\n\t\tdatasetChanged = false;\n\t}\n\telse\n\t{\n\t\tdatasetAlpha.start = 0;\n\t}\n\t\n\tvar date = new Date();\n\ttweenStartTime = date.getTime();\n\tprogress = 0;\n\ttweenFrames = 0;\n\t\n\tupdateKeyControl();\n\tupdateDatasetWidths();\n\t\n\tdocument.title = \'Krona - \' + selectedNode.name;\n\tupdateNavigationButtons();\n\tsnapshotButton.disabled = true;\n\t\n\tmaxAbsoluteDepthText.innerHTML = maxAbsoluteDepth - 1;\n\t\n\tmaxAbsoluteDepthButtonDecrease.disabled = (maxAbsoluteDepth == 2);\n\tmaxAbsoluteDepthButtonIncrease.disabled = (maxAbsoluteDepth == head.maxDepth);\n\t\n\tif ( collapse != collapseLast && search.value != \'\' )\n\t{\n\t\tonSearchChange();\n\t\tcollapseLast = collapse;\n\t}\n}\nfunction updateMaxAbsoluteDepth()\n{\n\twhile ( maxAbsoluteDepth > 1 && selectedNode.depth > maxAbsoluteDepth - 1 )\n\t{\n\t\tselectedNode = selectedNode.getParent();\n\t}\n}\nfunction updateNavigationButtons()\n{\n\tbackButton.disabled = (nodeHistoryPosition == 0);\n//\tupButton.disabled = (selectedNode.getParent() == 0);\n\tforwardButton.disabled = (nodeHistoryPosition == nodeHistory.length);\n}\nfunction useHue()\n{\n\treturn useHueCheckBox && useHueCheckBox.checked;\n}\n/*\nfunction zoomOut()\n{\n\treturn (\n\t\tselectedNodeLast != 0 &&\n\t\tselectedNodeLast.getDepth() < selectedNode.getDepth());\n}\n*/'
_KRONA_IMAGE_SHORTCUT = 'data:image/x-icon;base64,AAABAAMAEBAAAAEAIABoBAAANgAAABgYAAABACAAiAkAAJ4EAAAgIAAAAQAgAKgQAAAmDgAAKAAAABAAAAAgAAAAAQAgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAP///wCAgIAC////AP///wC0tKJHlZWSqI6OmuRxcXn9koaK9J2Ym8uNm5V73PPoFv///wAAAAAD////AP///wCAgIAC////AP//7xCHh4OodHSi/2xqw/9nZ9H/XVuU/9eFqv/njb7/yYiq/5GAiOifqKJV////AAAAAAL///8A////AO3t2w56eoHPdHLH/3Vz3vx4dtz7dXTb/mVimv/XhKn/8ZPG/fSVx/vvlcT/kG+E/32Himb///8AAAAAA////wB0dHageHbO/3585fl5d9r/dnTX/3V02/5mYpz80X2i/eaIuv/ukcD/wXak/I6Maf+OkXD9iIKjL////wCEhFo+c3Kv/4F+6P18etz/e3nd/3l33P57eeT/aWag/9F5oP/sib3+tmuZ/3uEXf/P1YX7zdKG/3x9dbX///8Ac3J9nYKA4f+CgOb7fnzf/3174P58et//bm2v8Vtgc7SdbIHNqWmM/4aLZP/L0YT/0dWH/tfdiP+go3P6YlyWJ3BvmduGg+n/g4Hk/oF+4/6AfuX/bm2k7pCUUUX///8Ajv/jCVBmXqPBxoH/1NmI/c7Uhf/b4Yn9wMWB/2BdcGBraqX6iYbs/4SB5P+Eguj7hYPm/2Vlcov///8AACRtB////wAAAIAWrLB29uHojv/X3on+3eSK+9PYh/9XWVN7bWum+ouI7/+Gg+f/hoTq+4iF6f9lZXKL////ACQAbQf///8AAABoFoF+Xva0snL/y818/uDmiPvZ34n/WVlTe3Jxm9uMiu//iofr/oiG6f6Jhu3/cnGo7oyQQ0X///8A/znjCXF8dKNotpL/Za2O/WuVe/+FkGv9mJ5w/2pqaGBzc3+djYrr/42K8fuKh+v/i4js/oqH6/93dbfxYmJztGaTec1yxJr/gOGu/4nptf+K6Lf+i+q8/2eXhfqDNG8ngIBKPnt6uP+Rjvf9jInr/4yJ7f+Miu3+jor0/25xoP920pz/gOGw/oHerv+H4rL/kPC9+5Ttvf98ioC1////AP///wB0dHSgioff/5SR+/mOi+7/jYvt/4+L8P5scJ78etKe/YXis/+H47P/kO+8/JLxv/92m4f9nXKNL////wD///8A7e3bDnt7hM+Ihdv/ko/4/JSS+PuUkPb+bnKg/4Lapv+Q7739kfO/+43puf9/qJH/mYeUZv///wAAAAADgICAAv///wD//+8QhoaBqIB/sf+Iht7/jYjw/2tvnf+D3Kf/ieS1/4bIpP+AkYfoqJOfVf///wAAAAAC////AP///wCAgIAC////AP///wCwsJdHlZWQqJORn+RzdHv9hZOK9Jecmcuej5l7/+jzFv///wAAAAAD////AP///wD4PwAA4A8AAMAHAACAAwAAgAEAAAABAAADgQAAA8EAAAPBAAADgQAAAAEAAIABAACAAwAAwAcAAOAPAAD4PwAAKAAAABgAAAAwAAAAAQAgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAP///wD///8A////AP///wEAAAAB////AP///wi6urdDubmxj6ennsuQkIzvc3Nz/YSIh/WWoZzWrrezoLe6t1X///8T////AP///wCAgIAC////AP///wD///8A////AP///wD///8A////Af///wD///8Ax8fDQJeYkLV4eH39WlmM/19dpP9dX6b/WFRr/611j/+/fJ//om2I/4N3ff+Jko7Ntri2Xv///wD///8AgICAAv///wD///8A////AP///wD///8B////AP///wOfn5t9c3N29GRjn/9oZ8P/c3LU/XVy3P9rbdH/YVt8/9eIrv/5mcz/75bE/t+Mt/67epz/fnN5/4uTkKP///8W////AAAAAAL///8A////AP///wH///8A////A5iYkZJoaH3/bWvB/3h23Px4dt3+dHPX/3Rx1v9sbcv/YFp6/82ApP/rj8D/65DA//KVxv/1mcn83424/5t1iP98gX7A////F////wCAgIAC////AAAAAAL///8AkJCHdWZlf/9zcs7/fHrj/Hh32/92ddn/dXTX/3Zz2P9vcM//YFt7/8t9of/pi73/54u7/+eNvf/vk8P/7ZbE/pxsh/xLVU7/iomJqP///wD///8A////Af///wCfn5Y4ZmV07XZ0zv9+fOX8enjc/3p43P94dtv/dnXZ/3d12f9xcdD/YFp6/8h5nf/mh7r/44e3/+qMvf/ljrv/jV59/3F+V/68wH3/gYNp/4uIlWf///8AAAAAAv///wJ3d22pc3G6/4KA5/t9e9//fHre/3t53f96eNz/eHbZ/3l23P50dNb9Ylx+/ch2m/3jg7b+44W2/+CItv+NXXz/bnpW/8XIgv/Y3Yv8ur57/3Fxbtni4uIa////AHl5XTdoZ4b4f3zb/4F+5P5+fN//fXvf/3x63v96edz/ennf/nt54P9xccb/Xlxy/7Rvj//gf7P/3IKw/olZeP9te1b/xsqD/9DVh//P1IX/09iG/pWYcf9nZ3Nt////AISEb35ycLb/hYPr/YB+4f9/feH/fnzh/3173/99e+L+enjX/2dmmfhlZXiwYGBiknJkaqGVZ37pg11x/3eBXP7GyoP/0NWH/87Uhf/R1ob/2eCK+8HFff9tbXWv////BG9vdb16eMn/hYPq/4F/4/+BfuP/gH7i/4B94v59e93/Y2KO+ISEbX7y8uQT////AP///weAjo5aT1tP5L7Cf//T2Ij+ztOF/9HWhv/S14b/2d+K/s3Sgv+BgnLcg4OSI11deeaBf9r/hYPo/4OB5P+DgeX/gX/i/4OB6v10c77/cXFjmv///wL///8A////AP///wD///8AeHN9apuecP/U2Yf/0teG/tPZhv/V24f/2uCJ/tTahv+KjGj6AABINV9egfuGg+P/hoTp/4WC5v+EgeX/goDk/4WD6/1wbq//WFgAQ////wAzMzMFAAAAAVVVVQP///8AAABjEoiLZ/Df5Y7/2d+J/dfch//W3If/2d+H/9vhif6ZnW3/AAA5P19egvuHheX/iIXq/4eE6P+Gg+f/hILl/4eE7P1xb7D/WFgAQ////wAzMzMFAAAAAVVVVQP///8AAABVEnd1WvDEx37/y9GD/dnfif/g54z/4OeL/97kiv6anWz/AAA1P15eeuaFg93/ioft/4iF6f+Hhen/hoPn/4mG7/14d8L/cXFimv///wP///8A////AP///wD///8AenV3a1dyZf9QY1//bGVW/pORY/+trnD/xch7/tTZhP+Pkmn6AABINXBwdb1/fc7/jYry/4mH6v+Jhur/iIbp/4iG6/6Gg+X/Z2aT+IKEaX7y8uQT////AP///wiXgY9bXndp5HXLnv95z6P+dsif/2Wrjv9RcWv/Z3Rk/nyBYv9xcWbbfHyDI4SEa354d73/kI31/YuI6/+LiOz/iofr/4mG6v+Lh+3+hoPi/21soPhnZ3qwYGBikmNza6JhinXpcsWa/3vdq/6C4a//iOi1/4zpt/+O6Lf/kOu8+3O1l/9jVmKv////BHl5WDdraor4iofn/4+M8v6Miez/i4nt/4uI7P+LiOv/i4jt/ouJ8P9+e83/XGBt/223jv952aj/e9uq/n3aqv+D36//huKy/4rntv+Q7r3/lvTC/nupj/9zYmxt////AP///wJ3d2qpfXvF/5KP9vyNiu7/jYru/4yJ7f+Miu7/i4jr/46L8P6FgN79XmZ5/XTMm/1+3q7+f9ys/4Lgr/+G5LP/iue2/4zot/+V9cL8iNWs/3BxcNni4uIa////AP///wCfn5I4aGd17YeE3v+Ukfj8jovu/46L7/+Oi+//jYru/46M8f+FgNv/XmV3/3jMnf+D4rH/g+Cw/4fks/+K57b/jem4/5TywP6Q57n/bYh5/5KIjWf///8AAAAAAgAAAAL///8AkJCFdWtqhf+IheL/lZL5/JCN8P+PjO//j4zv/5GO8/+Hgt3/X2Z4/33Rof+I5rb/h+Sz/4nmtf+N6rn/lPPB/pHruvxxm4T/h3yDqP///wD///8A////Af///wH///8A////A5aWj5JtbIP/g4DW/5SR9vyVkvj+kY7y/5KP9P+Ig97/X2d4/4DUpf+N6rn/jeq4/5Lwvv+U88D8h9qu/3KVgv+Lgoe/////F////wCAgIAC////AP///wD///8B////AP///wOfn5l9dHR39HVzsf+Fg97/k4/x/ZiW/v+Nh+f/YWl7/4ngrv+W+MX/kuy8/onesP52uZb/dH54/5aOk6P///8W////AAAAAAL///8A////AP///wD///8A////Af///wD///8Ax8fDQJWVjbV5eX79amic/3h2vv93crf/WV9r/3W1kP95vZn/aZ2C/3eCfP+Ui5DNu7i7Xv///wD///8AgICAAv///wD///8A////AP///wD///8A////AP///wEAAAAB////AP///wi6urdDuLitj6WmmcuPj4rvc3Nz/YiEh/Whlp3Wt66zoLq3ulX///8T////AP///wD///8C////AP///wD///8A////AP8B/wD8AH8A+AAfAOAADwDgAAcAwAAHAIAAAwCAAAMAgAABAAB8AQAAfgEAAP4BAAD+AQAAfgEAAHwBAIAAAQCAAAMAgAADAMAABwDgAAcA4AAPAPgAHwD8AH8A/wH/ACgAAAAgAAAAQAAAAAEAIAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAD///8A////AP///wD///8A////AP///wD///8A////AP///wD///8A////KO3t7XXFxcWwqqqq25eXl/RwcHD9mJiY9Kurq9vHx8ew8PDwdf///yj///8A////AP///wD///8A////AP///wD///8A////AP///wD///8AAAAAAP///wD///8A////AP///wD///8A////AP///wD///8A9/f3TLS0tLpvb3H/UVFR/1VVWf9bW27/WVlz/1FRUf96ZG//c2Nr/1lWV/9RUVH/d3V2/7m5ubr5+flM////AP///wD///8A////AP///wD///8A////AP///wAAAAAA////AP///wD///8A////AP///wD///8A/f39HLW1tatbW13/V1de/2FgkP9rabv/cG7S/29u0/9paL7/UVFR/9GIrv/wlcX/75bE/8+Irf+ZcYb/XVhb/2FfYP+8vLyr/v7+HP///wD///8A////AP///wD///8A////AAAAAAD///8A////AP///wD///8A////AOvr60l2dnnpVVVZ/2ZlnP9ycdL/c3LW/3Jx1f9xcNT/cG/T/2ppvv9RUVH/zoSs/+yRwf/uk8L/7pPC/++UxP/pk8D/pXaP/1hVV/9/fn/p7+/vSf///wD///8A////AP///wD///8AAAAAAP///wD///8A////AP///wDg4OBWX19h/11dcv9zccr/d3XZ/3Z02P91c9f/dHLX/3Nx1v9xcNT/a2q//1FRUf/Ngqn/6o6+/+uQwP/tksH/7pPD/++Vxf/xl8b/3I64/3Jiav9lY2T/2dnZVv///wD///8A////AP///wAAAAAA////AP///wD///8A5OTkQ1tbXf9hYIH/eHfY/3l32/94dtr/d3ba/3Z12f90c9f/c3LW/3Nx1f9tbMH/UVFR/8l/pf/mirr/6Iy8/+mOvv/qj7//7JHB/++VxP/xl8b/04qx/1RUUv9ZWVj/6+vrQ////wD///8A////AAAAAAD///8A////APX19Q9mZmjfYF98/3t53P97ed3/enjc/3l32/93dtr/dnXZ/3Z02P91c9f/dXTY/25tw/9RUVH/yHyj/+WIuf/libn/54q7/+mNvf/rj7//7JHB/9WJsf9XVlX/Z2hb/3BxX/9vb23f+fn5D////wD///8AAAAAAP///wD///8AkZCSnFpaZ/97edn/fHre/3t53v97ed3/enjc/3l33P94dtv/dnXZ/3V02P92dNj/cG7D/1FRUf/FeaH/4oW2/+OGt//liLj/54q6/+mOvv/Sha3/V1VV/2doW//FyYL/xcqC/19fWf+fn5+c////AP///wAAAAAA////ANjY2DhRUVL/dXPA/3994f9+fOD/fXvf/3x63v97ed3/enjc/3l32/94dtr/eHba/3d12f9xb8T/UVFR/8J1nf/fgLH/4IOz/+KEtf/liLn/zoCq/1dVVf9naFr/w8eA/83Shf/P1Yb/q652/1JSUf/h4eE4////AAAAAAD///8Afn6ApWNigv+BfuL/gH3h/3584P99e9//fHre/3x63v97ed3/eXjc/3l32/95d9v/d3ba/3Fwxf9RUVH/wXSc/9x9r//fgbL/4YO0/8t9pv9XVVT/Z2ha/8LHgf/N0oX/ztOF/8/Uhv/S14b/c3Vh/42Ni6X///8AAAAAAOfn5xJRUVH/eXbG/4F/4/+AfuL/f33h/3584P99e+D/fHrf/3x63v97ed3/e3ne/3p43f9ycMH/Y2OR/1FRUf+NY3j/w3Sd/9x9rv/IeaL/V1VU/2doWv/DyIH/zNGF/87Thf/Q1Yb/0daG/9TZh/+ytnj/UlJS/+/v7xIAAAAAq6urYF1cav+DgeX/goDk/4F/4/+AfuL/f33h/3994f9+fOD/fXvf/3x63v95d9X/YF99/1FRUb9sbG+AZmVmgG1qa4BRUVG/fWBv/1dVVP9naFr/xMiB/83Shv/O04b/z9WG/9HWhv/S14b/1NqH/9TZhv9fYFn/t7e3YAAAAAB6enyeaGeV/4SC5v+DgeX/goDk/4F+4/+BfuP/f33h/3584P9+fOD/fHrb/1xcbv9sbG6g39/fQP///wD///8A////ANra2kBbWlugVVVT/8LHgf/O04X/ztOF/8/Uhf/Q1Yb/0teG/9PZh//V24f/2N6I/4aIZv+IiIeeAAAAAFpaXMx0crX/hYLm/4SC5v+DgeX/goDk/4KA5P+Bf+P/gH7i/3994f9oZ5v/X19hv/Hx8SD///8A////AP///wD///8A////AOfn5yBcXFq/m55w/9DVhv/R1ob/0teG/9LXhv/T2Yb/1dqH/9jdiP/Z34j/o6Zy/2VlY8wAAAAAUVFR7Hx7zP+GhOj/hYLm/4SB5f+Egub/g4Hl/4KA5P+Bf+P/gX/j/1paZP+xsbFA////AP///wD///8A////AP///wD///8A////AKampkBqa17/0teG/9LXhv/T2Yf/1dqH/9bciP/Y3oj/2N6H/9rgiP+5vXr/UlJS7AAAAABRUVH7gX7W/4eE6P+GhOj/hYPn/4WC5v+EgeX/g4Hl/4OB5f9/fNv/UVFR/9zc3AD///8A////AP///wD///8A////AP///wD///8A0NDQAFRUU//R14b/1dqH/9Xbh//W3If/192H/9jeh//a4Ij/3OKI/8PJfv9RUVH7AAAAAFFRUfuCgNj/iIbq/4eE6P+HhOj/hoPn/4WC5v+Egub/hILm/4B+3f9RUVH/3NzcAP///wD///8A////AP///wD///8A////AP///wDLy8sAUlJR/87ThP/W3Ij/192H/9nfiP/Z34j/2uCI/9vhiP/d44j/xMh+/1FRUfsAAAAAUVFR7H99zv+Jh+v/iIXq/4iF6v+Hhen/hoTo/4aD5/+Fgub/hYLm/1taZP+xsbFA////AP///wD///8A////AP///wD///8A////AJWVlUBSUlH/UlJR/25vXf+Ul2z/ur97/9jeh//c44j/3uWJ/97lif+8wXr/UVFR7AAAAABaWlzMd3W4/4qH6/+Kh+v/iYbq/4iF6f+HhOj/iIXp/4aE6P+GhOj/a2qe/19fYr/x8fEg////AP///wD///8A////AP///wDp6ekgW1xcv2SWe/9nm4D/Wm9k/1daVv9RUVH/VlZT/3p8Yf+ipnH/ys+A/6ircv9lZmPMAAAAAHp6fJ5rapf/i4js/4uJ7f+KiOz/iYbq/4mG6v+Ihur/iIbq/4eE6P+FguP/Xl5w/2xsb6Df399A////AP///wD///8A2traQGVnZqBccGX/d9Sk/3vZqf+B367/g+Gw/33No/9qk33/W2lh/1ZXVP9RUVH/U1NR/3d3dZ4AAAAAq6urYF5ea/+Ni+//jInt/4uI7P+LiOz/iofr/4mG6v+Jhur/iIXp/4iF6f+EguD/ZGSB/1FRUb9ra2+AZWdmgGpubIBRUVG/Xn1u/3TOn/9516f/ftyr/4Herf+G47L/iOa1/4vot/+Q7bz/k+++/4PHo/9SVFP/oaGhYAAAAADn5+cSUVFR/4F/z/+Niu7/jInt/4yJ7f+LiOz/i4nt/4qI7P+LiOz/iofr/4mG6v+Jhur/fnzN/2ZljP9RUVH/YpB5/2++lf921KT/edem/3vZqf+A3q3/hOGw/4bjs/+J5rX/juu6/5Dsu/+U8cD/g8Wi/1JTUv/t7e0SAAAAAP///wB+foClZ2aG/46L8P+Oi/D/jYru/4yJ7f+Mie3/i4js/4uI7P+LiOz/iofr/4qH6/+KiOz/eXjA/1FRUf9zyJz/eden/3zaqf982qn/ftys/4Hfrv+G47L/iOW0/4rntv+N6rn/ku++/5XxwP9le2//iYqKpf///wAAAAAA////ANjY2DhRUVL/gH7L/4+M8P+Oi+//jovv/42K7v+Niu7/jIru/4yK7v+Mie3/jInt/4uI7P96eMD/UVFR/3bLoP972an/ftyr/4Herv+D4K//heOy/4jmtf+L6Lf/juu5/5DtvP+V8cD/gcGf/1NUU//l5eU4////AAAAAAD///8A////AJGRkpxdXGn/jYrp/5CN8f+PjPD/j4zw/46L7/+Oi+//jYru/42K7v+Niu7/jYru/3x5wv9RUVH/ec6i/4Herv+B367/hOGw/4bjsv+I5bT/i+i3/43quf+Q7bz/k+++/4/jtv9bY17/nZ6enP///wD///8AAAAAAP///wD///8A9vb2D2Zmad9mZYL/kI3w/5CN8f+QjfH/j4zw/4+M8P+PjPD/j4zw/46L7/+Oi+//fHrD/1FRUf9/0qb/hOGw/4Xjsv+F47L/iOa1/4vot/+O67r/kO28/5Lvvf+S7b3/ZHht/3N1dN/6+voP////AP///wAAAAAA////AP///wD///8A5OTkQ1xcXv9oZ4j/kI3v/5GN8f+RjfH/kI3x/5CN8f+QjfH/kI3x/5CN8f99e8P/UVFR/3/UqP+I5bT/iOW0/4vot/+L6Lf/juu5/4/su/+S773/kem6/2Z9cf9gYmH/6+vrQ////wD///8A////AAAAAAD///8A////AP///wD///8A4ODgVl9fYf9iYnf/iojh/5KP8/+Sj/P/kY7y/5GO8v+RjvL/kY7y/358xf9RUVH/hdmt/4vot/+O67n/juu5/5Dtu/+Q7bv/ku++/4rXrv9hcGj/aGpp/+np6Vb///8A////AP///wD///8AAAAAAP///wD///8A////AP///wD///8A7OzsSXd3eelWVlr/dXOr/5CO7v+TkPT/k5D0/5KP8/+Sj/P/f33F/1FRUf+I267/kO28/5Dsu/+Q7Lv/ku+9/5Douv9zoon/U1VU/4GDgunx8fFJ////AP///wD///8A////AP///wAAAAAA////AP///wD///8A////AP///wD///8A/f39HLW1tatbW13/WVlg/3Bunv+Gg9T/k4/y/5SR9f+Bfsf/UVFR/4vfsv+S773/kOu7/4TKpf9ul4H/VlpY/19hYP/AwMCr////HP///wD///8A////AP///wD///8A////AAAAAAD///8A////AP///wD///8A////AP///wD///8A////APf390y0tLS6cG9y/1FRUf9WVlr/Y2J2/2Fgdv9RUVH/ZYBx/2N1a/9VV1b/UVFR/3R2df+/v7+6+vr6TP///wD///8A////AP///wD///8A////AP///wD///8AAAAAAP///wD///8A////AP///wD///8A////AP///wD///8A////AP///wD///8o7e3tdcXFxbCqqqrbl5eX9HBwcP2YmJj0q6ur28fHx7Dv7+91////KP///wD///8A////AP///wD///8A////AP///wD///8A////AP///wAAAAAA//Af//+AA//+AAD//AAAf/gAAD/wAAAf4AAAD8AAAAfAAAAHgAAAA4AAAAOAAAADAAfAAQAP4AEAH/ABAB/wAQAf8AEAH/ABAA/gAQAHwAGAAAADgAAAA4AAAAPAAAAHwAAAB+AAAA/wAAAf+AAAP/wAAH/+AAD//4AD///wH/8='
_KRONA_IMAGE_HIDDEN = 'data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAgAAAAICAYAAADED76LAAAAAXNSR0IArs4c6QAAAAZiS0dEAP8A/wD/oL2nkwAAAAlwSFlzAAALEwAACxMBAJqcGAAAAAd0SU1FB9oLCBQhNQwWVnsAAAAidEVYdENvbW1lbnQAQ3JlYXRlZCB3aXRoIEdJTVAgb24gYSBNYWOHqHdDAAABE0lEQVQYGQEIAff+AwAAABkAAAAAAAAA+gAAAAAAAAAAAAAAAAAAAAAAAAAMAwAAAAAAAAANAAAAAAAAAPoAAAAAAAAADAAAAAYAAAD0AwAAAPoAAAAAAAAAAAAAAPoAAAAMAAAADQAAAPoAAAD6AAAAAAAAAAAAAAAAAAAAAAwAAAAZAAAADAAAAAAAAAAAAAAAAAAAAAAAAAAADAAAABkAAAAMAAAAAAAAAAAAAAAAAAAAAAAAAAAMAAAAGQAAAAwAAAAAAAAADAAAAAwAAAAABAAAAAAAAAAAAAAA8wAAAPQAAAAAAAAAAAAAAA0AAAAAAAAAAAwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAwAAAAZRssKC5OpXwYAAAAASUVORK5CYII='
_KRONA_IMAGE_LOADING = 'data:image/gif;base64,R0lGODlhEAAQAPIAAP///wAAAMLCwkJCQgAAAGJiYoKCgpKSkiH/C05FVFNDQVBFMi4wAwEAAAAh/hpDcmVhdGVkIHdpdGggYWpheGxvYWQuaW5mbwAh+QQJCgAAACwAAAAAEAAQAAADMwi63P4wyklrE2MIOggZnAdOmGYJRbExwroUmcG2LmDEwnHQLVsYOd2mBzkYDAdKa+dIAAAh+QQJCgAAACwAAAAAEAAQAAADNAi63P5OjCEgG4QMu7DmikRxQlFUYDEZIGBMRVsaqHwctXXf7WEYB4Ag1xjihkMZsiUkKhIAIfkECQoAAAAsAAAAABAAEAAAAzYIujIjK8pByJDMlFYvBoVjHA70GU7xSUJhmKtwHPAKzLO9HMaoKwJZ7Rf8AYPDDzKpZBqfvwQAIfkECQoAAAAsAAAAABAAEAAAAzMIumIlK8oyhpHsnFZfhYumCYUhDAQxRIdhHBGqRoKw0R8DYlJd8z0fMDgsGo/IpHI5TAAAIfkECQoAAAAsAAAAABAAEAAAAzIIunInK0rnZBTwGPNMgQwmdsNgXGJUlIWEuR5oWUIpz8pAEAMe6TwfwyYsGo/IpFKSAAAh+QQJCgAAACwAAAAAEAAQAAADMwi6IMKQORfjdOe82p4wGccc4CEuQradylesojEMBgsUc2G7sDX3lQGBMLAJibufbSlKAAAh+QQJCgAAACwAAAAAEAAQAAADMgi63P7wCRHZnFVdmgHu2nFwlWCI3WGc3TSWhUFGxTAUkGCbtgENBMJAEJsxgMLWzpEAACH5BAkKAAAALAAAAAAQABAAAAMyCLrc/jDKSatlQtScKdceCAjDII7HcQ4EMTCpyrCuUBjCYRgHVtqlAiB1YhiCnlsRkAAAOwAAAAAAAAAAAA=='
_KRONA_IMAGE_LOGO = '...'


# =============================================================================
# HELPERS
# =============================================================================

EXPECTED_TAXONOMIC_RANKS = [
    "domain",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
]

KNOWN_NON_SAMPLE_COLUMNS = {
    "SPPN",
    "OTU_XX",
    "Original_ID",
    "Original_OTUID",
    "OTU_Number",
    "domain",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
    "sequence",
    "Sequence",
    "sequence_lenght",
    "sequence_length",
    "Total_Abundance",
    "sintax_taxonomy",
    "primary_lifestyle",
    "Secondary_lifestyle",
    "secondary_lifestyle",
    "functional_group",
    "FAPROTAX_functions",
    "FAPROTAX_groups",
    "faprotax_functions",
    "faprotax_function_count",
    "matched_lifestyle",
    "matched_lifestyle_source",
}

def safe_filename(name):
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_") or "unnamed"

def clean_krona_value(value, column_name):
    if pd.isna(value):
        value = ""
    value = str(value).strip()

    if value.lower() in {"", "nan", "none", "null", "na", "n/a"}:
        if not FILL_UNCLASSIFIED:
            return ""
        if column_name == "primary_lifestyle":
            return "Primary_lifestyle_not_assigned"
        if column_name in {"Secondary_lifestyle", "secondary_lifestyle"}:
            return "Secondary_lifestyle_not_assigned"
        if column_name == "SPPN":
            return "SPPN_not_assigned"
        return f"Unclassified_{column_name}"

    value = re.sub(r"[\t\r\n]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

def detect_sample_columns(df):
    sample_columns = []

    for column in df.columns:
        if column in KNOWN_NON_SAMPLE_COLUMNS:
            continue
        if str(column).lower().endswith("_pvalue"):
            continue

        numeric_values = pd.to_numeric(df[column], errors="coerce")
        numeric_fraction = numeric_values.notna().mean()

        # MetaDiv sample columns are abundance vectors and should be overwhelmingly numeric.
        if numeric_fraction > 0.80:
            sample_columns.append(column)

    return sample_columns

def build_hierarchy(df):
    """
    Build the Krona hierarchy and optionally stop taxonomy at a
    user-selected rank (e.g. family, genus or species).
    """

    available_ranks = [
        r for r in EXPECTED_TAXONOMIC_RANKS
        if r in df.columns
    ]

    if not available_ranks:
        raise ValueError(
            "No recognized taxonomic columns were found. "
            "Expected at least one of: "
            + ", ".join(EXPECTED_TAXONOMIC_RANKS)
        )

    # ----------------------------------------------------------
    # Apply user-selected taxonomic cutoff
    # ----------------------------------------------------------

    if TAXONOMIC_CUTOFF is None:
        ranks = available_ranks.copy()

    else:
        cutoff = str(TAXONOMIC_CUTOFF).strip().lower()

        if cutoff not in EXPECTED_TAXONOMIC_RANKS:
            raise ValueError(
                f"Invalid TAXONOMIC_CUTOFF: {TAXONOMIC_CUTOFF!r}\n"
                "Choose one of: None, "
                + ", ".join(EXPECTED_TAXONOMIC_RANKS)
            )

        if cutoff not in available_ranks:
            raise ValueError(
                f"TAXONOMIC_CUTOFF='{cutoff}' was requested, "
                f"but the column '{cutoff}' is not present in the CSV.\n"
                "Available taxonomic ranks: "
                + ", ".join(available_ranks)
            )

        cutoff_position = EXPECTED_TAXONOMIC_RANKS.index(cutoff)

        ranks = [
            r for r in available_ranks
            if EXPECTED_TAXONOMIC_RANKS.index(r) <= cutoff_position
        ]

    # Taxonomy stops here.
    hierarchy = ranks.copy()

    # Optional ecological levels are added AFTER the taxonomic cutoff.
    if INCLUDE_PRIMARY_LIFESTYLE and "primary_lifestyle" in df.columns:
        hierarchy.append("primary_lifestyle")

    if INCLUDE_SECONDARY_LIFESTYLE:
        if "Secondary_lifestyle" in df.columns:
            hierarchy.append("Secondary_lifestyle")
        elif "secondary_lifestyle" in df.columns:
            hierarchy.append("secondary_lifestyle")

    if INCLUDE_SPPN and "SPPN" in df.columns:
        hierarchy.append("SPPN")

    return ranks, hierarchy

def ensure_total_abundance(df, sample_columns):
    """
    GLOBAL MODE abundance.

    Default behavior:
        SUM every detected sample/site abundance column.

    Fallback:
        if no sample/site columns can be detected, use an existing
        Total_Abundance column.
    """

    if sample_columns:
        numeric = (
            df[sample_columns]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .fillna(0)
        )

        # Always overwrite/create Total_Abundance from the actual
        # detected site/sample columns so GLOBAL truly means SUM ALL SITES.
        df["Total_Abundance"] = numeric.sum(
            axis=1
        )

        print(
            f"   GLOBAL MODE: summed "
            f"{len(sample_columns):,} detected site/sample columns "
            "into Total_Abundance."
        )

        return df

    if "Total_Abundance" in df.columns:
        df["Total_Abundance"] = pd.to_numeric(
            df["Total_Abundance"],
            errors="coerce",
        ).fillna(0)

        print(
            "   GLOBAL MODE: no site/sample columns were detected; "
            "using existing Total_Abundance as fallback."
        )

        return df

    raise ValueError(
        "Global mode requires detected site/sample abundance columns "
        "or an existing Total_Abundance column."
    )


def _krona_is_unclassified(value, rank=None):
    """
    Return True for blank / unclassified taxonomy labels.
    """
    if pd.isna(value):
        return True

    text = str(value).strip()
    if text == "":
        return True

    low = text.casefold()

    if low in {
        "nan", "none", "null", "na", "n/a",
        "unclassified", "uncultured", "unknown",
    }:
        return True

    if low.startswith("unclassified_"):
        return True

    return False


def _krona_taxon_key(value):
    """
    Case-insensitive key used only for matching the same taxon label.
    The original preferred spelling is preserved in the final lineage.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip()
    text = re.sub(r"[\t\r\n]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text.casefold()


def _krona_weight_series(df, sample_columns):
    """
    Weight lineage decisions by total abundance.

    Priority:
      1. Total_Abundance when available.
      2. Sum of detected sample columns.
      3. Equal row weight when neither is available.
    """
    if "Total_Abundance" in df.columns:
        return pd.to_numeric(
            df["Total_Abundance"],
            errors="coerce",
        ).fillna(0)

    valid_samples = [
        c for c in sample_columns
        if c in df.columns
    ]

    if valid_samples:
        numeric = (
            df[valid_samples]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .fillna(0)
        )

        return numeric.sum(axis=1)

    return pd.Series(
        1.0,
        index=df.index,
        dtype=float,
    )


def homogenize_taxonomy_for_krona(
    df,
    tax_ranks,
    sample_columns,
):
    """
    Homogenize parent lineages before Krona aggregation.

    Why this is necessary
    ---------------------
    Krona is path-based. The same genus will be drawn more than once if
    the input contains, for example:

        Russulales -> Russulaceae -> Russula
        Agaricales -> Russulaceae -> Russula
        Unclassified_order -> Unclassified_family -> Russula

    even though the terminal genus label is identical.

    Strategy
    --------
    For every classified taxon at each rank, MetaDiv_KRONA identifies the
    best-supported upstream lineage and applies it consistently to all rows
    containing that taxon.

    Candidate lineages are ranked by:
      1. number of classified upstream ranks (most complete first);
      2. summed abundance (highest first);
      3. number of supporting rows (highest first).

    This allows a complete lineage to fill incomplete parent ranks while
    abundance resolves conflicts between equally complete lineages.

    Returns
    -------
    homogenized_df, log_df
    """

    work = df.copy()

    ranks = [
        rank for rank in tax_ranks
        if rank in work.columns
    ]

    if len(ranks) < 2:
        return work, pd.DataFrame()

    # Clean taxonomy once before comparing paths.
    for rank in ranks:
        work[rank] = work[rank].apply(
            lambda value, r=rank:
            clean_krona_value(value, r)
        )

    weights = _krona_weight_series(
        work,
        sample_columns,
    )

    log_records = []

    # Shallow -> deep.
    # When a deeper taxon is reached, its parent ranks have already been
    # internally standardized as much as possible.
    for rank_index in range(
        1,
        len(ranks),
    ):
        rank = ranks[rank_index]
        parent_ranks = ranks[:rank_index]

        valid_mask = ~work[rank].apply(
            lambda value:
            _krona_is_unclassified(
                value,
                rank,
            )
        )

        if not valid_mask.any():
            continue

        taxon_keys = (
            work.loc[
                valid_mask,
                rank,
            ]
            .apply(
                _krona_taxon_key
            )
        )

        for taxon_key in sorted(
            set(taxon_keys)
        ):
            if not taxon_key:
                continue

            member_index = taxon_keys.index[
                taxon_keys == taxon_key
            ]

            if len(member_index) <= 1:
                continue

            # Collect alternative observed parent lineages.
            candidates = {}

            for idx in member_index:
                parent_path = tuple(
                    str(
                        work.at[
                            idx,
                            parent,
                        ]
                    ).strip()
                    for parent
                    in parent_ranks
                )

                completeness = sum(
                    not _krona_is_unclassified(
                        value,
                        parent,
                    )
                    for parent, value
                    in zip(
                        parent_ranks,
                        parent_path,
                    )
                )

                if parent_path not in candidates:
                    candidates[parent_path] = {
                        "completeness": completeness,
                        "abundance": 0.0,
                        "rows": 0,
                    }

                candidates[parent_path][
                    "abundance"
                ] += float(
                    weights.loc[idx]
                )

                candidates[parent_path][
                    "rows"
                ] += 1

            if len(candidates) <= 1:
                continue

            # Best lineage:
            # complete > abundant > frequently observed.
            canonical_path = max(
                candidates,
                key=lambda path: (
                    candidates[path][
                        "completeness"
                    ],
                    candidates[path][
                        "abundance"
                    ],
                    candidates[path][
                        "rows"
                    ],
                    tuple(
                        str(value)
                        for value
                        in path
                    ),
                ),
            )

            changed_rows = 0
            changed_abundance = 0.0

            for idx in member_index:
                old_path = tuple(
                    str(
                        work.at[
                            idx,
                            parent,
                        ]
                    ).strip()
                    for parent
                    in parent_ranks
                )

                if old_path == canonical_path:
                    continue

                for parent, value in zip(
                    parent_ranks,
                    canonical_path,
                ):
                    work.at[
                        idx,
                        parent,
                    ] = value

                changed_rows += 1
                changed_abundance += float(
                    weights.loc[idx]
                )

            if changed_rows > 0:
                # Preferred display spelling for the taxon itself:
                # use the most abundant observed spelling.
                spelling_weights = {}

                for idx in member_index:
                    spelling = str(
                        work.at[
                            idx,
                            rank,
                        ]
                    ).strip()

                    spelling_weights[
                        spelling
                    ] = (
                        spelling_weights.get(
                            spelling,
                            0.0,
                        )
                        + float(
                            weights.loc[idx]
                        )
                    )

                preferred_taxon = max(
                    spelling_weights,
                    key=spelling_weights.get,
                )

                work.loc[
                    member_index,
                    rank,
                ] = preferred_taxon

                lineage_text = " > ".join(
                    canonical_path
                    + (
                        preferred_taxon,
                    )
                )

                alternative_text = " || ".join(
                    " > ".join(
                        path
                        + (
                            preferred_taxon,
                        )
                    )
                    for path
                    in candidates
                )

                log_records.append(
                    {
                        "rank": rank,
                        "taxon": preferred_taxon,
                        "alternative_parent_lineages": len(
                            candidates
                        ),
                        "rows_changed": changed_rows,
                        "abundance_changed": changed_abundance,
                        "canonical_lineage": lineage_text,
                        "observed_lineages": alternative_text,
                    }
                )

    log_df = pd.DataFrame(
        log_records
    )

    return work, log_df



def create_krona_table(data, abundance_column, hierarchy_columns):
    if abundance_column not in data.columns:
        raise ValueError(f"Abundance column not found: {abundance_column}")

    work = data[[abundance_column] + hierarchy_columns].copy()
    work[abundance_column] = pd.to_numeric(
        work[abundance_column], errors="coerce"
    ).fillna(0)

    if DROP_ZERO_ABUNDANCE:
        work = work.loc[work[abundance_column] > 0].copy()

    for column in hierarchy_columns:
        work[column] = work[column].apply(
            lambda x: clean_krona_value(x, column)
        )

    # If the user explicitly chooses blank missing levels, remove empty labels
    # only at the trailing end. Internal blank paths are replaced to preserve tree integrity.
    if not FILL_UNCLASSIFIED:
        for column in hierarchy_columns:
            work[column] = work[column].replace("", f"Unclassified_{column}")

    grouped = (
        work.groupby(hierarchy_columns, dropna=False, as_index=False)[abundance_column]
            .sum()
            .rename(columns={abundance_column: "abundance"})
            .sort_values("abundance", ascending=False, kind="stable")
            .reset_index(drop=True)
    )

    if MAX_KRONA_PATHS is not None and len(grouped) > int(MAX_KRONA_PATHS):
        before = len(grouped)
        grouped = grouped.head(int(MAX_KRONA_PATHS)).copy()
        print(
            f"   ⚠️ Krona path limit applied: {before:,} -> {len(grouped):,} "
            f"(top paths by abundance)."
        )

    grouped.attrs["hierarchy"] = hierarchy_columns
    return grouped

def export_krona_text(krona_df, output_file):
    # Headerless TAB format, compatible with KronaTools if the user ever wants it.
    cols = ["abundance"] + list(krona_df.attrs.get("hierarchy", []))
    krona_df[cols].to_csv(
        output_file,
        sep="\t",
        index=False,
        header=False,
        encoding="utf-8",
    )

def _build_krona_tree(krona_table, hierarchy_columns, dataset_name):
    root = {"name": dataset_name, "magnitude": 0, "children": {}}
    columns = list(krona_table.columns)

    for row in krona_table.itertuples(index=False, name=None):
        row_map = dict(zip(columns, row))

        try:
            amount = int(float(row_map.get("abundance", 0) or 0))
        except Exception:
            amount = 0

        if amount <= 0:
            continue

        root["magnitude"] += amount
        current = root

        for column in hierarchy_columns:
            label = str(row_map.get(column, "") or "").strip()
            if not label:
                label = f"Unclassified_{column}"

            child = current["children"].get(label)
            if child is None:
                child = {"name": label, "magnitude": 0, "children": {}}
                current["children"][label] = child

            child["magnitude"] += amount
            current = child

    return root

def _krona_node_xml(node, indent=12):
    pad = " " * indent
    name = html_lib.escape(str(node.get("name", "")), quote=True)
    magnitude = int(node.get("magnitude", 0) or 0)

    out = [
        f'{pad}<node name="{name}">',
        f'{pad}  <magnitude><val>{magnitude}</val></magnitude>',
        f'{pad}  <score><val></val></score>',
    ]

    children = node.get("children", {})
    ordered = sorted(
        children.values(),
        key=lambda x: (-int(x.get("magnitude", 0) or 0), str(x.get("name", ""))),
    )

    for child in ordered:
        out.append(_krona_node_xml(child, indent + 2))

    out.append(f"{pad}</node>")
    return "\n".join(out)

def build_krona_html(krona_table, output_file, dataset_name):
    if krona_table.empty:
        raise ValueError("Cannot create Krona HTML from an empty table.")

    hierarchy = list(krona_table.attrs.get("hierarchy", []))
    tree = _build_krona_tree(krona_table, hierarchy, dataset_name)
    dataset_escaped = html_lib.escape(str(dataset_name), quote=False)
    node_xml = _krona_node_xml(tree, indent=12)

    document = (
        '<!DOCTYPE html>\n'
        '<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">\n'
        '<head>\n<meta charset="UTF-8"/>\n'
        f'<title>MetaDiv_KRONA - {dataset_escaped}</title>\n'
        f'<link rel="shortcut icon" href="{_KRONA_IMAGE_SHORTCUT}"/>\n'
        '<script>\n' + _KRONA_LEGACY_JS + '\n</script>\n</head>\n<body>\n'
        f'<img id="hiddenImage" src="{_KRONA_IMAGE_HIDDEN}" style="display:none"/>\n'
        f'<img id="loadingImage" src="{_KRONA_IMAGE_LOADING}" style="display:none"/>\n'
        f'<img id="logo" src="{_KRONA_IMAGE_LOGO}" style="display:none"/>\n'
        '<noscript>Javascript must be enabled to view this page.</noscript>\n'
        '<div style="display:none">\n  <krona>\n'
        '    <attributes magnitude="magnitude">\n'
        '      <attribute display="Amount">magnitude</attribute>\n'
        '      <attribute display="Score">score</attribute>\n'
        '    </attributes>\n'
        '    <datasets>\n'
        f'      <dataset>{dataset_escaped}</dataset>\n'
        '    </datasets>\n'
        + node_xml + '\n'
        '  </krona>\n</div>\n</body>\n</html>\n'
    )

    Path(output_file).write_text(document, encoding="utf-8")
    return Path(output_file)



# =============================================================================
# EXPORT MODES
# =============================================================================
#
# EXPORT_PER_SAMPLE = False
#     ONE global chart:
#         taxonomy -> optional lifestyle
#
# EXPORT_PER_SAMPLE = True
#     ONE chart with every sample kept separate:
#         sample -> taxonomy -> optional lifestyle
#
# The TRUE mode internally converts the wide abundance matrix to long format.
# It does NOT create a separate HTML file for every sample.
# =============================================================================


def canonicalize_optional_columns(df):
    """Normalize optional column names used by different MetaDiv tables."""
    df = df.copy()

    if (
        "Secondary_lifestyle" not in df.columns
        and "secondary_lifestyle" in df.columns
    ):
        df["Secondary_lifestyle"] = df["secondary_lifestyle"]

    return df


def create_all_samples_long_table(
    data,
    sample_columns,
    hierarchy_columns,
):
    """
    Convert the wide MetaDiv abundance matrix to LONG format for
    a TRUE multi-dataset Krona chart.

    Output columns:

        Dataset | abundance | taxonomy...

    Example:

        Dataset   abundance   phylum          class              genus
        Site_A    120         Basidiomycota   Agaricomycetes     Russula
        Site_A     35         Ascomycota      Sordariomycetes    Fusarium
        Site_B     80         Basidiomycota   Agaricomycetes     Russula

    Dataset is NOT part of the taxonomic hierarchy.

    The same taxonomic tree is shared by every Dataset, while Krona stores
    one abundance value per Dataset at each node. This is what enables the
    dataset selector inside ONE Krona HTML.
    """

    if not sample_columns:
        raise ValueError(
            'MODE="samples" requires sample/site abundance columns.'
        )

    long_parts = []

    for sample in sample_columns:

        work = data[
            [sample] + hierarchy_columns
        ].copy()

        work[sample] = pd.to_numeric(
            work[sample],
            errors="coerce",
        ).fillna(0)

        if DROP_ZERO_ABUNDANCE:
            work = work.loc[
                work[sample] > 0
            ].copy()

        if work.empty:
            continue

        # Taxonomy has already been homogenized before this function.
        # Here we only standardize remaining empty labels.
        for column in hierarchy_columns:
            work[column] = work[column].apply(
                lambda value, col=column:
                clean_krona_value(value, col)
            )

        # Aggregate identical taxonomic/ecological paths WITHIN this site.
        work = (
            work.groupby(
                hierarchy_columns,
                dropna=False,
                as_index=False,
            )[sample]
            .sum()
            .rename(
                columns={
                    sample: "abundance"
                }
            )
        )

        # Dataset must be the FIRST column in the long-format table.
        work.insert(
            0,
            "Dataset",
            str(sample),
        )

        long_parts.append(work)

    if not long_parts:
        raise ValueError(
            "No positive sample/site abundances were available for Krona."
        )

    long_table = pd.concat(
        long_parts,
        ignore_index=True,
    )

    # Re-aggregate defensively in case identical paths were produced twice.
    long_table = (
        long_table.groupby(
            ["Dataset"] + hierarchy_columns,
            dropna=False,
            as_index=False,
        )["abundance"]
        .sum()
    )

    # Dataset first, then abundance, then the common biological hierarchy.
    long_table = long_table[
        ["Dataset", "abundance"] + hierarchy_columns
    ]

    long_table = (
        long_table
        .sort_values(
            ["Dataset", "abundance"],
            ascending=[True, False],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    if (
        MAX_KRONA_PATHS is not None
        and len(long_table) > int(MAX_KRONA_PATHS)
    ):
        before = len(long_table)

        long_table = (
            long_table
            .sort_values(
                "abundance",
                ascending=False,
                kind="stable",
            )
            .head(
                int(MAX_KRONA_PATHS)
            )
            .sort_values(
                ["Dataset", "abundance"],
                ascending=[True, False],
                kind="stable",
            )
            .reset_index(drop=True)
        )

        print(
            f"   WARNING: Krona path limit applied: "
            f"{before:,} -> {len(long_table):,}"
        )

    # IMPORTANT:
    # hierarchy contains taxonomy/lifestyle only.
    # Dataset is stored separately for Krona's dataset selector.
    long_table.attrs["hierarchy"] = list(hierarchy_columns)
    long_table.attrs["datasets"] = [
        str(sample)
        for sample in sample_columns
        if str(sample) in set(long_table["Dataset"])
    ]

    return long_table


# =============================================================================
# TRUE MULTI-DATASET KRONA HTML
# =============================================================================

def _build_krona_multidataset_tree(
    krona_table,
    hierarchy_columns,
    dataset_names,
    root_name,
):
    """
    Build ONE shared taxonomic tree with a vector of abundances
    (one value per Dataset) stored at every node.
    """

    dataset_index = {
        name: i
        for i, name in enumerate(dataset_names)
    }

    n_datasets = len(dataset_names)

    root = {
        "name": root_name,
        "magnitudes": [0] * n_datasets,
        "children": {},
    }

    columns = list(krona_table.columns)

    for row in krona_table.itertuples(
        index=False,
        name=None,
    ):
        row_map = dict(zip(columns, row))

        dataset = str(
            row_map.get("Dataset", "")
            or ""
        ).strip()

        if dataset not in dataset_index:
            continue

        ds_i = dataset_index[dataset]

        try:
            amount = int(
                float(
                    row_map.get(
                        "abundance",
                        0,
                    )
                    or 0
                )
            )
        except Exception:
            amount = 0

        if amount <= 0:
            continue

        root["magnitudes"][ds_i] += amount

        current = root

        for column in hierarchy_columns:

            label = str(
                row_map.get(
                    column,
                    "",
                )
                or ""
            ).strip()

            if not label:
                label = f"Unclassified_{column}"

            child = current["children"].get(label)

            if child is None:
                child = {
                    "name": label,
                    "magnitudes": [0] * n_datasets,
                    "children": {},
                }

                current["children"][label] = child

            child["magnitudes"][ds_i] += amount

            current = child

    return root


def _krona_multidataset_node_xml(
    node,
    dataset_count,
    indent=12,
):
    """
    Serialize one Krona node.

    Krona's legacy renderer expects one <val> per Dataset:

        <magnitude>
            <val>Site_A abundance</val>
            <val>Site_B abundance</val>
            ...
        </magnitude>
    """

    pad = " " * indent

    name = html_lib.escape(
        str(
            node.get(
                "name",
                "",
            )
        ),
        quote=True,
    )

    magnitudes = list(
        node.get(
            "magnitudes",
            [],
        )
    )

    if len(magnitudes) < dataset_count:
        magnitudes.extend(
            [0] * (
                dataset_count
                - len(magnitudes)
            )
        )

    out = [
        f'{pad}<node name="{name}">',
        f"{pad}  <magnitude>",
    ]

    for value in magnitudes[:dataset_count]:
        out.append(
            f"{pad}    <val>{int(value or 0)}</val>"
        )

    out.append(
        f"{pad}  </magnitude>"
    )

    # Keep score vectors aligned with the number of datasets.
    out.append(
        f"{pad}  <score>"
    )

    for _ in range(dataset_count):
        out.append(
            f"{pad}    <val></val>"
        )

    out.append(
        f"{pad}  </score>"
    )

    children = node.get(
        "children",
        {},
    )

    # Sort by total abundance across all datasets.
    ordered = sorted(
        children.values(),
        key=lambda child: (
            -sum(
                int(v or 0)
                for v in child.get(
                    "magnitudes",
                    [],
                )
            ),
            str(
                child.get(
                    "name",
                    "",
                )
            ),
        ),
    )

    for child in ordered:
        out.append(
            _krona_multidataset_node_xml(
                child,
                dataset_count,
                indent + 2,
            )
        )

    out.append(
        f"{pad}</node>"
    )

    return "\n".join(out)


def build_krona_multidataset_html(
    krona_table,
    output_file,
    chart_name,
):
    """
    Create ONE self-contained Krona HTML with multiple selectable Datasets.

    Each site/sample is listed in <datasets>, while the taxonomy is shared.
    """

    if krona_table.empty:
        raise ValueError(
            "Cannot create multi-dataset Krona HTML from an empty table."
        )

    hierarchy = list(
        krona_table.attrs.get(
            "hierarchy",
            [],
        )
    )

    dataset_names = list(
        krona_table.attrs.get(
            "datasets",
            [],
        )
    )

    if not dataset_names:
        dataset_names = list(
            dict.fromkeys(
                krona_table[
                    "Dataset"
                ].astype(str)
            )
        )

    if not dataset_names:
        raise ValueError(
            "No Dataset names were found for samples mode."
        )

    tree = _build_krona_multidataset_tree(
        krona_table=krona_table,
        hierarchy_columns=hierarchy,
        dataset_names=dataset_names,
        root_name=chart_name,
    )

    chart_escaped = html_lib.escape(
        str(chart_name),
        quote=False,
    )

    datasets_xml = "\n".join(
        "      <dataset>"
        + html_lib.escape(
            str(dataset),
            quote=False,
        )
        + "</dataset>"
        for dataset in dataset_names
    )

    node_xml = _krona_multidataset_node_xml(
        tree,
        dataset_count=len(dataset_names),
        indent=12,
    )

    document = (
        '<!DOCTYPE html>\n'
        '<html xmlns="http://www.w3.org/1999/xhtml" '
        'xml:lang="en" lang="en">\n'
        '<head>\n'
        '<meta charset="UTF-8"/>\n'
        f'<title>MetaDiv_KRONA - {chart_escaped}</title>\n'
        f'<link rel="shortcut icon" href="{_KRONA_IMAGE_SHORTCUT}"/>\n'
        '<script>\n'
        + _KRONA_LEGACY_JS
        + '\n</script>\n'
        '</head>\n'
        '<body>\n'
        f'<img id="hiddenImage" src="{_KRONA_IMAGE_HIDDEN}" '
        'style="display:none"/>\n'
        f'<img id="loadingImage" src="{_KRONA_IMAGE_LOADING}" '
        'style="display:none"/>\n'
        f'<img id="logo" src="{_KRONA_IMAGE_LOGO}" '
        'style="display:none"/>\n'
        '<noscript>Javascript must be enabled to view this page.</noscript>\n'
        '<div style="display:none">\n'
        '  <krona>\n'
        '    <attributes magnitude="magnitude">\n'
        '      <attribute display="Amount">magnitude</attribute>\n'
        '      <attribute display="Score">score</attribute>\n'
        '    </attributes>\n'
        '    <datasets>\n'
        + datasets_xml
        + '\n'
        '    </datasets>\n'
        + node_xml
        + '\n'
        '  </krona>\n'
        '</div>\n'
        '</body>\n'
        '</html>\n'
    )

    Path(
        output_file
    ).write_text(
        document,
        encoding="utf-8",
    )

    return Path(output_file)



def process_one_csv(input_path):
    """
    Process one MetaDiv CSV.

    MODE = "global":
        one global HTML using the SUM of all detected sites/samples

    MODE = "samples":
        one HTML containing all sites/samples as TRUE Krona Datasets.
        Dataset is selectable in Krona and is NOT part of taxonomy.
    """

    print("\n" + "=" * 78)
    print(f"PROCESSING: {input_path.name}")
    print("=" * 78)

    df = pd.read_csv(
        input_path,
        low_memory=False,
        encoding="utf-8-sig",
    )

    df = canonicalize_optional_columns(df)

    tax_ranks, hierarchy = build_hierarchy(df)
    sample_columns = detect_sample_columns(df)

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    print(
        "Taxonomic hierarchy: "
        + " -> ".join(tax_ranks)
    )
    print(
        f"Sample abundance columns detected: "
        f"{len(sample_columns):,}"
    )

    dataset_dir = (
        OUTPUT_DIR
        / safe_filename(input_path.stem)
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # -----------------------------------------------------------------
    # HOMOGENIZE TAXONOMIC LINEAGES
    # -----------------------------------------------------------------

    taxonomy_log_path = None
    taxonomy_log_rows = 0

    if HOMOGENIZE_TAXONOMY:

        df, taxonomy_log = homogenize_taxonomy_for_krona(
            df=df,
            tax_ranks=tax_ranks,
            sample_columns=sample_columns,
        )

        taxonomy_log_rows = len(
            taxonomy_log
        )

        if taxonomy_log_rows > 0:

            taxonomy_log_path = (
                dataset_dir
                / "KRONA_TAXONOMY_HOMOGENIZATION.csv"
            )

            taxonomy_log.to_csv(
                taxonomy_log_path,
                index=False,
                encoding="utf-8-sig",
            )

            print()
            print(
                "Taxonomy homogenization:"
            )
            print(
                f"  Conflicting taxa corrected: "
                f"{taxonomy_log_rows:,}"
            )
            print(
                f"  Log: "
                f"{taxonomy_log_path.resolve()}"
            )

        else:

            print()
            print(
                "Taxonomy homogenization: "
                "no conflicting parent lineages detected."
            )

    # -----------------------------------------------------------------
    # MODE 1 — GLOBAL
    # -----------------------------------------------------------------

    if not EXPORT_PER_SAMPLE:

        df = ensure_total_abundance(
            df,
            sample_columns,
        )

        krona_table = create_krona_table(
            data=df,
            abundance_column="Total_Abundance",
            hierarchy_columns=hierarchy,
        )

        txt_path = (
            dataset_dir
            / "krona_Global.txt"
        )

        html_path = (
            dataset_dir
            / "krona_Global.html"
        )

        export_krona_text(
            krona_table,
            txt_path,
        )

        if CREATE_HTML:
            build_krona_html(
                krona_table,
                html_path,
                dataset_name=(
                    f"{input_path.stem} | Global"
                ),
            )

        mode_text = (
            "GLOBAL: all samples summed together"
        )

        first_hierarchy = hierarchy

        long_csv_path = None

    # -----------------------------------------------------------------
    # MODE 2 — TRUE MULTI-DATASET KRONA
    # -----------------------------------------------------------------

    else:

        krona_table = create_all_samples_long_table(
            data=df,
            sample_columns=sample_columns,
            hierarchy_columns=hierarchy,
        )

        txt_path = (
            dataset_dir
            / "krona_All_Samples_long.txt"
        )

        html_path = (
            dataset_dir
            / "krona_All_Samples.html"
        )

        long_csv_path = (
            dataset_dir
            / "krona_All_Samples_long.csv"
        )

        # Human-readable long format.
        # Dataset is physically the FIRST column and is repeated on every row.
        krona_table.to_csv(
            long_csv_path,
            index=False,
            encoding="utf-8-sig",
        )

        krona_table.to_csv(
            txt_path,
            sep="\t",
            index=False,
            header=True,
            encoding="utf-8",
        )

        if CREATE_HTML:
            build_krona_multidataset_html(
                krona_table=krona_table,
                output_file=html_path,
                chart_name=(
                    f"{input_path.stem} | Samples"
                ),
            )

        mode_text = (
            "SAMPLES: one HTML with true Krona datasets; "
            "each site/sample is selectable in the Dataset menu"
        )

        # Dataset is NOT part of taxonomy.
        first_hierarchy = hierarchy

    # -----------------------------------------------------------------
    # INFO
    # -----------------------------------------------------------------

    info_path = (
        dataset_dir
        / "KRONA_EXPORT_INFO.txt"
    )

    info_lines = [
        "MetaDiv_KRONA standalone export",
        "===============================",
        "",
        f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Input file: {input_path.name}",
        f"Rows: {len(df)}",
        f"Detected samples: {len(sample_columns)}",
        "",
        f"MODE: {MODE}",
        f"Mode: {mode_text}",
        "",
        "Krona hierarchy:",
        " -> ".join(first_hierarchy),
        "",
        f"Taxonomic cutoff: {TAXONOMIC_CUTOFF}",
        f"Homogenize taxonomy: {HOMOGENIZE_TAXONOMY}",
        f"Conflicting taxa corrected: {taxonomy_log_rows}",
        f"Include primary lifestyle: {INCLUDE_PRIMARY_LIFESTYLE}",
        f"Include secondary lifestyle: {INCLUDE_SECONDARY_LIFESTYLE}",
        f"Include SPPN: {INCLUDE_SPPN}",
        f"Max Krona paths: {MAX_KRONA_PATHS}",
        "",
        f"TXT: {txt_path.name}",
        f"HTML: {html_path.name}",
    ]

    if long_csv_path is not None:
        info_lines.extend(
            [
                f"Long-format CSV: {long_csv_path.name}",
                "",
                "Long-format structure:",
                "Dataset -> abundance -> taxonomy -> optional lifestyle",
            ]
        )

    info_path.write_text(
        "\n".join(info_lines),
        encoding="utf-8",
    )

    print()
    print(f"Mode: {mode_text}")
    print(
        "Hierarchy: "
        + " -> ".join(first_hierarchy)
    )
    print(f"TXT : {txt_path.resolve()}")

    if CREATE_HTML:
        print(f"HTML: {html_path.resolve()}")

    if long_csv_path is not None:
        print(
            f"LONG: {long_csv_path.resolve()}"
        )

    print(
        f"Krona paths: {len(krona_table):,}"
    )

    print()
    print("Preview:")
    display(
        krona_table.head(20)
    )

    return {
        "input": input_path,
        "output_dir": dataset_dir,
        "html": html_path if CREATE_HTML else None,
        "txt": txt_path,
        "long_csv": long_csv_path,
        "table": krona_table,
    }


# =============================================================================
# MAIN
# =============================================================================

INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if INPUT_FILE is None:

    input_files = sorted(
        INPUT_DIR.glob("*.csv")
    )

else:

    selected = (
        INPUT_DIR
        / Path(str(INPUT_FILE)).name
    )

    if not selected.exists():
        raise FileNotFoundError(
            "Selected input file was not found: "
            f"{selected.resolve()}"
        )

    input_files = [
        selected
    ]


if not input_files:

    raise FileNotFoundError(
        f"No CSV files were found inside: "
        f"{INPUT_DIR.resolve()}"
    )


print("=" * 78)
print("MetaDiv_KRONA — STANDALONE")
print("=" * 78)

if EXPORT_PER_SAMPLE:
    print(
        "MODE = samples: ONE HTML with ALL samples kept separate"
    )
    print(
        "TREE: sample -> taxonomy -> optional lifestyle"
    )
else:
    print(
        "MODE = global: ONE GLOBAL HTML with ALL detected sites/samples summed"
    )
    print(
        "TREE: taxonomy -> optional lifestyle"
    )

print()
print(
    f"Input CSV files: {len(input_files)}"
)

for file in input_files:
    print(
        f"- {file.name}"
    )


results = []

for input_path in input_files:

    results.append(
        process_one_csv(
            input_path
        )
    )


print("\n" + "=" * 78)
print("METADIV_KRONA COMPLETED")
print("=" * 78)

print(
    f"Datasets processed: {len(results)}"
)

if EXPORT_PER_SAMPLE:

    print(
        "Each input CSV produced ONE HTML containing "
        "all sites/samples as selectable Krona Datasets."
    )

else:

    print(
        "Each dataset produced ONE global HTML using the sum of all detected sites/samples."
    )

print(
    f"Output directory: {OUTPUT_DIR.resolve()}"
)


MetaDiv_KRONA — STANDALONE
MODE = global: ONE GLOBAL HTML with ALL detected sites/samples summed
TREE: taxonomy -> optional lifestyle

Input CSV files: 1
- Final_Database_species_only_all_prokaryotes_p08_sppn08.csv

PROCESSING: Final_Database_species_only_all_prokaryotes_p08_sppn08.csv
Rows: 2,516
Columns: 1,160
Taxonomic hierarchy: domain -> phylum -> class -> order -> family -> genus
Sample abundance columns detected: 1,138

Taxonomy homogenization:
  Conflicting taxa corrected: 4
  Log: C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_KRONA\output_KRONA\Final_Database_species_only_all_prokaryotes_p08_sppn08\KRONA_TAXONOMY_HOMOGENIZATION.csv
   GLOBAL MODE: summed 1,138 detected site/sample columns into Total_Abundance.

Mode: GLOBAL: all samples summed together
Hierarchy: domain -> phylum -> class -> order -> family -> genus
TXT : C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_KRONA\output_KRON

,domain,phylum,class,order,family,genus,abundance
0,bacteria,cyanobacteria,vampirivibrionia,obscuribacterales,obscuribacteraceae,obscuribacteraceae,46098
1,bacteria,cyanobacteria,cyanobacteriia,cyanobacteriales,Unclassified_family,Unclassified_genus,39609
2,bacteria,cyanobacteria,Unclassified_class,Unclassified_order,Unclassified_family,Unclassified_genus,37658
3,bacteria,cyanobacteria,cyanobacteriia,Unclassified_order,Unclassified_family,Unclassified_genus,27833
4,bacteria,cyanobacteria,synechococcophycideae,pseudanabaenales,pseudanabaenaceae,Unclassified_genus,22718
5,bacteria,cyanobacteria,cyanobacteriia,cyanobacteriales,uncultured,uncultured,21921
6,bacteria,cyanobacteria,cyanobacteriia,oxyphotobacteria_incertae_sedis,unknown_family,uncultured,21735
7,bacteria,cyanobacteria,cyanobacteriia,cyanobacteriales,coleofasciculaceae,microcoleus_sag_1449-1a,18470
8,bacteria,cyanobacteria,cyanobacteriia,cyanobacteriales,rubidibacteraceae,uncultured,17024
9,bacteria,cyanobacteria,cyanobacteriia,oxyphotobacteria_incertae_sedis,unknown_family,Unclassified_genus,16165



METADIV_KRONA COMPLETED
Datasets processed: 1
Each dataset produced ONE global HTML using the sum of all detected sites/samples.
Output directory: C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_KRONA\output_KRONA
